# Synapse — Single Colab Notebook
This notebook recreates your Synapse Python project as a **single Colab** by writing each module to disk (so imports keep working), with a heading for each part.


## 0) Colab workspace setup
Your code expects `/workspace/.env`. This cell creates that folder in Colab.


In [ ]:
import os
os.makedirs('/workspace', exist_ok=True)
print('✅ /workspace ready')


✅ /workspace ready


## 1) Install dependencies
Run this once per Colab runtime. `detectron2` can be heavy; it’s separated into an optional cell.


In [ ]:
!pip -q install openai  fastapi uvicorn python-dotenv supabase boto3 requests psutil   pymupdf pillow numpy doclayout-yolo huggingface-hub opencv-python-headless   transformers accelerate safetensors sentencepiece   pytesseract surya-ocr sentence-transformers scikit-learn
print('? Base deps installed (DocLayout-YOLO + OCR + Transformers)')


? Base deps installed (DocLayout-YOLO + OCR + Transformers)


In [ ]:
!pip -q install -U pip
!pip -q install doclayout-yolo==0.0.4 huggingface-hub opencv-python-headless

In [ ]:
!python -c "import doclayout_yolo; print('doclayout_yolo OK')"

doclayout_yolo OK


In [ ]:
# System OCR dependency (Tesseract)
!apt-get update -y
!apt-get install -y tesseract-ocr

# Notes:
# - We do NOT need Detectron2/LayoutParser for DocLayout-YOLO.
# - SuryaOCR is optional (academic phase).
print('? System deps installed (tesseract-ocr)')


Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Ign:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Ign:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Ign:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Ign:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRel

## 2) Environment variables (`/workspace/.env`)
Create `/workspace/.env` with your real values. This template is safe to edit.


In [ ]:
%%writefile /workspace/.env
SUPABASE_URL=YOUR_SUPABASE_URL
SUPABASE_SERVICE_ROLE_KEY=YOUR_SUPABASE_SERVICE_ROLE_KEY

GOOGLE_CLIENT_ID=YOUR_GOOGLE_CLIENT_ID
GOOGLE_CLIENT_SECRET=YOUR_GOOGLE_CLIENT_SECRET

R2_ENDPOINT=YOUR_S3_OR_R2_ENDPOINT
R2_BUCKET=YOUR_BUCKET
R2_ACCESS_KEY=YOUR_ACCESS_KEY  # optional if using AWS instance role
R2_SECRET_KEY=YOUR_SECRET_KEY  # optional if using AWS instance role
OPENAI_API_KEY=YOUR_OPENAI_API_KEY
CHAT_GPT_MODEL=gpt-4o-mini
CHAT_MATCH_RPC=match_chunk_embeddings
CHAT_CANDIDATE_SCAN=1200
CHAT_MAX_TOKENS=700

# Optional tuning
SYNC_WORKERS=8
LAYOUT_WORKERS=8
LAYOUT_BATCH_PAGES=16
LAYOUT_HALF=1
LAYOUT_RENDER_SCALE=1.25
LAYOUT_RENDER_CHUNK_PAGES=8
LAYOUT_PIPELINE_QUEUE_CHUNKS=8



# Public URL (Cloudflare quick tunnel). Filled after running the tunnel cell.
BACKEND_API_URL=
NEXT_PUBLIC_RUNPOD_API_URL=

# === Synapse pipeline config ===
PIPELINE_STAGES=sync,layout_parser,text_extraction,image_captioning,chunking,embedding
EXTRACTION_PARALLEL_STAGES=text_extraction,image_captioning
CHUNK_WORKERS=8

# ============================
# Visual understanding (image_captioning) — A100 40GB
# ============================

# Parallelism
# Single GPU: keep 1 GPU worker; scale throughput with batch sizes.
CAPTION_WORKERS=8

# Core filters / speed
VIS_MIN_BLOCK_SCORE=0.25
VIS_MIN_BBOX_AREA_PX=2500
VIS_MAX_VISUALS_PER_PAGE_FINANCIAL=5

# Doc profiling (auto: scanned vs financial vs digital)
VIS_PROFILE_PAGES=5
VIS_SCANNED_CHARS_PER_PAGE=600
VIS_FIN_LARGE_BBOX_AREA_PX=35000
VIS_FIN_LARGE_VISUALS_PER_PAGE=2
VIS_FIN_MENTIONS_MAX=1

# OCR policy
VIS_OCR_MODE=auto
VIS_FIGURE_OCR_MODE=auto

# OCR engine (GPU)
VIS_OCR_ENGINE=surya
VIS_OCR_BATCH=16

# OCR boxes (expensive; keep off for speed)
VIS_OCR_WITH_BOXES=0
VIS_OCR_BOXES_MAX=200

# Chart region OCR (legend/axes) when full OCR is weak
VIS_CHART_REGION_OCR=1
VIS_CHART_REGION_OCR_MIN_LEN=60

# Table/chart engines
VIS_TABLE_ENGINE=pdf_native
VIS_CHART_ENGINE=baseline

# Backwards compatible output
VIS_WRITE_LEGACY_CAPTIONS=1

# Tesseract config (used if you switch engine or fall back)
CAPTION_ENABLE_OCR=1
CAPTION_OCR_LANG=eng
CAPTION_OCR_CONFIG=--psm 6
CAPTION_OCR_CONFIG_SPARSE=--psm 11

# Qwen fallback (hard cases only)
VIS_ENABLE_QWEN_FALLBACK=1
VIS_QWEN_MODE=auto
VIS_QWEN_BATCH=2
VIS_QWEN_MODEL=Qwen/Qwen2-VL-2B-Instruct
VIS_QWEN_MAX_TOKENS=384
VIS_QWEN_MIN_CAPTION_CHARS=40
VIS_QWEN_STRICT=0

# Optional GLM-OCR backend (only used if VIS_OCR_ENGINE=glm_ocr)
VIS_GLM_OCR_MODEL=zai-org/GLM-OCR
VIS_GLM_OCR_MAX_TOKENS=512
VIS_GLM_OCR_PROMPT=Recognize all text.

CHUNK_STRATEGY=layout_structured

# Size control (good defaults for research + reports)
CHUNK_TARGET_TOKENS=420
CHUNK_MAX_TOKENS=700
CHUNK_MIN_TOKENS=120
CHUNK_OVERLAP_BLOCKS=1

# Visual attachment limits
CHUNK_MAX_VISUAL_SNIPPETS=6
CHUNK_MAX_VISUAL_ATTACHMENTS=3

# Heading detection sensitivity
CHUNK_HEADING_MAX_CHARS=90

# Optional: semantic boundary detection (GPU-accelerated if enabled)
# Keep OFF by default for speed (structure-based chunking already strong for PDFs).
CHUNK_SEMANTIC=0
CHUNK_SEMANTIC_MODEL=sentence-transformers/all-MiniLM-L6-v2
CHUNK_SEMANTIC_BATCH=64
CHUNK_SEMANTIC_THRESHOLD=0.25


# ===== Embedding (pgvector) =====
EMBED_WORKERS=3
EMBED_MODEL=BAAI/bge-large-en-v1.5
EMBED_DEVICE=cuda
EMBED_BATCH=64
EMBED_UPSERT_CHUNK=300
EMBED_TABLE=chunk_embeddings

# ===== Clustering (library-level) =====
# CLUSTER_WORKERS=5
# CLUSTER_RUN_TABLE=library_cluster_runs
# CLUSTER_TABLE=library_clusters
# CLUSTER_FETCH_PAGE=1000
# CLUSTER_BATCH=4096
# CLUSTER_UPSERT_CHUNK=2000
# CLUSTER_K_MIN=8
# CLUSTER_K_MAX=50
# Optional override if you want a fixed k:
# CLUSTER_K=30

EMBED_IDLE_LIMIT=0
# CLUSTER_IDLE_LIMIT=0


CHAT_USE_WORKERS=1
CHAT_RETRIEVER_WORKERS=5
CHAT_ENABLE_RERANK=1
CHAT_RERANK_MODEL=BAAI/bge-reranker-base
CHAT_HOP_TIMEOUT=20
CHAT_RERANK_DEVICE=cuda

Overwriting /workspace/.env


## 3) Project files
Each section writes one of your `.py` modules to the Colab filesystem.


### hardware.py


In [ ]:
%%writefile hardware.py
import os
import psutil
import torch

def get_cpu_count():
    return os.cpu_count() or 1

def get_ram_gb():
    return int(psutil.virtual_memory().total / (1024 ** 3))

def get_vram_gb():
    if not torch.cuda.is_available():
        return 0
    props = torch.cuda.get_device_properties(0)
    return int(props.total_memory / (1024 ** 3))

def auto_worker_plan():
    cpu = get_cpu_count()
    ram = get_ram_gb()
    vram = get_vram_gb()

    sync = min(max(2, cpu // 2), 8)
    extract = min(max(2, cpu // 3), 6)
    embed = 1 if vram >= 8 else 0
    cluster = 1

    sync = int(os.getenv("SYNC_WORKERS", sync))
    extract = int(os.getenv("EXTRACT_WORKERS", extract))
    embed = int(os.getenv("EMBED_WORKERS", embed))
    cluster = int(os.getenv("CLUSTER_WORKERS", cluster))

    return {
        "cpu": cpu,
        "ram_gb": ram,
        "vram_gb": vram,
        "sync_workers": sync,
        "extract_workers": extract,
        "embed_workers": embed,
        "cluster_workers": cluster,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

Overwriting hardware.py


### batch_creator.py


In [ ]:
%%writefile batch_creator.py
import math
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv("/workspace/.env")

supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_SERVICE_ROLE_KEY"))


def create_library_batches(
    organization_id: str,
    library_id: str,
    worker_count: int,
    stage: str = "sync",
    clear_existing: bool = True,
):
    if worker_count <= 0:
        worker_count = 1

    if clear_existing:
        # Delete stage jobs first (FK), then batches.
        supabase.table("batch_stage_jobs").delete().eq("library_id", library_id).execute()
        supabase.table("library_batches").delete().eq("library_id", library_id).execute()

    docs = (
        supabase.table("documents")
        .select("id, gdrive_file_id, title")
        .eq("library_id", library_id)
        .order("gdrive_file_id", desc=False)
        .order("title", desc=False)
        .execute()
        .data
        or []
    )

    total = len(docs)
    if total == 0:
        return {"created": 0, "batch_size": 0}

    batch_size = max(1, math.ceil(total / worker_count))
    batches = [docs[i : i + batch_size] for i in range(0, total, batch_size)]

    for idx, batch in enumerate(batches):
        doc_ids = [d["id"] for d in batch]
        inserted = (
            supabase.table("library_batches")
            .insert(
                {
                    "organization_id": organization_id,
                    "library_id": library_id,
                    "batch_index": idx,
                    "status": "queued",
                    "doc_ids": doc_ids,
                    "doc_count": len(doc_ids),
                }
            )
            .execute()
        )

        if not inserted.data:
            raise RuntimeError("Failed to insert library_batches row")
        batch_id = inserted.data[0]["id"]
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": organization_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": len(doc_ids),
            }
        ).execute()

    return {"created": len(batches), "batch_size": batch_size}


Overwriting batch_creator.py


# chat_runtime

In [ ]:
%%writefile chat_runtime.py
import os
import json
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from supabase import create_client

load_dotenv("/workspace/.env")


def _get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = _get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = _get_env("SUPABASE_SERVICE_ROLE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


_embedder = None


def _get_embedder():
    global _embedder
    if _embedder is not None:
        return _embedder
    from sentence_transformers import SentenceTransformer  # type: ignore

    model_id = os.getenv("EMBED_MODEL", "BAAI/bge-large-en-v1.5")
    device = os.getenv("EMBED_DEVICE", "").strip()
    if not device:
        try:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"
    _embedder = SentenceTransformer(model_id, device=device)
    return _embedder


def embed_query(text: str) -> List[float]:
    emb = _get_embedder().encode([text], normalize_embeddings=True, show_progress_bar=False)
    # numpy -> list
    try:
        import numpy as np

        if isinstance(emb, np.ndarray):
            return emb[0].astype("float32").tolist()
    except Exception:
        pass
    return [float(x) for x in emb[0]]

def _keywords(q: str) -> List[str]:
    import re

    q = (q or "").strip().lower()
    toks = re.findall(r"[a-z0-9][a-z0-9_\-]{2,}", q)
    stop = {
        "the","and","for","with","from","that","this","what","when","where","which","who","how",
        "into","about","their","there","have","has","had","are","was","were","will","would","can",
        "could","should","may","might","also",
    }
    out: List[str] = []
    for t in toks:
        if t in stop:
            continue
        if len(t) <= 2:
            continue
        out.append(t)
    return out[:12]

def keyword_search_chunks(
    organization_id: str,
    library_ids: List[str],
    query_text: str,
    top_k: int = 8,
) -> List[Dict[str, Any]]:
    """
    Cheap lexical fallback (works even if the vector match RPC isn't installed).
    Not as strong as BM25/FTS, but saves us from hard failures.
    """
    library_ids = [str(x) for x in (library_ids or []) if str(x)]
    if not library_ids:
        return []

    kws = _keywords(query_text)
    if not kws:
        return []

    terms = kws[:4]
    # PostgREST OR expression
    or_expr = ",".join([f"text.ilike.*{t}*" for t in terms])

    results: List[Dict[str, Any]] = []
    limit = max(40, int(top_k) * 8)
    for lib_id in library_ids[:8]:
        try:
            res = (
                supabase.table("chunk_embeddings")
                .select("chunk_id,doc_id,library_id,page_start,page_end,text,embedding_text")
                .eq("organization_id", organization_id)
                .eq("library_id", lib_id)
                .or_(or_expr)
                .order("updated_at", desc=True)
                .limit(limit)
                .execute()
            )
        except Exception:
            continue
        for r in (res.data or []):
            if not isinstance(r, dict):
                continue
            results.append(
                {
                    "chunk_id": r.get("chunk_id"),
                    "doc_id": r.get("doc_id"),
                    "library_id": r.get("library_id"),
                    "page_start": r.get("page_start"),
                    "page_end": r.get("page_end"),
                    "text": r.get("text") or r.get("embedding_text"),
                    "score": None,
                }
            )
    return results[: max(1, int(top_k))]

def _cosine_topk_from_rows(query_vec: List[float], rows: List[Dict[str, Any]], k: int) -> List[Dict[str, Any]]:
    # Fallback when RPC isn't available: rows must include `embedding` as list[float].
    try:
        import numpy as np

        q = np.array(query_vec, dtype="float32")
        q = q / (np.linalg.norm(q) + 1e-9)
        mats = []
        keep = []
        for r in rows:
            v = r.get("embedding")
            if not isinstance(v, list) or not v:
                continue
            mats.append(np.array(v, dtype="float32"))
            keep.append(r)
        if not mats:
            return []
        M = np.vstack(mats)
        M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
        sims = (M @ q).reshape(-1)
        idx = np.argsort(-sims)[: max(1, int(k))]
        out = []
        for i in idx.tolist():
            r = dict(keep[i])
            r["score"] = float(sims[i])
            out.append(r)
        return out
    except Exception:
        return rows[:k]


def retrieve_chunks(
    organization_id: str,
    library_ids: List[str],
    query_vec: List[float],
    top_k: int = 8,
) -> List[Dict[str, Any]]:
    """
    Returns chunk rows with at least:
      chunk_id, doc_id, library_id, page_start, page_end, text, score
    Strategy:
      1) Try Supabase RPC `match_chunk_embeddings` if user created it.
      2) Fallback: pull a limited set of rows + compute cosine locally (OK for small libs).
    """
    library_ids = [str(x) for x in (library_ids or []) if str(x)]
    if not library_ids:
        return []

    rpc_name = os.getenv("CHAT_MATCH_RPC", "match_chunk_embeddings").strip() or "match_chunk_embeddings"
    try:
        resp = supabase.rpc(
            rpc_name,
            {
                "p_organization_id": organization_id,
                "p_library_ids": library_ids,
                "p_query_embedding": query_vec,
                "p_match_count": int(top_k),
            },
        ).execute()
        rows = resp.data or []
        # normalize shape
        out = []
        for r in rows:
            if not isinstance(r, dict):
                continue
            out.append(
                {
                    "chunk_id": r.get("chunk_id"),
                    "doc_id": r.get("doc_id"),
                    "library_id": r.get("library_id"),
                    "page_start": r.get("page_start"),
                    "page_end": r.get("page_end"),
                    "text": r.get("text") or r.get("embedding_text"),
                    "score": r.get("score") or r.get("similarity"),
                }
            )
        return out
    except Exception:
        # If RPC isn't installed (or pgvector can't serialize), fall back to keyword search.
        query_text = os.getenv("CHAT_LAST_QUERY_TEXT", "")  # optional hook
        if query_text:
            return keyword_search_chunks(organization_id, library_ids, query_text, top_k=top_k)
        return []


def hydrate_doc_titles(doc_ids: List[str]) -> Dict[str, Dict[str, Any]]:
    doc_ids = [str(x) for x in (doc_ids or []) if str(x)]
    if not doc_ids:
        return {}
    # Pull Drive file id + path-in-source (when available) so the frontend can open PDFs.
    # NOTE: our `documents` table does not have a `source_path` column.
    res = supabase.table("documents").select("id,title,storage_path_raw,path_in_source,gdrive_file_id").in_("id", doc_ids).execute()
    out: Dict[str, Dict[str, Any]] = {}
    for r in (res.data or []):
        if not isinstance(r, dict):
            continue
        out[str(r.get("id"))] = {
            "doc_title": r.get("title"),
            "storage_path_raw": r.get("storage_path_raw"),
            "path_in_source": r.get("path_in_source"),
            "gdrive_file_id": r.get("gdrive_file_id"),
        }
    return out


def build_evidence_brief(rows: List[Dict[str, Any]], max_chars: int = 14000) -> str:
    """
    Compact evidence fed to GPT: keep it human-friendly (title + pages) and avoid leaking internal ids.
    """
    parts = []
    used = 0
    for r in rows:
        doc_title = str(r.get("doc_title") or "").strip()
        p1 = r.get("page_start")
        p2 = r.get("page_end")
        txt = (r.get("text") or "").strip()
        if not txt:
            continue
        header = f"[pdf={doc_title or 'Untitled PDF'} pages={p1}-{p2} score={r.get('score')}]"
        snippet = txt[:1200]
        block = f"{header}\n{snippet}"
        if used + len(block) + 2 > max_chars:
            break
        parts.append(block)
        used += len(block) + 2
    return "\n\n".join(parts)


Overwriting chat_runtime.py


# chat_api

In [ ]:
%%writefile chat_api.py
import os
import json
import traceback
from typing import Any, Dict, List, Optional

from dotenv import load_dotenv
from fastapi import APIRouter, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

from chat_runtime import build_evidence_brief, embed_query, hydrate_doc_titles, retrieve_chunks

load_dotenv("/workspace/.env")

router = APIRouter()


class ChatRequest(BaseModel):
    organization_id: str = Field(..., description="Organization UUID")
    library_ids: List[str] = Field(default_factory=list)
    message: str
    max_hops: int = 4
    top_k: int = 10
    thread_summary: Optional[str] = None
    history: Optional[List[Dict[str, str]]] = None
    client_request_id: Optional[str] = None

class CompactRequest(BaseModel):
    organization_id: str
    messages: List[Dict[str, str]]  # {role, content}


def _get_openai_client():
    api_key = (os.getenv("OPENAI_API_KEY") or "").strip()
    if not api_key:
        raise RuntimeError("Missing OPENAI_API_KEY in /workspace/.env")
    from openai import OpenAI  # type: ignore

    return OpenAI(api_key=api_key)


def _gpt_model() -> str:
    return (os.getenv("CHAT_GPT_MODEL") or "gpt-4o-mini").strip() or "gpt-4o-mini"


def _json_extract(content: str) -> Dict[str, Any]:
    try:
        return json.loads(content)
    except Exception:
        # Try to find a JSON object in the response
        start = content.find("{")
        end = content.rfind("}")
        if start >= 0 and end > start:
            try:
                return json.loads(content[start : end + 1])
            except Exception:
                pass
    return {}


def _followup_decision_prompt(user_query: str, evidence: str) -> List[Dict[str, str]]:
    sys = (
        "You are Synapse, a retrieval orchestrator. Decide if the provided evidence is sufficient to answer.\n"
        "Return STRICT JSON only.\n"
        "If sufficient, set next_action='NA'.\n"
        "If insufficient, set next_action='FOLLOWUP' and provide a single followup_query.\n"
        "Also provide up to 4 sub_queries that would help answer.\n"
        "Schema:\n"
        "{\n"
        "  \"next_action\": \"NA\"|\"FOLLOWUP\",\n"
        "  \"followup_query\": string|null,\n"
        "  \"why_missing\": string,\n"
        "  \"sub_queries\": [{\"query\": string, \"priority\": 1|2|3}]\n"
        "}\n"
    )
    user = f"USER_QUERY:\n{user_query}\n\nEVIDENCE:\n{evidence}\n"
    return [{"role": "system", "content": sys}, {"role": "user", "content": user}]


def _final_answer_prompt(user_query: str, evidence: str) -> List[Dict[str, str]]:
    sys = (
        "You are Synapse.\n"
        "Prefer answering using the EVIDENCE provided.\n"
        "If the evidence is insufficient, answer using your general knowledge.\n"
        "Write a helpful answer, not just a single-line definition.\n"
        "If the user asks for a definition/acronym expansion, include 2-4 extra sentences of nearby context or practical meaning when possible.\n"
        "If helpful, include 2-5 short bullet points (e.g. why it matters, common pitfalls, where it appears).\n"
        "Do NOT mention whether you did or did not find information in the user's PDFs.\n"
        "Do NOT add inline citations like '(Source: ...)'. Sources are shown separately in the UI.\n"
        "Do NOT include chunk_id, doc_id, library_id, embedding ids, or any internal identifiers in the answer.\n"
        "Be concise.\n"
        "At the very end, output a single line exactly: SOURCES_USED: yes|no (yes only if you actually used the EVIDENCE).\n"
    )
    user = f"USER_QUERY:\n{user_query}\n\nEVIDENCE:\n{evidence}\n"
    return [{"role": "system", "content": sys}, {"role": "user", "content": user}]


def _ungrounded_answer_prompt(user_query: str, convo: str) -> List[Dict[str, str]]:
    sys = (
        "You are Synapse.\n"
        "No relevant sources were retrieved from the user's libraries.\n"
        "Answer the user's question using your general knowledge.\n"
        "Be clear and helpful.\n"
        "Do NOT claim you quoted or verified anything from the user's PDFs.\n"
    )
    user = f"{(convo + '\\n\\n') if convo else ''}USER_QUERY:\n{user_query}\n"
    return [{"role": "system", "content": sys}, {"role": "user", "content": user}]

def _conversation_prefix(summary: Optional[str], history: Optional[List[Dict[str, str]]]) -> str:
    parts = []
    if summary:
        parts.append(f"THREAD_SUMMARY:\n{summary.strip()}")
    if history:
        # only keep last few turns to avoid growing prompts
        lines = []
        for m in history[-16:]:
            role = str(m.get("role") or "user")
            content = str(m.get("content") or "")
            if not content:
                continue
            lines.append(f"{role}: {content}")
        if lines:
            parts.append("RECENT_TURNS:\n" + "\n".join(lines))
    return "\n\n".join(parts).strip()


def _verbose_errors_enabled() -> bool:
    v = str(os.getenv("CHAT_VERBOSE_ERRORS", "1")).strip().lower()
    return v not in {"0", "false", "off", "no"}


def _error_response(exc: Exception, where: str):
    if _verbose_errors_enabled():
        return JSONResponse(
            status_code=500,
            content={
                "error": f"Chat backend crashed in {where}.",
                "exception_type": type(exc).__name__,
                "exception_message": str(exc),
                "traceback": traceback.format_exc()[-8000:],
            },
        )
    raise exc


def _compact_impl(req: CompactRequest):
    if not req.organization_id or not req.messages:
        raise HTTPException(status_code=400, detail="Missing organization_id or messages.")

    client = _get_openai_client()
    model = _gpt_model()

    turns = []
    for m in req.messages[-60:]:
        role = str(m.get("role") or "")
        content = str(m.get("content") or "")
        if not content:
            continue
        turns.append(f"{role}: {content}")

    sys = (
        "You are Synapse. Summarize the conversation so far for future continuation.\n"
        "Return STRICT JSON only: {\"summary\": string, \"title\": string}.\n"
        "The summary must preserve: user goals, constraints, decisions, definitions, and any specific entities/numbers.\n"
        "The title should be short (4-8 words) and describe the chat.\n"
    )
    user = "CONVERSATION:\n" + "\n".join(turns)

    out = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": sys}, {"role": "user", "content": user}],
        temperature=0.2,
        max_tokens=320,
    )
    content = (out.choices[0].message.content or "").strip()
    j = _json_extract(content)
    summary = str(j.get("summary") or "").strip()
    title = str(j.get("title") or "").strip() or "Continuation"
    if not summary:
        raise HTTPException(status_code=500, detail="Failed to generate summary.")
    return {"summary": summary, "title": title}


@router.post("/chat/compact")
def compact(req: CompactRequest):
    try:
        return _compact_impl(req)
    except Exception as exc:
        return _error_response(exc, "/chat/compact")


@router.post("/chat/compact/")
def compact_slash(req: CompactRequest):
    try:
        return _compact_impl(req)
    except Exception as exc:
        return _error_response(exc, "/chat/compact/")


def _chat_impl(req: ChatRequest):
    if not req.organization_id or not req.message.strip():
        raise HTTPException(status_code=400, detail="Missing organization_id or message.")
    if not req.library_ids:
        raise HTTPException(status_code=400, detail="Select at least one processed library.")

    top_k = max(3, min(int(req.top_k or 12), int(os.getenv('CHAT_MAX_TOP_K', '20'))))
    max_hops = max(0, min(int(req.max_hops or 4), int(os.getenv('CHAT_MAX_HOPS', '5'))))

    use_workers = str(os.getenv("CHAT_USE_WORKERS", "1")).strip() not in {"0", "false", "False"}

    rows: List[Dict[str, Any]] = []
    followups: List[Dict[str, Any]] = []

    convo = _conversation_prefix(req.thread_summary, req.history)

    if use_workers:
        try:
            from chat_queue import (
                create_chat_job,
                enqueue_retrieval_tasks,
                wait_hop_results,
                mark_chat_job_done,
                mark_chat_job_failed,
            )

            job = create_chat_job(req.organization_id, req.library_ids, req.message, top_k=top_k, max_hops=max_hops)
            chat_job_id = str(job.get("id"))

            client = _get_openai_client()
            model = _gpt_model()

            # hop 0 retrieval tasks (vector + keyword)
            qvec = embed_query(req.message)
            enqueue_retrieval_tasks(
                chat_job_id,
                req.organization_id,
                req.library_ids,
                hop=0,
                query_text=req.message,
                query_embedding=qvec,
                kinds=["vector", "keyword"],
                top_k=top_k,
            )
            results = wait_hop_results(chat_job_id, hop=0, kinds=["vector", "keyword"], timeout_s=float(os.getenv("CHAT_HOP_TIMEOUT", "20")))
            for r in results:
                for ch in (r.get("chunks") or []):
                    if isinstance(ch, dict):
                        rows.append(ch)

            if not rows:
                ans = client.chat.completions.create(
                    model=model,
                    messages=_ungrounded_answer_prompt(req.message, convo),
                    temperature=0.3,
                    max_tokens=int(os.getenv("CHAT_MAX_TOKENS", "700")),
                )
                answer = (ans.choices[0].message.content or "").strip()
                payload = {"answer": answer, "sources": [], "followups": [], "client_request_id": req.client_request_id}
                mark_chat_job_done(chat_job_id, payload)
                return payload

            # Curious hop loop: GPT decides follow-up; workers fetch; repeat.
            # Attach doc titles early so the LLM can cite by PDF name (not internal ids).
            doc_ids0 = sorted({str(r.get("doc_id")) for r in rows if str(r.get("doc_id") or "")})
            meta0 = hydrate_doc_titles(doc_ids0)
            for rr in rows:
                did = str(rr.get("doc_id") or "")
                if did and did in meta0:
                    rr["doc_title"] = meta0[did].get("doc_title") or None
            evidence = build_evidence_brief(rows)
            for hop in range(max_hops):
                dec = client.chat.completions.create(
                    model=model,
                    messages=_followup_decision_prompt(req.message, (convo + "\n\n" + evidence).strip() if convo else evidence),
                    temperature=0.2,
                    max_tokens=350,
                )
                content = (dec.choices[0].message.content or "").strip()
                j = _json_extract(content)
                next_action = str(j.get("next_action") or "").upper()
                if next_action == "NA":
                    break
                if next_action != "FOLLOWUP":
                    break
                fq = str(j.get("followup_query") or "").strip()
                if not fq:
                    break
                followups.append({"hop": hop + 1, "query": fq})

                fq_vec = embed_query(fq)
                enqueue_retrieval_tasks(
                    chat_job_id,
                    req.organization_id,
                    req.library_ids,
                    hop=hop + 1,
                    query_text=fq,
                    query_embedding=fq_vec,
                    kinds=["vector", "keyword"],
                    top_k=max(3, top_k // 2),
                )
                hop_results = wait_hop_results(chat_job_id, hop=hop + 1, kinds=["vector", "keyword"], timeout_s=float(os.getenv("CHAT_HOP_TIMEOUT", "20")))
                seen = {str(r.get("chunk_id")) for r in rows if r.get("chunk_id")}
                for hr in hop_results:
                    for ch in (hr.get("chunks") or []):
                        if not isinstance(ch, dict):
                            continue
                        cid = str(ch.get("chunk_id") or "")
                        if cid and cid in seen:
                            continue
                        rows.append(ch)
                        if cid:
                            seen.add(cid)

                # Re-hydrate doc titles as we add more rows.
                doc_ids_h = sorted({str(r.get("doc_id")) for r in rows if str(r.get("doc_id") or "")})
                meta_h = hydrate_doc_titles(doc_ids_h)
                for rr in rows:
                    did = str(rr.get("doc_id") or "")
                    if did and did in meta_h:
                        rr["doc_title"] = meta_h[did].get("doc_title") or None
                evidence = build_evidence_brief(rows)

            ans = client.chat.completions.create(
                model=model,
                messages=_final_answer_prompt(req.message, (convo + "\n\n" + evidence).strip() if convo else evidence),
                temperature=0.2,
                max_tokens=int(os.getenv("CHAT_MAX_TOKENS", "700")),
            )
            answer = (ans.choices[0].message.content or "").strip()

            # Hydrate sources
            doc_ids = [str(r.get("doc_id")) for r in rows if r.get("doc_id")]
            meta = hydrate_doc_titles(doc_ids)
            sources = []
            for r in rows[: min(len(rows), 20)]:
                did = str(r.get("doc_id") or "")
                m = meta.get(did) or {}
                sources.append(
                    {
                        "library_id": r.get("library_id"),
                        "doc_id": r.get("doc_id"),
                        "doc_title": m.get("doc_title"),
                        "storage_path_raw": m.get("storage_path_raw"),
                        "path_in_source": m.get("path_in_source"),
                        "gdrive_file_id": m.get("gdrive_file_id"),
                        "page_start": r.get("page_start"),
                        "page_end": r.get("page_end"),
                        "chunk_id": r.get("chunk_id"),
                        "score": r.get("score"),
                    }
                )

            # If the model says it can't find info, do not show misleading sources.
            # sources_used signal (we strip from display on the frontend)
            sources_used = "SOURCES_USED: yes" in answer
            answer = answer.replace("SOURCES_USED: yes", "").replace("SOURCES_USED: no", "").strip()
            retrieval_confident = False
            try:
                best_score = 0.0
                for rr in rows:
                    sc = rr.get('score')
                    if sc is None:
                        continue
                    try:
                        best_score = max(best_score, float(sc))
                    except Exception:
                        pass
                retrieval_confident = best_score >= float(os.getenv('CHAT_MIN_SOURCE_SCORE', '0.18'))
            except Exception:
                retrieval_confident = False
            if not sources_used and not retrieval_confident:
                sources = []

            payload = {"answer": answer, "sources": sources, "followups": followups, "client_request_id": req.client_request_id}
            mark_chat_job_done(chat_job_id, payload)
            return payload
        except Exception as exc:
            try:
                # Best-effort: mark job failed if it exists
                mark_chat_job_failed(chat_job_id, str(exc))  # type: ignore[name-defined]
            except Exception:
                pass
            # fall back to inline mode
            use_workers = False

    # Inline fallback (no queue tables / workers)
    # Provide the query text as a last-resort hint for keyword fallback inside chat_runtime.
    os.environ["CHAT_LAST_QUERY_TEXT"] = req.message
    qvec = embed_query(req.message)
    rows = retrieve_chunks(req.organization_id, req.library_ids, qvec, top_k=top_k)
    if not rows:
        # Keyword fallback when vector RPC isn't installed yet.
        try:
            from chat_runtime import keyword_search_chunks
            rows = keyword_search_chunks(req.organization_id, req.library_ids, req.message, top_k=top_k)
        except Exception:
            rows = []

    if not rows:
        ans = client.chat.completions.create(
            model=model,
            messages=_ungrounded_answer_prompt(req.message, convo),
            temperature=0.3,
            max_tokens=int(os.getenv("CHAT_MAX_TOKENS", "700")),
        )
        answer = (ans.choices[0].message.content or "").strip()
        return {"answer": answer, "sources": [], "followups": []}

    client = _get_openai_client()
    model = _gpt_model()

    # CuriousLLM-style hop loop (small, budgeted)
    doc_ids0 = sorted({str(r.get("doc_id")) for r in rows if str(r.get("doc_id") or "")})
    meta0 = hydrate_doc_titles(doc_ids0)
    for rr in rows:
        did = str(rr.get("doc_id") or "")
        if did and did in meta0:
            rr["doc_title"] = meta0[did].get("doc_title") or None
    evidence = build_evidence_brief(rows)
    for hop in range(max_hops):
        dec = client.chat.completions.create(
            model=model,
            messages=_followup_decision_prompt(req.message, (convo + "\n\n" + evidence).strip() if convo else evidence),
            temperature=0.2,
            max_tokens=350,
        )
        content = (dec.choices[0].message.content or "").strip()
        j = _json_extract(content)
        next_action = str(j.get("next_action") or "").upper()
        if next_action == "NA":
            break
        if next_action != "FOLLOWUP":
            # If model returns something unexpected, stop rather than looping.
            break
        fq = str(j.get("followup_query") or "").strip()
        if not fq:
            break
        followups.append({"hop": hop + 1, "query": fq})
        fq_vec = embed_query(fq)
        more = retrieve_chunks(req.organization_id, req.library_ids, fq_vec, top_k=max(3, top_k // 2))
        # Merge (dedupe by chunk_id)
        seen = {str(r.get("chunk_id")) for r in rows if r.get("chunk_id")}
        for r in more:
            cid = str(r.get("chunk_id") or "")
            if cid and cid in seen:
                continue
            rows.append(r)
            if cid:
                seen.add(cid)
        doc_ids_h = sorted({str(r.get("doc_id")) for r in rows if str(r.get("doc_id") or "")})
        meta_h = hydrate_doc_titles(doc_ids_h)
        for rr in rows:
            did = str(rr.get("doc_id") or "")
            if did and did in meta_h:
                rr["doc_title"] = meta_h[did].get("doc_title") or None
        evidence = build_evidence_brief(rows)

    # Final answer
    ans = client.chat.completions.create(
        model=model,
        messages=_final_answer_prompt(req.message, (convo + "\n\n" + evidence).strip() if convo else evidence),
        temperature=0.2,
        max_tokens=int(os.getenv("CHAT_MAX_TOKENS", "700")),
    )
    answer = (ans.choices[0].message.content or "").strip()

    # Hydrate sources
    doc_ids = [str(r.get("doc_id")) for r in rows if r.get("doc_id")]
    meta = hydrate_doc_titles(doc_ids)
    sources = []
    for r in rows[: min(len(rows), 20)]:
        did = str(r.get("doc_id") or "")
        m = meta.get(did) or {}
        sources.append(
            {
                "library_id": r.get("library_id"),
                "doc_id": r.get("doc_id"),
                "doc_title": m.get("doc_title"),
                "storage_path_raw": m.get("storage_path_raw"),
                "path_in_source": m.get("path_in_source"),
                "gdrive_file_id": m.get("gdrive_file_id"),
                "page_start": r.get("page_start"),
                "page_end": r.get("page_end"),
                "chunk_id": r.get("chunk_id"),
                "score": r.get("score"),
            }
        )

    sources_used = "SOURCES_USED: yes" in answer
    answer = answer.replace("SOURCES_USED: yes", "").replace("SOURCES_USED: no", "").strip()
    retrieval_confident = False
    try:
        best_score = 0.0
        for rr in rows:
            sc = rr.get('score')
            if sc is None:
                continue
            try:
                best_score = max(best_score, float(sc))
            except Exception:
                pass
        retrieval_confident = best_score >= float(os.getenv('CHAT_MIN_SOURCE_SCORE', '0.18'))
    except Exception:
        retrieval_confident = False
    if not sources_used and not retrieval_confident:
        sources = []

    return {"answer": answer, "sources": sources, "followups": followups, "client_request_id": req.client_request_id}


@router.post("/chat")
def chat(req: ChatRequest):
    try:
        return _chat_impl(req)
    except Exception as exc:
        return _error_response(exc, "/chat")


@router.post("/chat/")
def chat_slash(req: ChatRequest):
    try:
        return _chat_impl(req)
    except Exception as exc:
        return _error_response(exc, "/chat/")


Overwriting chat_api.py


# chat_retriever_worker

In [ ]:
%%writefile chat_retriever_worker.py
import os
import re
import time
from typing import Any, Dict, List

from dotenv import load_dotenv

from chat_queue import (
    claim_retrieval_task,
    complete_retrieval_task,
    fail_retrieval_task,
)
from chat_runtime import retrieve_chunks

load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "chat-retriever-1")


def _keywords(q: str) -> List[str]:
    q = (q or "").strip().lower()
    toks = re.findall(r"[a-z0-9][a-z0-9_\-]{2,}", q)
    stop = {
        "the",
        "and",
        "for",
        "with",
        "from",
        "that",
        "this",
        "what",
        "when",
        "where",
        "which",
        "who",
        "how",
        "into",
        "about",
        "their",
        "there",
        "have",
        "has",
        "had",
        "are",
        "was",
        "were",
        "will",
        "would",
        "can",
        "could",
        "should",
        "may",
        "might",
        "also",
    }
    out = []
    for t in toks:
        if t in stop:
            continue
        if len(t) <= 2:
            continue
        out.append(t)
    return out[:12]


def _keyword_search(organization_id: str, library_ids: List[str], query_text: str, top_k: int) -> List[Dict[str, Any]]:
    """
    Cheap lexical fallback using PostgREST filters.
    Not as good as proper FTS/BM25, but helpful for names/numbers when vector misses.
    """
    from chat_queue import supabase  # reuse client

    kws = _keywords(query_text)
    if not kws:
        return []

    # Use only a few terms to keep query reasonable.
    terms = kws[:4]
    pattern = "|".join([re.escape(t) for t in terms])
    # PostgREST: ilike any of terms by OR chaining.
    # Note: supabase-py supports .or_() with a raw expression.
    or_expr = ",".join([f"text.ilike.*{t}*" for t in terms])

    results: List[Dict[str, Any]] = []
    for lib_id in library_ids[:8]:
        q = (
            supabase.table("chunk_embeddings")
            .select("chunk_id,doc_id,library_id,page_start,page_end,text")
            .eq("organization_id", organization_id)
            .eq("library_id", lib_id)
            .or_(or_expr)
            .limit(max(30, int(top_k) * 5))
        )
        res = q.execute()
        for r in (res.data or []):
            if isinstance(r, dict):
                r["score"] = None
                results.append(r)

    # Lightweight rerank with a cross-encoder if enabled.
    try:
        if int(os.getenv("CHAT_ENABLE_RERANK", "1")) != 1:
            return results[:top_k]
    except Exception:
        return results[:top_k]

    rerank_model = os.getenv("CHAT_RERANK_MODEL", "BAAI/bge-reranker-base").strip() or "BAAI/bge-reranker-base"
    try:
        from sentence_transformers import CrossEncoder  # type: ignore

        ce = CrossEncoder(rerank_model, device=os.getenv("CHAT_RERANK_DEVICE", "cuda" if os.getenv("CUDA_VISIBLE_DEVICES") else "cpu"))
        pairs = []
        kept = []
        for r in results[:300]:
            txt = str(r.get("text") or "").strip()
            if not txt:
                continue
            pairs.append((query_text, txt[:1200]))
            kept.append(r)
        if not pairs:
            return results[:top_k]
        scores = ce.predict(pairs)
        scored = []
        for r, sc in zip(kept, scores):
            rr = dict(r)
            rr["score"] = float(sc)
            scored.append(rr)
        scored.sort(key=lambda x: float(x.get("score") or 0), reverse=True)
        return scored[:top_k]
    except Exception:
        return results[:top_k]


def run_task(task: Dict[str, Any]) -> Dict[str, Any]:
    kind = str(task.get("kind") or "")
    organization_id = str(task.get("organization_id") or "")
    library_ids = task.get("library_ids") or []
    if not isinstance(library_ids, list):
        library_ids = []

    payload = task.get("payload") or {}
    query_text = str(payload.get("query_text") or "")
    top_k = int(payload.get("top_k") or 10)
    query_embedding = payload.get("query_embedding")

    if kind == "vector":
        if not isinstance(query_embedding, list) or not query_embedding:
            return {"chunks": []}
        chunks = retrieve_chunks(organization_id, library_ids, query_embedding, top_k=top_k)
        return {"chunks": chunks}

    if kind == "keyword":
        chunks = _keyword_search(organization_id, library_ids, query_text, top_k=top_k)
        return {"chunks": chunks}

    return {"chunks": []}


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("CHAT_IDLE_LIMIT", "600"))
    while True:
        task = claim_retrieval_task(WORKER_ID)
        if not task:
            idle += 1
            if idle >= idle_limit:
                print("No chat retrieval tasks remaining. Exiting.")
                return
            time.sleep(0.5)
            continue
        idle = 0
        try:
            result = run_task(task)
            complete_retrieval_task(task["id"], result)
        except Exception as exc:
            fail_retrieval_task(task["id"], str(exc))
            time.sleep(0.2)


if __name__ == "__main__":
    worker_loop()



Overwriting chat_retriever_worker.py


### sync_worker.py


In [ ]:
%%writefile sync_worker.py
import os
import time
import random
from datetime import datetime, timezone
from batch_creator import create_library_batches
import boto3
import requests
from supabase import create_client, Client
from dotenv import load_dotenv
load_dotenv("/workspace/.env")


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing env var: {name}")
    return value


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

GOOGLE_CLIENT_ID = get_env("GOOGLE_CLIENT_ID")
GOOGLE_CLIENT_SECRET = get_env("GOOGLE_CLIENT_SECRET")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _is_retryable_supabase_error(exc: Exception) -> bool:
    # Supabase/PostgREST sometimes returns Cloudflare HTML (502/521) which the client can't JSON-decode.
    # We retry those, plus common transient network/timeouts and 429s.
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True

    try:
        if exc.args and isinstance(exc.args[0], dict):
            code = str(exc.args[0].get("code") or "")
            details = str(exc.args[0].get("details") or "").lower()
            if code in {"502", "521", "429"}:
                return True
            if "bad gateway" in details or "web server is down" in details:
                return True
    except Exception:
        pass

    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    """
    Execute a Supabase query with retries/backoff on transient 5xx/429/timeout.

    This is important because sync/layout can generate lots of requests (multiple workers),
    and Supabase can intermittently respond with Cloudflare HTML 502 which breaks JSON parsing.
    """
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))

    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            # Exponential backoff with jitter
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)

    if last_exc:
        raise last_exc


def slugify(value: str) -> str:
    return "".join(c.lower() if c.isalnum() else "-" for c in value).strip("-")


def get_access_token(refresh_token: str) -> str:
    url = "https://oauth2.googleapis.com/token"
    payload = {
        "client_id": GOOGLE_CLIENT_ID,
        "client_secret": GOOGLE_CLIENT_SECRET,
        "refresh_token": refresh_token,
        "grant_type": "refresh_token",
    }
    res = requests.post(url, data=payload, timeout=30)
    res.raise_for_status()
    return res.json()["access_token"]


def list_drive_files(access_token: str, folder_id: str):
    files = []
    page_token = None
    headers = {"Authorization": f"Bearer {access_token}"}
    while True:
        params = {
            "q": f"'{folder_id}' in parents and trashed=false",
            "fields": "nextPageToken, files(id, name, mimeType, size, modifiedTime)",
            "pageSize": 1000,
            "pageToken": page_token,
            "supportsAllDrives": "true",
            "includeItemsFromAllDrives": "true",
        }
        res = requests.get(
            "https://www.googleapis.com/drive/v3/files",
            headers=headers,
            params=params,
            timeout=30,
        )
        res.raise_for_status()
        data = res.json()
        files.extend(data.get("files", []))
        page_token = data.get("nextPageToken")
        if not page_token:
            break
    return files

def download_drive_file(access_token: str, file_id: str, mime_type: str | None):
    headers = {"Authorization": f"Bearer {access_token}"}

    export_map = {
        "application/vnd.google-apps.document": "application/pdf",
        "application/vnd.google-apps.spreadsheet": "text/csv",
        "application/vnd.google-apps.presentation": "application/pdf",
        "application/vnd.google-apps.drawing": "application/pdf",
    }

    if mime_type in export_map:
        export_mime = export_map[mime_type]
        url = f"https://www.googleapis.com/drive/v3/files/{file_id}/export"
        res = requests.get(
            url,
            headers=headers,
            params={"mimeType": export_mime, "supportsAllDrives": "true"},
            timeout=120,
        )
        res.raise_for_status()
        return res.content, export_mime

    url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media"
    res = requests.get(
        url,
        headers=headers,
        params={"supportsAllDrives": "true"},
        timeout=120,
    )
    res.raise_for_status()
    return res.content, mime_type




def upload_to_r2(
    org_id: str,
    org_name: str,
    library_id: str,
    library_name: str,
    file_id: str,
    filename: str,
    content: bytes
) -> str:
    safe_name = filename.replace("/", "_").replace("\\", "_")
    org_slug = slugify(org_name or "org")
    lib_slug = slugify(library_name or "library")
    key = (
        f"org_{org_slug}_{org_id}/"
        f"library_{lib_slug}_{library_id}/raw/"
        f"{file_id}-{safe_name}"
    )
    s3.put_object(Bucket=R2_BUCKET, Key=key, Body=content)
    return key


def ensure_extension(filename: str, mime_type: str | None) -> str:
    ext_map = {
        "application/pdf": ".pdf",
        "text/csv": ".csv",
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    }
    if not mime_type:
        return filename
    ext = ext_map.get(mime_type)
    if not ext:
        return filename
    if filename.lower().endswith(ext):
        return filename
    return f"{filename}{ext}"


def parse_drive_time(value: str | None):
    if not value:
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except Exception:
        return None


def claim_job():
    q = (
        supabase.table("processing_jobs")
        .select("*")
        .eq("status", "queued")
        .in_("type", ["library_preprocess", "library_sync"])
        .order("created_at")
        .limit(1)
    )
    jobs = _sb_execute(q, context="processing_jobs.select(queued)")
    if not jobs.data:
        return None

    job = jobs.data[0]
    claimed_q = (
        supabase.table("processing_jobs")
        .update({"status": "running"})
        .eq("id", job["id"])
        .eq("status", "queued")
    )
    claimed = _sb_execute(claimed_q, context="processing_jobs.update(claim)")
    if not claimed.data:
        return None
    return claimed.data[0]


def complete_job(job_id: str, error: str | None = None):
    q = (
        supabase.table("processing_jobs")
        .update(
            {
                "status": "failed" if error else "done",
                "last_error": error,
            }
        )
        .eq("id", job_id)
    )
    _sb_execute(q, context="processing_jobs.update(complete)")


def run_sync(job):
    """Phase-2: bootstrapper for sync.

    Claims a `processing_jobs` row and prepares the per-batch queue:
    - upserts `documents` metadata from Drive
    - creates `library_batches`
    - enqueues `batch_stage_jobs(stage=sync)`
    """
    library_id = job["library_id"]
    org_id = job["organization_id"]

    try:
        sync_workers = int(os.getenv("SYNC_WORKERS", "3"))

        org = _sb_execute(
            supabase.table("organizations").select("name").eq("id", org_id).single(),
            context="organizations.select(name)",
        )
        lib = _sb_execute(
            supabase.table("libraries").select("name").eq("id", library_id).single(),
            context="libraries.select(name)",
        )
        org_name = org.data["name"] if org.data else "org"
        lib_name = lib.data["name"] if lib.data else "library"

        source = (
            supabase.table("library_sources")
            .select("*")
            .eq("library_id", library_id)
            .eq("organization_id", org_id)
            .limit(1)
        )
        source = _sb_execute(source, context="library_sources.select")
        if not source.data:
            raise RuntimeError("No library source found for sync job")

        source = source.data[0]
        access_token = get_access_token(source["refresh_token"])
        folder_id = source["folder_id"]

        _sb_execute(
            supabase.table("libraries").update(
                {
                    "status": "processing",
                    "pipeline_status": "running",
                    "pipeline_stage": "sync",
                    "pipeline_progress_percent": 0,
                    "pipeline_error": None,
                    "pipeline_started_at": now_iso(),
                    "pipeline_finished_at": None,
                    "completed_batches": 0,
                }
            ).eq("id", library_id),
            context="libraries.update(pipeline_start)",
        )

        files = list_drive_files(access_token, folder_id)

        existing = _sb_execute(
            supabase.table("documents").select("id, gdrive_file_id").eq("library_id", library_id),
            context="documents.select(existing)",
        )
        existing_map = {d["gdrive_file_id"]: d for d in existing.data if d.get("gdrive_file_id")}
        seen = set()

        # Bulk upsert documents metadata (reduces request count, avoids intermittent Supabase 502s).
        upserts = []
        for f in files:
            file_id = f["id"]
            seen.add(file_id)
            upserts.append(
                {
                    "organization_id": org_id,
                    "library_id": library_id,
                    "title": f.get("name") or file_id,
                    "mime_type": f.get("mimeType"),
                    "file_size_bytes": int(f.get("size") or 0),
                    "status": "pending",
                    "gdrive_file_id": file_id,
                }
            )

        chunk_size = int(os.getenv("DOC_META_UPSERT_CHUNK", "200"))
        for i in range(0, len(upserts), chunk_size):
            chunk = upserts[i : i + chunk_size]
            _sb_execute(
                supabase.table("documents").upsert(chunk, on_conflict="library_id,gdrive_file_id"),
                context=f"documents.upsert(meta) [{i}:{i+len(chunk)}]",
            )

        # Mark docs that disappeared from Drive in bulk.
        to_skip_ids = [doc["id"] for fid, doc in existing_map.items() if fid not in seen]
        for i in range(0, len(to_skip_ids), chunk_size):
            ids_chunk = to_skip_ids[i : i + chunk_size]
            _sb_execute(
                supabase.table("documents")
                .update({"status": "skipped", "skipped_reason": "deleted"})
                .in_("id", ids_chunk),
                context=f"documents.update(skipped) [{i}:{i+len(ids_chunk)}]",
            )

        batches = create_library_batches(org_id, library_id, worker_count=sync_workers, stage="sync")
        total_batches = int(batches.get("created") or 0)

        _sb_execute(
            supabase.table("libraries").update(
                {
                    "total_batches": total_batches,
                    "completed_batches": 0,
                    "pipeline_progress_percent": 0,
                    "pipeline_error": None,
                }
            ).eq("id", library_id),
            context="libraries.update(total_batches)",
        )

        print(f"[bootstrap] library={library_id} docs={len(files)} batches={total_batches} workers={sync_workers}")

    except Exception as exc:
        _sb_execute(
            supabase.table("libraries").update(
                {
                    "status": "error",
                    "pipeline_status": "failed",
                    "pipeline_stage": "sync",
                    "pipeline_error": str(exc),
                }
            ).eq("id", library_id),
            context="libraries.update(sync_failed)",
        )
        raise

def claim_sync_stage_job(worker_id: str):
    jobs_q = (
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "sync")
        .eq("status", "queued")
        .order("created_at")
        .limit(1)
    )
    jobs = _sb_execute(jobs_q, context="batch_stage_jobs.select(sync.queued)")
    if not jobs.data:
        return None

    job = jobs.data[0]
    claimed_q = (
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued")
    )
    claimed = _sb_execute(claimed_q, context="batch_stage_jobs.update(sync.claim)")
    if not claimed.data:
        return None
    return claimed.data[0]

def _update_library_progress(library_id: str):
    # Back-compat wrapper: keep the old name but compute progress from stage jobs.
    _update_pipeline_progress(library_id, current_stage="sync")


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["sync"]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _count_remaining_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .neq("status", "done"),
        context=f"batch_stage_jobs.count(remaining:{stage})",
    )
    return int(resp.count or 0)


def _update_pipeline_progress(library_id: str, current_stage: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)

    # For UI "Batches X/Y" we show completion for the current stage.
    # `completed_batches` represents fully-processed batches (i.e. reached the last stage),
    # not "batches completed in the current stage".
    completed_batches = _count_done_stage_jobs(library_id, _pipeline_stages()[-1])

    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": current_stage,
                "completed_batches": completed_batches,
                "pipeline_progress_percent": progress,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline_progress)",
    )

def run_sync_stage_job(stage_job):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    try:
        batch = _sb_execute(
            supabase.table("library_batches").select("id, doc_ids, doc_count").eq("id", batch_id).single(),
            context="library_batches.select(batch)",
        )
        doc_ids = (batch.data or {}).get("doc_ids") or []
        _sb_execute(
            supabase.table("library_batches").update({"status": "running"}).eq("id", batch_id),
            context="library_batches.update(running)",
        )
        org = _sb_execute(
            supabase.table("organizations").select("name").eq("id", org_id).single(),
            context="organizations.select(name)",
        )
        lib = _sb_execute(
            supabase.table("libraries").select("name, status, pipeline_status").eq("id", library_id).single(),
            context="libraries.select(name,status)",
        )
        org_name = org.data["name"] if org.data else "org"
        lib_name = lib.data["name"] if lib.data else "library"

        # If the library was deleted/cancelled while workers are running, stop cleanly.
        lib_status = (lib.data or {}).get("status")
        pipe_status = (lib.data or {}).get("pipeline_status")
        if not lib.data or lib_status in {"deleted"} or pipe_status in {"cancelled"}:
            _sb_execute(
                supabase.table("batch_stage_jobs").update(
                    {"status": "cancelled", "finished_at": now_iso(), "last_error": "library cancelled/deleted"}
                ).eq("id", job_id),
                context="batch_stage_jobs.update(cancelled)",
            )
            return

        source = (
            supabase.table("library_sources")
            .select("refresh_token")
            .eq("library_id", library_id)
            .eq("organization_id", org_id)
            .single()
        )
        source = _sb_execute(source, context="library_sources.select(refresh_token)")
        refresh = (source.data or {}).get("refresh_token")
        if not refresh:
            raise RuntimeError("Missing refresh_token for library_sources")
        access_token = get_access_token(refresh)
        total = int(stage_job.get("progress_total") or len(doc_ids) or 0)
        current = int(stage_job.get("progress_current") or 0)
        if total <= 0:
            total = len(doc_ids)

        # Fetch document rows for this batch in chunks (avoid one query per doc_id).
        docs_by_id: dict[str, dict] = {}
        id_chunk = int(os.getenv("DOC_FETCH_CHUNK", "100"))
        for i in range(0, len(doc_ids), id_chunk):
            chunk = doc_ids[i : i + id_chunk]
            resp = _sb_execute(
                supabase.table("documents")
                .select("id, gdrive_file_id, title, mime_type, file_size_bytes")
                .in_("id", chunk),
                context=f"documents.select(batch) [{i}:{i+len(chunk)}]",
            )
            for d in resp.data or []:
                docs_by_id[d["id"]] = d

        progress_every = int(os.getenv("STAGE_PROGRESS_EVERY", "5"))
        update_chunk = int(os.getenv("DOC_STORAGE_UPSERT_CHUNK", "20"))
        storage_updates = []

        for doc_id in doc_ids:
            d = docs_by_id.get(doc_id)
            if not d or not d.get("gdrive_file_id"):
                continue
            file_id = d["gdrive_file_id"]
            name = d.get("title") or file_id
            mime = d.get("mime_type")
            content, effective_mime = download_drive_file(access_token, file_id, mime)
            name = ensure_extension(name, effective_mime)
            storage_path = upload_to_r2(org_id, org_name, library_id, lib_name, file_id, name, content)

            # Use upsert for batch efficiency, but include required columns so we never attempt to insert
            # a partial row (which would violate NOT NULL constraints like organization_id/library_id).
            storage_updates.append(
                {
                    "id": doc_id,
                    "organization_id": org_id,
                    "library_id": library_id,
                    "gdrive_file_id": file_id,
                    "title": name,
                    "mime_type": effective_mime or mime,
                    "file_size_bytes": int(d.get("file_size_bytes") or 0),
                    "status": "pending",
                    "storage_path": storage_path,
                    "storage_path_raw": storage_path,
                }
            )

            current += 1

            # Batch doc updates to reduce Supabase write load.
            if len(storage_updates) >= update_chunk:
                _sb_execute(
                    supabase.table("documents").upsert(storage_updates, on_conflict="id"),
                    context="documents.upsert(storage_path)",
                )
                storage_updates = []

            # Throttle progress updates (don’t write every doc).
            if current % progress_every == 0:
                _sb_execute(
                    supabase.table("batch_stage_jobs").update(
                        {"progress_current": current, "progress_total": total}
                    ).eq("id", job_id),
                    context="batch_stage_jobs.update(progress)",
                )

        if storage_updates:
            _sb_execute(
                supabase.table("documents").upsert(storage_updates, on_conflict="id"),
                context="documents.upsert(storage_path.final)",
            )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "done", "finished_at": now_iso(), "progress_current": total, "progress_total": total}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(done)",
        )
        _sb_execute(
            supabase.table("library_batches").update({"status": "completed", "completed_at": now_iso()}).eq(
                "id", batch_id
            ),
            context="library_batches.update(completed)",
        )

        # Enqueue next stage (layout parsing) for this batch.
        _sb_execute(
            supabase.table("batch_stage_jobs").upsert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": "layout_parser",
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": total,
            },
            on_conflict="batch_id,stage",
            ),
            context="batch_stage_jobs.upsert(layout_parser)",
        )
        _update_pipeline_progress(library_id, current_stage="sync")
        # If all sync batches are done, finalize library sync.
        remaining = _sb_execute(
            supabase.table("batch_stage_jobs")
            .select("id", count="exact")
            .eq("library_id", library_id)
            .eq("stage", "sync")
            .neq("status", "done"),
            context="batch_stage_jobs.count(sync.remaining)",
        )
        if int(remaining.count or 0) == 0:
            # If a next stage exists (layout_parser), transition the library instead of marking pipeline completed.
            stages = _pipeline_stages()
            next_stage = None
            if "sync" in stages:
                idx = stages.index("sync")
                if idx < len(stages) - 1:
                    next_stage = stages[idx + 1]

            if next_stage:
                # Keep the library in processing state so the UI continues polling.
                _sb_execute(
                    supabase.table("libraries").update(
                        {
                            "status": "processing",
                            "pipeline_status": "running",
                            "pipeline_stage": next_stage,
                            "pipeline_error": None,
                        }
                    ).eq("id", library_id),
                    context="libraries.update(transition_next_stage)",
                )
                _update_pipeline_progress(library_id, current_stage=next_stage)
            else:
                finished = now_iso()
                _sb_execute(
                    supabase.table("libraries").update(
                        {
                            "status": "ready",
                            "last_synced_at": finished,
                            "pipeline_status": "completed",
                            "pipeline_stage": "sync",
                            "pipeline_progress_percent": 100,
                            "pipeline_error": None,
                            "pipeline_finished_at": finished,
                        }
                    ).eq("id", library_id),
                    context="libraries.update(sync.completed)",
                )
    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(sync.failed)",
        )
        _sb_execute(
            supabase.table("libraries").update(
                {
                    "status": "error",
                    "pipeline_status": "failed",
                    "pipeline_stage": "sync",
                    "pipeline_error": str(exc),
                }
            ).eq("id", library_id),
            context="libraries.update(sync.failed)",
        )
        raise

def worker_loop():
    while True:
        job = claim_job()
        if not job:
            time.sleep(3)
            continue

        try:
            run_sync(job)
            complete_job(job["id"])
        except Exception as exc:
            _sb_execute(
                supabase.table("libraries").update(
                    {
                        "status": "error",
                        "pipeline_status": "failed",
                        "pipeline_stage": "sync",
                        "pipeline_error": str(exc),
                    }
                ).eq("id", job["library_id"]),
                context="libraries.update(worker_loop.failed)",
            )
            complete_job(job["id"], str(exc))
            time.sleep(2)


if __name__ == "__main__":
    worker_loop()


Overwriting sync_worker.py


### worker_bootstrap.py


In [ ]:
%%writefile worker_bootstrap.py
# worker_bootstrap.py
import os
import time
import signal
import multiprocessing as mp
from dotenv import load_dotenv
from hardware import auto_worker_plan
from sync_worker import (
    run_sync,
    claim_job,
    complete_job,
    claim_sync_stage_job,
    run_sync_stage_job,
)

load_dotenv("/workspace/.env")

_POOL_STARTED = False
_POOL_LOCK_PATH = os.getenv("WORKER_POOL_LOCK", "/tmp/synapse_worker_pool.lock")


def _pid_alive(pid: int) -> bool:
    try:
        # Works on Unix. On Windows, this raises for most PIDs; Colab is Unix.
        os.kill(pid, 0)
        return True
    except Exception:
        return False


def _acquire_pool_lock() -> bool:
    """
    Prevent accidentally starting multiple worker pools (common when the backend cell is re-run,
    or a server reload/import occurs). If an existing pool PID is alive, we refuse to start a new pool.
    """
    try:
        if os.path.exists(_POOL_LOCK_PATH):
            raw = ""
            try:
                with open(_POOL_LOCK_PATH, "r", encoding="utf-8") as f:
                    raw = (f.read() or "").strip()
            except Exception:
                raw = ""
            if raw.isdigit() and _pid_alive(int(raw)):
                print(f"[worker-pool] lock exists; pool already started (pid={raw}).")
                return False
            # Stale lock
            try:
                os.remove(_POOL_LOCK_PATH)
            except Exception:
                pass

        with open(_POOL_LOCK_PATH, "w", encoding="utf-8") as f:
            f.write(str(os.getpid()))
        return True
    except Exception:
        # If lock can't be created, still start (best effort).
        return True

def preprocess_worker(worker_id: int, stop_event: mp.Event):
    print(f"[preprocess-{worker_id}] started")
    while not stop_event.is_set():
        job = claim_job()
        if not job:
            time.sleep(2)
            continue
        try:
            run_sync(job)
            complete_job(job["id"])
        except Exception as exc:
            complete_job(job["id"], str(exc))
            time.sleep(1)
    print(f"[preprocess-{worker_id}] stopped")


def sync_batch_worker(worker_id: int, stop_event: mp.Event):
    wid = f"sync-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_sync_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_sync_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def layout_worker(worker_id: int, stop_event: mp.Event):
    # Import inside the process so uvicorn can boot even if layout deps aren't installed yet,
    # and to avoid initializing CUDA in the parent process.
    from layout_worker import claim_layout_stage_job, run_layout_stage_job
    wid = f"layout-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_layout_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_layout_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def extraction_worker(worker_id: int, stop_event: mp.Event):
    # Import inside the process so uvicorn can boot even if extraction deps aren't installed yet.
    from extraction_worker import claim_text_extraction_stage_job, run_text_extraction_stage_job

    wid = f"extract-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_text_extraction_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_text_extraction_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def caption_worker(worker_id: int, stop_event: mp.Event):
    # Import inside the process so uvicorn can boot even if caption deps aren't installed yet.
    from caption_worker import claim_caption_stage_job, run_caption_stage_job

    wid = f"caption-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_caption_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_caption_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def chunk_worker(worker_id: int, stop_event: mp.Event):
    # Import inside the process so uvicorn can boot even if deps aren't installed yet.
    from chunk_worker import claim_chunk_stage_job, run_chunk_stage_job

    wid = f"chunk-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_chunk_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_chunk_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def embed_worker(worker_id: int, stop_event: mp.Event):
    # Import inside the process to avoid loading torch/models in the parent.
    from embed_worker import claim_embedding_stage_job, run_embedding_stage_job

    wid = f"embed-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_embedding_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_embedding_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def cluster_worker(worker_id: int, stop_event: mp.Event):
    from cluster_worker import claim_clustering_stage_job, run_clustering_stage_job

    wid = f"cluster-{worker_id}"
    print(f"[{wid}] started")
    while not stop_event.is_set():
        stage_job = claim_clustering_stage_job(wid)
        if not stage_job:
            time.sleep(2)
            continue
        try:
            run_clustering_stage_job(stage_job)
        except Exception:
            time.sleep(1)
    print(f"[{wid}] stopped")


def start_worker_pool():
    global _POOL_STARTED
    if _POOL_STARTED:
        print("[worker-pool] already started in this process.")
        return mp.Event(), []

    if not _acquire_pool_lock():
        # Another pool is running; avoid spawning duplicates.
        return mp.Event(), []

    # CUDA + multiprocessing requires spawn on many platforms (and is the safest default in Colab).
    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass

    plan = auto_worker_plan()
    count = plan["sync_workers"]
    extract_count = int(os.getenv("EXTRACT_WORKERS", str(plan.get("extract_workers") or 2)))
    caption_count = int(os.getenv("CAPTION_WORKERS", "1"))
    chunk_count = int(os.getenv("CHUNK_WORKERS", "1"))
    embed_count = int(os.getenv("EMBED_WORKERS", str(plan.get("embed_workers") or 0)))
    # Clustering is currently disabled in Synapse (we may re-enable later).
    # Default to 0 so a missing env var never spawns unexpected cluster workers.
    cluster_count = int(os.getenv("CLUSTER_WORKERS", "0"))

    # For GPU stages, more processes often *reduces* throughput due to contention.
    # Default to a single layout worker when a GPU is present; allow env override.
    default_layout = 1 if plan.get("gpu") else 1
    layout_count = int(os.getenv("LAYOUT_WORKERS", str(default_layout)))
    print("Worker plan:", plan)
    print("Layout workers:", layout_count)
    print("Extract workers:", extract_count)
    print("Caption workers:", caption_count)
    print("Chunk workers:", chunk_count)
    print("Embed workers:", embed_count)
    print("Cluster workers:", cluster_count)

    stop_event = mp.Event()
    procs = []
    # One preprocess worker creates batches; N sync workers process batches in parallel.
    procs.append(mp.Process(target=preprocess_worker, args=(1, stop_event)))
    procs[-1].start()

    for i in range(count):
        p = mp.Process(target=sync_batch_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(layout_count):
        p = mp.Process(target=layout_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(extract_count):
        p = mp.Process(target=extraction_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(caption_count):
        p = mp.Process(target=caption_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(chunk_count):
        p = mp.Process(target=chunk_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(embed_count):
        p = mp.Process(target=embed_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)
    for i in range(cluster_count):
        p = mp.Process(target=cluster_worker, args=(i + 1, stop_event))
        p.start()
        procs.append(p)

    _POOL_STARTED = True
    return stop_event, procs

def stop_worker_pool(stop_event, procs):
    stop_event.set()
    for p in procs:
        p.join(timeout=5)


Overwriting worker_bootstrap.py


### layout_worker.py (Phase-3: layout_parser stage)


In [ ]:
%%writefile layout_worker.py
import io
import json
import os
import random
import queue
import tempfile
import threading
import time
from datetime import datetime, timezone

import boto3
import fitz  # PyMuPDF
import numpy as np
from PIL import Image
from dotenv import load_dotenv
from supabase import create_client

# Note: in Colab this file is written by the notebook via `%%writefile layout_worker.py`.
load_dotenv("/workspace/.env")

# Reduce CUDA memory fragmentation (must be set before importing torch in the worker).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

WORKER_ID = os.getenv("WORKER_ID", "layout-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    try:
        if exc.args and isinstance(exc.args[0], dict):
            code = str(exc.args[0].get("code") or "")
            details = str(exc.args[0].get("details") or "").lower()
            if code in {"502", "521", "429"}:
                return True
            if "bad gateway" in details or "web server is down" in details:
                return True
    except Exception:
        pass
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def _parallel_extraction_stages():
    # These stages can run in parallel after layout parsing.
    raw = os.getenv("EXTRACTION_PARALLEL_STAGES", "text_extraction,image_captioning")
    return [s.strip() for s in raw.split(",") if s.strip()]


def _ensure_stage_job_exists(
    org_id: str,
    library_id: str,
    batch_id: str,
    stage: str,
    progress_total: int,
):
    """
    Create the batch stage job iff it doesn't already exist.
    Avoids upsert resetting status for an already-running/done job.
    """
    existing = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).limit(1),
        context=f"batch_stage_jobs.select(exists:{stage})",
    )
    if existing.data:
        return
    _sb_execute(
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": int(progress_total or 0),
            }
        ),
        context=f"batch_stage_jobs.insert({stage})",
    )


def _enqueue_after_layout(org_id: str, library_id: str, batch_id: str, progress_total: int):
    """
    After layout_parser completes, fan-out into extraction stages in parallel.
    """
    stages = _pipeline_stages()
    try:
        idx = stages.index("layout_parser")
    except ValueError:
        return

    # Only stages that appear after layout_parser are considered.
    after = set(stages[idx + 1 :])
    fanout = [s for s in _parallel_extraction_stages() if s in after]
    if fanout:
        for st in fanout:
            _ensure_stage_job_exists(org_id, library_id, batch_id, st, progress_total)
        return

    # Fallback to sequential enqueue if no fanout stages are configured.
    if idx < len(stages) - 1:
        next_stage = stages[idx + 1]
        if next_stage:
            _ensure_stage_job_exists(org_id, library_id, batch_id, next_stage, progress_total)


# DocLayout-YOLO detector. Lazy-loaded on first job so uvicorn can boot cleanly.
_yolo_model = None
_yolo_device = None


def get_yolo():
    global _yolo_model, _yolo_device
    if _yolo_model is not None:
        return _yolo_model, _yolo_device

    try:
        import torch

        _yolo_device = "cuda:0" if torch.cuda.is_available() else "cpu"
    except Exception:
        _yolo_device = "cpu"

    from huggingface_hub import hf_hub_download
    from doclayout_yolo import YOLOv10

    repo_id = os.getenv("DOCLAYOUT_YOLO_REPO", "juliozhao/DocLayout-YOLO-DocStructBench")
    filename = os.getenv("DOCLAYOUT_YOLO_FILENAME", "doclayout_yolo_docstructbench_imgsz1024.pt")
    weights_path = hf_hub_download(repo_id=repo_id, filename=filename)

    _yolo_model = YOLOv10(weights_path)
    return _yolo_model, _yolo_device


def claim_layout_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "layout_parser")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(layout_parser.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]

    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(layout_parser.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def fetch_r2_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
    return obj["Body"].read()


def put_r2_json(key: str, payload: dict):
    s3.put_object(
        Bucket=R2_BUCKET,
        Key=key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )


def render_pages(pdf_bytes: bytes, max_pages: int | None = None):
    # Backwards-compatible helper (renders all pages). Prefer chunked rendering for large PDFs.
    scale = float(os.getenv("LAYOUT_RENDER_SCALE", "1.5"))
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    n = doc.page_count
    if max_pages is not None:
        n = min(n, max_pages)
    pages = []
    for i in range(n):
        page = doc.load_page(i)
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        pages.append((i, np.array(img)))
    return pages


def render_pages_chunked(pdf_bytes: bytes, chunk_pages: int):
    """
    Render pages in chunks to keep RAM stable and reduce peak memory for large PDFs.
    """
    scale = float(os.getenv("LAYOUT_RENDER_SCALE", "1.5"))
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    n = doc.page_count
    if chunk_pages <= 0:
        chunk_pages = 1

    for offset in range(0, n, chunk_pages):
        pages = []
        end = min(n, offset + chunk_pages)
        for i in range(offset, end):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            pages.append((i, np.array(img)))
        yield pages


def detect_layout(pages):
    model, device = get_yolo()
    page_indices = [p[0] for p in pages]

    imgsz = int(os.getenv("LAYOUT_IMGSZ", "768"))
    conf = float(os.getenv("LAYOUT_CONF", "0.2"))
    batch_pages = int(os.getenv("LAYOUT_BATCH_PAGES", "16"))
    if batch_pages <= 0:
        batch_pages = 1

    def _predict(inputs):
        kwargs = {"imgsz": imgsz, "conf": conf, "device": device}
        try:
            kwargs["half"] = bool(int(os.getenv("LAYOUT_HALF", "1")))
        except Exception:
            pass

        try:
            return model.predict(inputs, **kwargs)
        except TypeError:
            # Some wrappers don't accept `half`.
            kwargs.pop("half", None)
            return model.predict(inputs, **kwargs)

    results = []

    # Try in-memory inference first (much faster: avoids writing PNGs to disk).
    # If the wrapper doesn't support it, we fall back to temp PNG files.
    use_temp_files = False
    try:
        # Use PIL images to keep memory smaller than raw NumPy in some backends.
        sample_imgs = [Image.fromarray(p[1]) for p in pages[:1]]
        _ = _predict(sample_imgs)
    except Exception:
        use_temp_files = True

    if not use_temp_files:
        pil_images = [Image.fromarray(image_np) for _, image_np in pages]
        for offset in range(0, len(pil_images), batch_pages):
            chunk_imgs = pil_images[offset : offset + batch_pages]
            chunk_page_indices = page_indices[offset : offset + batch_pages]
            det_res = _predict(chunk_imgs)

            for page_index, r in zip(chunk_page_indices, det_res):
                blocks = []
                boxes = getattr(r, "boxes", None)
                names = getattr(r, "names", None)

                if boxes is not None and getattr(boxes, "xyxy", None) is not None:
                    xyxy = boxes.xyxy
                    confs = getattr(boxes, "conf", None)
                    clss = getattr(boxes, "cls", None)
                    try:
                        xyxy = xyxy.cpu().numpy()
                        confs = confs.cpu().numpy() if confs is not None else None
                        clss = clss.cpu().numpy().astype(int) if clss is not None else None
                    except Exception:
                        pass

                    for i, bb in enumerate(xyxy):
                        cls_id = int(clss[i]) if clss is not None else -1
                        label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                        score = float(confs[i]) if confs is not None else 1.0
                        x1, y1, x2, y2 = [float(v) for v in bb]
                        blocks.append({"type": label, "score": score, "bbox": [x1, y1, x2, y2]})

                results.append({"page": int(page_index), "blocks": blocks})

            try:
                import torch

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

        return results

    # Fallback: temp PNG files.
    with tempfile.TemporaryDirectory() as td:
        paths = []
        for page_index, image_np in pages:
            p = os.path.join(td, f"page_{page_index}.png")
            Image.fromarray(image_np).save(p)
            paths.append(p)

        for offset in range(0, len(paths), batch_pages):
            chunk_paths = paths[offset : offset + batch_pages]
            chunk_page_indices = page_indices[offset : offset + batch_pages]
            det_res = _predict(chunk_paths)

            for page_index, r in zip(chunk_page_indices, det_res):
                blocks = []
                boxes = getattr(r, "boxes", None)
                names = getattr(r, "names", None)

                if boxes is not None and getattr(boxes, "xyxy", None) is not None:
                    xyxy = boxes.xyxy
                    confs = getattr(boxes, "conf", None)
                    clss = getattr(boxes, "cls", None)
                    try:
                        xyxy = xyxy.cpu().numpy()
                        confs = confs.cpu().numpy() if confs is not None else None
                        clss = clss.cpu().numpy().astype(int) if clss is not None else None
                    except Exception:
                        pass

                    for i, bb in enumerate(xyxy):
                        cls_id = int(clss[i]) if clss is not None else -1
                        label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                        score = float(confs[i]) if confs is not None else 1.0
                        x1, y1, x2, y2 = [float(v) for v in bb]
                        blocks.append({"type": label, "score": score, "bbox": [x1, y1, x2, y2]})

                results.append({"page": int(page_index), "blocks": blocks})

            try:
                import torch

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

        return results


def detect_layout_pil(page_indices, pil_images):
    """
    Same as detect_layout(), but takes PIL images directly to avoid extra conversions and disk writes.
    Falls back to temp files if the wrapper doesn't accept in-memory images.
    """
    model, device = get_yolo()

    imgsz = int(os.getenv("LAYOUT_IMGSZ", "768"))
    conf = float(os.getenv("LAYOUT_CONF", "0.2"))
    batch_pages = int(os.getenv("LAYOUT_BATCH_PAGES", "16"))
    if batch_pages <= 0:
        batch_pages = 1

    def _predict(inputs):
        kwargs = {"imgsz": imgsz, "conf": conf, "device": device}
        try:
            kwargs["half"] = bool(int(os.getenv("LAYOUT_HALF", "1")))
        except Exception:
            pass
        try:
            return model.predict(inputs, **kwargs)
        except TypeError:
            kwargs.pop("half", None)
            return model.predict(inputs, **kwargs)

    results = []

    use_temp_files = False
    try:
        if pil_images:
            _ = _predict([pil_images[0]])
    except Exception:
        use_temp_files = True

    if not use_temp_files:
        for offset in range(0, len(pil_images), batch_pages):
            chunk_imgs = pil_images[offset : offset + batch_pages]
            chunk_page_indices = page_indices[offset : offset + batch_pages]
            det_res = _predict(chunk_imgs)

            for page_index, r in zip(chunk_page_indices, det_res):
                blocks = []
                boxes = getattr(r, "boxes", None)
                names = getattr(r, "names", None)

                if boxes is not None and getattr(boxes, "xyxy", None) is not None:
                    xyxy = boxes.xyxy
                    confs = getattr(boxes, "conf", None)
                    clss = getattr(boxes, "cls", None)
                    try:
                        xyxy = xyxy.cpu().numpy()
                        confs = confs.cpu().numpy() if confs is not None else None
                        clss = clss.cpu().numpy().astype(int) if clss is not None else None
                    except Exception:
                        pass

                    for i, bb in enumerate(xyxy):
                        cls_id = int(clss[i]) if clss is not None else -1
                        label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                        score = float(confs[i]) if confs is not None else 1.0
                        x1, y1, x2, y2 = [float(v) for v in bb]
                        blocks.append({"type": label, "score": score, "bbox": [x1, y1, x2, y2]})

                results.append({"page": int(page_index), "blocks": blocks})

            try:
                import torch

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

        return results

    # Fallback: temp PNG files.
    with tempfile.TemporaryDirectory() as td:
        paths = []
        for page_index, img in zip(page_indices, pil_images):
            p = os.path.join(td, f"page_{page_index}.png")
            img.save(p)
            paths.append(p)

        for offset in range(0, len(paths), batch_pages):
            chunk_paths = paths[offset : offset + batch_pages]
            chunk_page_indices = page_indices[offset : offset + batch_pages]
            det_res = _predict(chunk_paths)

            for page_index, r in zip(chunk_page_indices, det_res):
                blocks = []
                boxes = getattr(r, "boxes", None)
                names = getattr(r, "names", None)

                if boxes is not None and getattr(boxes, "xyxy", None) is not None:
                    xyxy = boxes.xyxy
                    confs = getattr(boxes, "conf", None)
                    clss = getattr(boxes, "cls", None)
                    try:
                        xyxy = xyxy.cpu().numpy()
                        confs = confs.cpu().numpy() if confs is not None else None
                        clss = clss.cpu().numpy().astype(int) if clss is not None else None
                    except Exception:
                        pass

                    for i, bb in enumerate(xyxy):
                        cls_id = int(clss[i]) if clss is not None else -1
                        label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                        score = float(confs[i]) if confs is not None else 1.0
                        x1, y1, x2, y2 = [float(v) for v in bb]
                        blocks.append({"type": label, "score": score, "bbox": [x1, y1, x2, y2]})

                results.append({"page": int(page_index), "blocks": blocks})

            try:
                import torch

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

        return results


def detect_layout_for_pdf_streaming(pdf_bytes: bytes):
    """
    Producer/consumer pipeline:
    - Producer thread renders PDF pages to PIL images in chunks and pushes into a bounded queue.
    - Consumer (GPU) batches those chunks and runs YOLO, overlapping CPU render with GPU inference.
    """
    chunk_pages = int(os.getenv("LAYOUT_RENDER_CHUNK_PAGES", "8"))
    if chunk_pages <= 0:
        chunk_pages = 8

    max_queue_chunks = int(os.getenv("LAYOUT_PIPELINE_QUEUE_CHUNKS", "6"))
    if max_queue_chunks <= 0:
        max_queue_chunks = 6

    q: queue.Queue = queue.Queue(maxsize=max_queue_chunks)
    sentinel = object()

    def producer():
        try:
            scale = float(os.getenv("LAYOUT_RENDER_SCALE", "1.5"))
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")
            n = doc.page_count
            for offset in range(0, n, chunk_pages):
                end = min(n, offset + chunk_pages)
                idxs = []
                imgs = []
                for i in range(offset, end):
                    page = doc.load_page(i)
                    pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
                    img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
                    idxs.append(i)
                    imgs.append(img)
                q.put((idxs, imgs))
        finally:
            q.put(sentinel)

    t = threading.Thread(target=producer, daemon=True)
    t.start()

    layout_all = []
    while True:
        item = q.get()
        if item is sentinel:
            break
        idxs, imgs = item
        layout_all.extend(detect_layout_pil(idxs, imgs))

    t.join(timeout=30)
    return layout_all


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["layout_parser"]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _update_library_progress(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        res = _sb_execute(
            supabase.table("batch_stage_jobs")
            .select("id", count="exact")
            .eq("library_id", library_id)
            .eq("stage", st)
            .eq("status", "done"),
            context=f"batch_stage_jobs.count(done:{st})",
        )
        done_total += int(res.count or 0)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)

    # `completed_batches` represents fully-processed batches (i.e. reached the last stage),
    # not "batches completed in the current stage".
    completed_batches = _count_done_stage_jobs(library_id, _pipeline_stages()[-1])

    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": "layout_parser",
                "completed_batches": completed_batches,
                "pipeline_progress_percent": progress,
            }
        ).eq("id", library_id),
        context="libraries.update(layout_progress)",
    )


def _maybe_finalize_pipeline(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.finalize)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    for st in stages:
        res = _sb_execute(
            supabase.table("batch_stage_jobs")
            .select("id", count="exact")
            .eq("library_id", library_id)
            .eq("stage", st)
            .eq("status", "done"),
            context=f"batch_stage_jobs.count(done:{st}.finalize)",
        )
        if int(res.count or 0) < total_batches:
            return

    finished = now_iso()
    _sb_execute(
        supabase.table("libraries").update(
            {
                "status": "ready",
                "pipeline_status": "completed",
                "pipeline_stage": stages[-1],
                "pipeline_progress_percent": 100,
                "pipeline_error": None,
                "pipeline_finished_at": finished,
                "completed_batches": total_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline.completed)",
    )


def run_layout_stage_job(stage_job):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    batch = _sb_execute(
        supabase.table("library_batches").select("doc_ids, doc_count").eq("id", batch_id).single(),
        context="library_batches.select(doc_ids)",
    )
    doc_ids = (batch.data or {}).get("doc_ids") or []
    total = int(stage_job.get("progress_total") or (batch.data or {}).get("doc_count") or len(doc_ids) or 0)
    current = int(stage_job.get("progress_current") or 0)

    _sb_execute(
        supabase.table("libraries").update({"pipeline_status": "running", "pipeline_stage": "layout_parser"}).eq(
            "id", library_id
        ),
        context="libraries.update(stage=layout_parser)",
    )

    try:
        for doc_id in doc_ids:
            # stop early if canceled
            lib_check = _sb_execute(
                supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
                context="libraries.select(cancel_check)",
            )
            if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
                return

            doc = _sb_execute(
                supabase.table("documents").select("id, storage_path_raw, mime_type").eq("id", doc_id).single(),
                context="documents.select(storage_path_raw)",
            )
            if not doc.data:
                continue
            key = doc.data.get("storage_path_raw")
            if not key:
                continue
            mime = (doc.data.get("mime_type") or "").lower()
            if "pdf" not in mime:
                current += 1
                _sb_execute(
                    supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                        "id", job_id
                    ),
                    context="batch_stage_jobs.update(progress.nonpdf)",
                )
                continue

            pdf_bytes = fetch_r2_bytes(key)
            layout = detect_layout_for_pdf_streaming(pdf_bytes)

            out_key = f"layout/{org_id}/{library_id}/{doc_id}.json"
            # Persist render_scale so downstream stages (cropping / bbox->pdf conversions) stay consistent.
            put_r2_json(
                out_key,
                {
                    "doc_id": doc_id,
                    "library_id": library_id,
                    "organization_id": org_id,
                    "created_at": now_iso(),
                    "render_scale": float(os.getenv("LAYOUT_RENDER_SCALE", "1.5")),
                    "layout": layout,
                },
            )

            current += 1
            _sb_execute(
                supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                    "id", job_id
                ),
                context="batch_stage_jobs.update(progress)",
            )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {
                    "status": "done",
                    "finished_at": now_iso(),
                    "progress_current": total,
                    "progress_total": total,
                }
            ).eq("id", job_id),
            context="batch_stage_jobs.update(done)",
        )

        # Enqueue extraction fanout (text_extraction + image_captioning) in parallel.
        _enqueue_after_layout(org_id=org_id, library_id=library_id, batch_id=batch_id, progress_total=total)

        _update_library_progress(library_id)
        _maybe_finalize_pipeline(library_id)

    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(failed)",
        )
        _sb_execute(
            supabase.table("libraries").update(
                {
                    "pipeline_status": "failed",
                    "pipeline_stage": "layout_parser",
                    "pipeline_error": str(exc),
                    "status": "error",
                }
            ).eq("id", library_id),
            context="libraries.update(layout_failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("LAYOUT_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_layout_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No layout jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_layout_stage_job(job)


if __name__ == "__main__":
    worker_loop()


Overwriting layout_worker.py


### extraction_bootstrap.py


In [ ]:
%%writefile extraction_bootstrap.py
import os
import subprocess

WORKERS = int(os.getenv("EXTRACT_WORKERS", "3"))
LIB_ID = os.getenv("LIB_ID")  # optional: restrict to one library

procs = []
for i in range(1, WORKERS + 1):
    env = os.environ.copy()
    env["WORKER_ID"] = f"worker-{i}"
    if LIB_ID:
        env["LIB_ID"] = LIB_ID
    procs.append(
        subprocess.Popen(["python", "extraction_worker.py"], env=env)
    )

for p in procs:
    p.wait()


Overwriting extraction_bootstrap.py


### extraction_worker.py


In [ ]:
%%writefile extraction_worker.py
import json
import os
import random
import time
from datetime import datetime, timezone

import boto3
import fitz  # PyMuPDF
from dotenv import load_dotenv
from supabase import create_client

# Note: in Colab this file is written by the notebook via `%%writefile extraction_worker.py`.
load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "extract-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    try:
        if exc.args and isinstance(exc.args[0], dict):
            code = str(exc.args[0].get("code") or "")
            details = str(exc.args[0].get("details") or "").lower()
            if code in {"502", "521", "429"}:
                return True
            if "bad gateway" in details or "web server is down" in details:
                return True
    except Exception:
        pass
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def fetch_r2_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
    return obj["Body"].read()


def fetch_r2_json(key: str) -> dict | None:
    try:
        raw = fetch_r2_bytes(key)
    except Exception:
        return None
    try:
        return json.loads(raw.decode("utf-8"))
    except Exception:
        return None


def put_r2_json(key: str, payload: dict):
    s3.put_object(
        Bucket=R2_BUCKET,
        Key=key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["text_extraction"]


def _parallel_extraction_stages():
    raw = os.getenv("EXTRACTION_PARALLEL_STAGES", "text_extraction,image_captioning")
    return [s.strip() for s in raw.split(",") if s.strip()]


def _ensure_stage_job_exists(
    org_id: str,
    library_id: str,
    batch_id: str,
    stage: str,
    progress_total: int,
):
    existing = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).limit(1),
        context=f"batch_stage_jobs.select(exists:{stage})",
    )
    if existing.data:
        return
    _sb_execute(
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": int(progress_total or 0),
            }
        ),
        context=f"batch_stage_jobs.insert({stage})",
    )


def _count_batch_stage_done(batch_id: str, stage: str) -> bool:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).eq("status", "done").limit(1),
        context=f"batch_stage_jobs.select(done:{stage})",
    )
    return bool(resp.data)


def _maybe_enqueue_next_after_parallel(org_id: str, library_id: str, batch_id: str, progress_total: int):
    """
    If this batch is part of the parallel extraction fanout, only enqueue the next stage
    after *all* parallel extraction stages are done.
    """
    stages = _pipeline_stages()
    parallel = [s for s in _parallel_extraction_stages() if s in stages]
    if not parallel:
        return

    # Only coordinate when both stages exist in the pipeline.
    if any(not _count_batch_stage_done(batch_id, st) for st in parallel):
        return

    # Enqueue the first stage that appears after the last parallel stage occurrence.
    last_idx = max(stages.index(st) for st in parallel)
    if last_idx < len(stages) - 1:
        next_stage = stages[last_idx + 1]
        if next_stage:
            _ensure_stage_job_exists(org_id, library_id, batch_id, next_stage, progress_total)


def claim_text_extraction_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "text_extraction")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(text_extraction.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]

    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(text_extraction.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _stage_order(stage: str) -> int:
    stages = _pipeline_stages()
    try:
        return stages.index(stage)
    except ValueError:
        return 10_000


def _compute_next_stage(library_id: str) -> str:
    remaining = _sb_execute(
        supabase.table("batch_stage_jobs").select("stage, status").eq("library_id", library_id).neq("status", "done"),
        context="batch_stage_jobs.select(remaining)",
    )
    stages = [str(r.get("stage") or "") for r in (remaining.data or []) if isinstance(r, dict)]
    stages = [s for s in stages if s]
    if not stages:
        return _pipeline_stages()[-1]
    stages.sort(key=_stage_order)
    return stages[0]


def _update_library_progress(library_id: str, stage: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)

    # `completed_batches` represents fully-processed batches (i.e. reached the last stage),
    # not "batches completed in the current stage".
    completed_batches = _count_done_stage_jobs(library_id, _pipeline_stages()[-1])

    next_stage = _compute_next_stage(library_id)
    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": next_stage,
                "completed_batches": completed_batches,
                "pipeline_progress_percent": progress,
            }
        ).eq("id", library_id),
        context="libraries.update(extraction_progress)",
    )


def _maybe_finalize_pipeline(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.finalize)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    for st in stages:
        if _count_done_stage_jobs(library_id, st) < total_batches:
            return

    finished = now_iso()
    _sb_execute(
        supabase.table("libraries").update(
            {
                "status": "ready",
                "pipeline_status": "completed",
                "pipeline_stage": stages[-1],
                "pipeline_progress_percent": 100,
                "pipeline_error": None,
                "pipeline_finished_at": finished,
                "completed_batches": total_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline.completed)",
    )


def _normalize_type(t: str) -> str:
    t = (t or "").strip().lower()
    # DocLayout-YOLO / DocStructBench labels can be a bit different than LayoutParser defaults.
    # Treat any text-like label as "text" so we actually extract most paragraphs.
    if t in {
        "text",
        "plain text",
        "title",
        "section header",
        "header",
        "footer",
        "caption",
        "footnote",
        "list",
        "reference",
        "references",
    }:
        return "text"
    if t in {"abandon", "ignore", "background"}:
        return "ignore"
    if t in {"table"}:
        return "table"
    if t in {"figure", "image", "graph"}:
        return "figure"
    return t or "unknown"


def _bbox_img_to_pdf(bbox_img, render_scale: float):
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox_img]
    except Exception:
        return None
    if render_scale <= 0:
        render_scale = 1.0
    return [x1 / render_scale, y1 / render_scale, x2 / render_scale, y2 / render_scale]


def _compute_links(page_blocks):
    # Simple heuristic: link each figure/table to the nearest text block beneath it with column overlap.
    # Returns list of link dicts.
    links = []
    text_blocks = [b for b in page_blocks if b.get("kind") == "text" and (b.get("text") or "").strip()]
    visual_blocks = [b for b in page_blocks if b.get("kind") in {"figure", "table"}]

    for vb in visual_blocks:
        vbbox = vb.get("bbox_pdf") or vb.get("bbox")
        if not vbbox:
            continue
        vx1, vy1, vx2, vy2 = vbbox
        best = None
        best_dy = None
        for tb in text_blocks:
            tbbox = tb.get("bbox_pdf") or tb.get("bbox")
            if not tbbox:
                continue
            tx1, ty1, tx2, ty2 = tbbox
            # must be below (or slightly above) and overlap in x (same column)
            overlap_x = min(vx2, tx2) - max(vx1, tx1)
            if overlap_x <= 0:
                continue
            dy = ty1 - vy2
            if dy < -10:  # allow tiny overlap, but avoid far-above paragraphs
                continue
            if dy > 200:  # too far away to be a caption
                continue
            if best_dy is None or dy < best_dy:
                best = tb
                best_dy = dy
        if best:
            links.append(
                {
                    "visual_block_id": vb["block_id"],
                    "caption_block_id": best["block_id"],
                    "relation": "caption",
                    "page": vb.get("page"),
                    "score": float(max(0.0, 1.0 - (best_dy or 0) / 200.0)),
                }
            )

    return links


def run_text_extraction_stage_job(stage_job):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    # Stop early if canceled
    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    batch = _sb_execute(
        supabase.table("library_batches").select("doc_ids, doc_count").eq("id", batch_id).single(),
        context="library_batches.select(doc_ids)",
    )
    doc_ids = (batch.data or {}).get("doc_ids") or []
    total = int(stage_job.get("progress_total") or (batch.data or {}).get("doc_count") or len(doc_ids) or 0)
    current = int(stage_job.get("progress_current") or 0)

    _sb_execute(
        supabase.table("libraries").update({"pipeline_status": "running", "pipeline_stage": "text_extraction"}).eq(
            "id", library_id
        ),
        context="libraries.update(stage=text_extraction)",
    )

    # Load docs metadata for the batch in chunks (avoid 1 query per doc).
    docs_by_id: dict[str, dict] = {}
    fetch_chunk = int(os.getenv("DOC_FETCH_CHUNK", "100"))
    for i in range(0, len(doc_ids), fetch_chunk):
        chunk = doc_ids[i : i + fetch_chunk]
        resp = _sb_execute(
            # Include NOT NULL columns so our bulk upsert never attempts to insert a partial row.
            supabase.table("documents").select(
                "id, organization_id, library_id, title, gdrive_file_id, mime_type, file_size_bytes, status, storage_path_raw"
            ).in_("id", chunk),
            context="documents.select(batch)",
        )
        for d in resp.data or []:
            docs_by_id[d["id"]] = d

    progress_every = int(os.getenv("STAGE_PROGRESS_EVERY", "3"))
    min_chars = int(os.getenv("EXTRACT_MIN_CHARS", "20"))

    doc_updates = []
    doc_update_chunk = int(os.getenv("DOC_TEXT_UPSERT_CHUNK", "50"))

    try:
        for doc_id in doc_ids:
            d = docs_by_id.get(doc_id) or {}
            key = d.get("storage_path_raw")
            mime = (d.get("mime_type") or "").lower()
            if not key or ("pdf" not in mime):
                current += 1
                if current % progress_every == 0:
                    _sb_execute(
                        supabase.table("batch_stage_jobs").update(
                            {"progress_current": current, "progress_total": total}
                        ).eq("id", job_id),
                        context="batch_stage_jobs.update(progress)",
                    )
                continue

            pdf_bytes = fetch_r2_bytes(key)
            doc = fitz.open(stream=pdf_bytes, filetype="pdf")

            # Load layout artifact (optional but recommended).
            layout_key = f"layout/{org_id}/{library_id}/{doc_id}.json"
            layout = fetch_r2_json(layout_key) or {}
            render_scale = float(layout.get("render_scale") or os.getenv("LAYOUT_RENDER_SCALE", "1.5"))

            pages_layout = layout.get("layout") or []
            by_page = {int(p.get("page")): p for p in pages_layout if isinstance(p, dict) and "page" in p}

            out_pages = []
            all_links = []

            for page_index in range(doc.page_count):
                page = doc.load_page(page_index)
                page_info = by_page.get(page_index) or {}
                blocks = page_info.get("blocks") or []
                out_blocks = []

                for bi, b in enumerate(blocks):
                    btype = _normalize_type(b.get("type") if isinstance(b, dict) else "")
                    if btype == "ignore":
                        continue
                    bbox_img = (b.get("bbox") if isinstance(b, dict) else None) or None
                    bbox_pdf = _bbox_img_to_pdf(bbox_img, render_scale) if bbox_img else None

                    block_id = f"p{page_index}_b{bi}"
                    rec = {
                        "block_id": block_id,
                        "page": page_index,
                        "type": b.get("type") if isinstance(b, dict) else None,
                        "kind": btype,
                        "score": float((b.get("score") or 1.0) if isinstance(b, dict) else 1.0),
                        "bbox_img": bbox_img,
                        "bbox_pdf": bbox_pdf,
                    }

                    if btype == "text" and bbox_pdf:
                        x1, y1, x2, y2 = bbox_pdf
                        clip = fitz.Rect(x1, y1, x2, y2)
                        txt = (page.get_text("text", clip=clip) or "").strip()
                        rec["text"] = txt
                        rec["char_count"] = len(txt)
                        rec["needs_ocr"] = len(txt) < min_chars
                    elif btype in {"figure", "table"}:
                        rec["needs_caption"] = True
                    out_blocks.append(rec)

                out_pages.append({"page": page_index, "blocks": out_blocks})
                all_links.extend(_compute_links(out_blocks))

            out_key = f"text/{org_id}/{library_id}/{doc_id}.json"
            put_r2_json(
                out_key,
                {
                    "doc_id": doc_id,
                    "library_id": library_id,
                    "organization_id": org_id,
                    "created_at": now_iso(),
                    "render_scale": render_scale,
                    "layout_key": layout_key,
                    "source_pdf_key": key,
                    "pages": out_pages,
                    "links": all_links,
                },
            )

            # Use upsert for batch efficiency, but include required columns so we never attempt to insert
            # a partial row (which would violate NOT NULL constraints like organization_id/library_id/title).
            doc_updates.append(
                {
                    "id": doc_id,
                    "organization_id": d.get("organization_id") or org_id,
                    "library_id": d.get("library_id") or library_id,
                    "title": d.get("title") or doc_id,
                    "gdrive_file_id": d.get("gdrive_file_id"),
                    "mime_type": d.get("mime_type"),
                    "file_size_bytes": d.get("file_size_bytes"),
                    "status": d.get("status") or "pending",
                    "storage_path_raw": d.get("storage_path_raw") or key,
                    "storage_path_text": out_key,
                }
            )
            if len(doc_updates) >= doc_update_chunk:
                _sb_execute(
                    supabase.table("documents").upsert(doc_updates, on_conflict="id"),
                    context="documents.upsert(storage_path_text)",
                )
                doc_updates = []

            current += 1
            if current % progress_every == 0:
                _sb_execute(
                    supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                        "id", job_id
                    ),
                    context="batch_stage_jobs.update(progress)",
                )

        if doc_updates:
            _sb_execute(
                supabase.table("documents").upsert(doc_updates, on_conflict="id"),
                context="documents.upsert(storage_path_text.final)",
            )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "done", "finished_at": now_iso(), "progress_current": total, "progress_total": total}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(done)",
        )

        # Captioning runs in parallel with text extraction; only fan-in to next stage when both are done.
        _maybe_enqueue_next_after_parallel(org_id=org_id, library_id=library_id, batch_id=batch_id, progress_total=total)

        _update_library_progress(library_id, stage="text_extraction")
        _maybe_finalize_pipeline(library_id)

    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(failed)",
        )
        _sb_execute(
            supabase.table("libraries").update(
                {
                    "pipeline_status": "failed",
                    "pipeline_stage": "text_extraction",
                    "pipeline_error": str(exc),
                    "status": "error",
                }
            ).eq("id", library_id),
            context="libraries.update(text_extraction.failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("EXTRACT_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_text_extraction_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No extraction jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_text_extraction_stage_job(job)


if __name__ == "__main__":
    worker_loop()


Overwriting extraction_worker.py


# caption worker

In [ ]:
%%writefile caption_worker.py
import io
import json
import os
import random
import re
import csv
import time
from datetime import datetime, timezone
from typing import Any, Optional, Tuple, List, Dict
import statistics

import boto3
import fitz  # PyMuPDF
from PIL import Image
from dotenv import load_dotenv
from supabase import create_client

load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "caption-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    try:
        if exc.args and isinstance(exc.args[0], dict):
            code = str(exc.args[0].get("code") or "")
            details = str(exc.args[0].get("details") or "").lower()
            if code in {"502", "521", "429"}:
                return True
            if "bad gateway" in details or "web server is down" in details:
                return True
    except Exception:
        pass
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def fetch_r2_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
    return obj["Body"].read()


def fetch_r2_json(key: str) -> dict | None:
    try:
        raw = fetch_r2_bytes(key)
    except Exception:
        return None
    try:
        return json.loads(raw.decode("utf-8"))
    except Exception:
        return None


def put_r2_json(key: str, payload: dict):
    s3.put_object(
        Bucket=R2_BUCKET,
        Key=key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )


def put_r2_png(key: str, image: Image.Image):
    buf = io.BytesIO()
    image.save(buf, format="PNG", optimize=True)
    s3.put_object(Bucket=R2_BUCKET, Key=key, Body=buf.getvalue(), ContentType="image/png")


def put_r2_text(key: str, text: str, content_type: str = "text/plain; charset=utf-8"):
    s3.put_object(Bucket=R2_BUCKET, Key=key, Body=(text or "").encode("utf-8"), ContentType=content_type)


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["image_captioning"]


def _parallel_extraction_stages():
    raw = os.getenv("EXTRACTION_PARALLEL_STAGES", "text_extraction,image_captioning")
    return [s.strip() for s in raw.split(",") if s.strip()]


def _ensure_stage_job_exists(
    org_id: str,
    library_id: str,
    batch_id: str,
    stage: str,
    progress_total: int,
):
    existing = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).limit(1),
        context=f"batch_stage_jobs.select(exists:{stage})",
    )
    if existing.data:
        return
    _sb_execute(
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": int(progress_total or 0),
            }
        ),
        context=f"batch_stage_jobs.insert({stage})",
    )


def _count_batch_stage_done(batch_id: str, stage: str) -> bool:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id")
        .eq("batch_id", batch_id)
        .eq("stage", stage)
        .eq("status", "done")
        .limit(1),
        context=f"batch_stage_jobs.select(done:{stage})",
    )
    return bool(resp.data)


def _maybe_enqueue_next_after_parallel(org_id: str, library_id: str, batch_id: str, progress_total: int):
    stages = _pipeline_stages()
    parallel = [s for s in _parallel_extraction_stages() if s in stages]
    if not parallel:
        return
    if any(not _count_batch_stage_done(batch_id, st) for st in parallel):
        return

    last_idx = max(stages.index(st) for st in parallel)
    if last_idx < len(stages) - 1:
        next_stage = stages[last_idx + 1]
        if next_stage:
            _ensure_stage_job_exists(org_id, library_id, batch_id, next_stage, progress_total)


def _normalize_type(t: str) -> str:
    t = (t or "").strip().lower()
    if t in {"abandon", "ignore", "background"}:
        return "ignore"
    # DocLayout-YOLO may emit `figure_caption` / `table_caption`. Those are text regions,
    # not visuals we should crop/OCR; we already link captions via text_extraction.
    if "caption" in t and ("figure" in t or "table" in t):
        return "ignore"
    if "table" in t:
        return "table"
    if "equation" in t or "formula" in t or t in {"math"}:
        return "formula"
    if "figure" in t or "image" in t or "graph" in t or "chart" in t or "picture" in t:
        return "figure"
    return "other"


_CAPTION_RE = re.compile(r"^(fig(?:ure)?|table)\s*[\.:]?\s*\d+", re.IGNORECASE)
_FIG_MENTION_RE = re.compile(r"\bfig(?:\.|ure)?\s*(\d+)\b", re.IGNORECASE)
_TABLE_MENTION_RE = re.compile(r"\btable\s*(\d+)\b", re.IGNORECASE)
_NUM_TOKEN_RE = re.compile(r"(?<!\w)(\$?\d+(?:[\.,]\d+)*(?:%|x|k|m|b)?)(?!\w)", re.IGNORECASE)
_UNIT_RE = re.compile(r"\b(usd|eur|gbp|pkr|rs|inr|million|billion|bn|mm|%|percent)\b", re.IGNORECASE)


def _clean_text(s: str) -> str:
    s = (s or "").replace("\x00", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _extract_ref_number(kind: str, text: str) -> Optional[str]:
    t = (text or "").strip()
    if not t:
        return None
    if kind == "table":
        m = _TABLE_MENTION_RE.search(t)
        return m.group(1) if m else None
    # default to figure
    m = _FIG_MENTION_RE.search(t)
    return m.group(1) if m else None


def _collect_ref_numbers(kind: str, texts: List[str]) -> List[str]:
    out: List[str] = []
    for s in texts:
        n = _extract_ref_number(kind, s)
        if n:
            out.append(n)
    return out


def _is_heading_like(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return False
    # crude heuristic: very short uppercase or title-like with no punctuation
    if len(t) <= 40 and t.upper() == t and any(ch.isalpha() for ch in t):
        return True
    if len(t) <= 25 and t.endswith(":"):
        return True
    return False


def _merge_caption_blocks(
    blocks: List[dict],
    start_block_id: str,
    max_blocks: int = 4,
) -> Optional[str]:
    """
    Given a list of text blocks (each dict should include block_id, bbox_img, text),
    merge consecutive caption blocks to handle multi-block captions.
    """
    # Sort blocks by y then x for stable reading order.
    b2 = []
    for b in blocks:
        if not isinstance(b, dict):
            continue
        if (b.get("kind") or "") != "text":
            continue
        bid = b.get("block_id")
        bb = b.get("bbox_img")
        txt = _clean_text(str(b.get("text") or ""))
        if not bid or not bb or not txt:
            continue
        try:
            x1, y1, x2, y2 = [float(v) for v in bb]
        except Exception:
            continue
        b2.append((bid, fitz.Rect(x1, y1, x2, y2), txt))
    b2.sort(key=lambda t: (t[1].y0, t[1].x0))

    idx = None
    for i, (bid, _r, _txt) in enumerate(b2):
        if bid == start_block_id:
            idx = i
            break
    if idx is None:
        return None

    merged = [b2[idx][2]]
    base_rect = b2[idx][1]
    for j in range(idx + 1, min(len(b2), idx + max_blocks)):
        bid, r, txt = b2[j]
        # Stop if new caption starts or looks like a section heading.
        if _CAPTION_RE.match(txt) or _is_heading_like(txt):
            break
        # Keep only blocks close below and with decent x-overlap.
        dy = r.y0 - base_rect.y1
        if dy < -2:
            break
        if dy > float(os.getenv("VIS_CAPTION_MERGE_MAX_DY", "80")):
            break
        if _rect_x_overlap(base_rect, r) < float(os.getenv("VIS_CAPTION_MERGE_MIN_XO", "0.25")):
            break
        merged.append(txt)
        base_rect = r
    out = _clean_text(" ".join(merged))
    return out or None


def _pick_best_caption_candidate(
    kind: str,
    caption_candidates: List[dict],
    prefer_numbers: List[str],
) -> Tuple[Optional[str], List[dict]]:
    """
    Choose best caption among candidates, preferring number match when available.
    caption_candidates items may include text_snippet/text.
    """
    if not caption_candidates:
        return None, []

    # Extract candidate texts
    cands = []
    for c in caption_candidates:
        if not isinstance(c, dict):
            continue
        txt = _clean_text(str(c.get("text") or c.get("text_snippet") or ""))
        if not txt:
            continue
        n = _extract_ref_number(kind, txt)
        cands.append((c, txt, n))

    if not cands:
        return None, []

    if prefer_numbers:
        for num in prefer_numbers:
            for c, txt, n in cands:
                if n == num:
                    return txt, caption_candidates

    # Fallback: first candidate already sorted by score upstream
    return cands[0][1], caption_candidates


def _cross_page_caption_fallback(
    text_doc: Optional[dict],
    page_index: int,
    kind: str,
) -> Tuple[Optional[str], List[dict]]:
    """
    If caption isn't on the same page, scan next page top-area blocks for caption-like text.
    Uses text_extraction artifact when available.
    """
    if not text_doc:
        return None, []
    pages = text_doc.get("pages") or []
    by_page = {int(p.get("page")): p for p in pages if isinstance(p, dict) and "page" in p}
    nxt = by_page.get(int(page_index) + 1) or None
    if not nxt:
        return None, []
    blocks = nxt.get("blocks") or []
    top_y = float(os.getenv("VIS_CROSSPAGE_TOP_Y_PX", "220"))
    cands = []
    for b in blocks:
        if not isinstance(b, dict):
            continue
        if (b.get("kind") or "") != "text":
            continue
        bb = b.get("bbox_img")
        if not bb:
            continue
        try:
            _x1, y1, _x2, _y2 = [float(v) for v in bb]
        except Exception:
            continue
        if y1 > top_y:
            continue
        txt = _clean_text(str(b.get("text") or ""))
        if not txt:
            continue
        if _CAPTION_RE.match(txt) or (_extract_ref_number(kind, txt) is not None):
            cands.append({"block_id": b.get("block_id"), "text": txt, "score": 0.0})
    if not cands:
        return None, []
    return _clean_text(cands[0]["text"]), cands[:3]


def _infer_doc_profile(text_doc: Optional[dict], pages_layout: List[dict]) -> str:
    """
    Lightweight doc profiling:
    - scanned: low text density
    - financial_report: sparse Fig/Table mentions but many large visuals
    - digital_text_rich: default
    """
    sample_pages = int(os.getenv("VIS_PROFILE_PAGES", "5"))

    # Text density
    text_chars = 0
    pages_count = 0
    mentions = 0
    if text_doc:
        for p in (text_doc.get("pages") or [])[:sample_pages]:
            pages_count += 1
            for b in (p.get("blocks") or []):
                if isinstance(b, dict) and (b.get("kind") == "text"):
                    t = str(b.get("text") or "")
                    text_chars += len(t)
                    low = t.lower()
                    if "fig" in low or "figure" in low or "table" in low:
                        mentions += 1

    avg_chars = (text_chars / max(1, pages_count)) if pages_count else 0
    if avg_chars < int(os.getenv("VIS_SCANNED_CHARS_PER_PAGE", "600")):
        return "scanned"

    # Large visual frequency
    large_pages = 0
    checked = 0
    for p in pages_layout[:sample_pages]:
        checked += 1
        blocks = p.get("blocks") or []
        big = 0
        for b in blocks:
            if not isinstance(b, dict):
                continue
            t = _normalize_type(b.get("type") or "")
            if t not in {"figure", "table"}:
                continue
            bb = b.get("bbox")
            if not bb:
                continue
            try:
                x1, y1, x2, y2 = [float(v) for v in bb]
            except Exception:
                continue
            area = max(0.0, (x2 - x1)) * max(0.0, (y2 - y1))
            if area >= float(os.getenv("VIS_FIN_LARGE_BBOX_AREA_PX", "35000")):
                big += 1
        if big >= int(os.getenv("VIS_FIN_LARGE_VISUALS_PER_PAGE", "2")):
            large_pages += 1

    if checked and (mentions <= int(os.getenv("VIS_FIN_MENTIONS_MAX", "1"))) and (large_pages / checked) >= 0.4:
        return "financial_report"
    return "digital_text_rich"


def _rect_x_overlap(a: fitz.Rect, b: fitz.Rect) -> float:
    inter = max(0.0, min(a.x1, b.x1) - max(a.x0, b.x0))
    denom = max(1.0, a.width)
    return float(inter / denom)


def _best_caption_near_bbox(page: fitz.Page, bbox_pdf: fitz.Rect, kind: str) -> tuple[str | None, list[dict]]:
    """
    Heuristic caption finder:
    - scans page text blocks
    - prefers blocks directly below the visual bbox, with x-overlap
    - boosts blocks that look like "Figure 2:" / "Table 1."
    Returns (best_caption, candidates_debug).
    """
    try:
        blocks = page.get_text("blocks") or []
    except Exception:
        blocks = []

    candidates: list[tuple[float, str, fitz.Rect]] = []
    debug: list[dict] = []

    for b in blocks:
        try:
            x0, y0, x1, y1, text = float(b[0]), float(b[1]), float(b[2]), float(b[3]), str(b[4] or "")
        except Exception:
            continue
        text = _clean_text(text)
        if not text:
            continue

        r = fitz.Rect(x0, y0, x1, y1)
        # Distance: prefer below; penalize above.
        dy_below = r.y0 - bbox_pdf.y1
        dy_above = bbox_pdf.y0 - r.y1
        if dy_below >= 0:
            dy = dy_below
        else:
            dy = abs(dy_above) * 1.6

        xo = _rect_x_overlap(bbox_pdf, r)
        xo_pen = (1.0 - xo) * 140.0

        looks_like_caption = bool(_CAPTION_RE.match(text))
        kind_hint = kind in text.lower()
        boost = 0.0
        if looks_like_caption:
            boost -= 60.0
        if kind_hint:
            boost -= 20.0

        score = float(dy + xo_pen + boost)
        # Ignore very far candidates.
        if score > 800:
            continue
        candidates.append((score, text, r))
        debug.append({"score": round(score, 2), "text": text[:220], "rect": [x0, y0, x1, y1]})

    candidates.sort(key=lambda t: t[0])
    best = candidates[0][1] if candidates else None
    return best, debug[:10]


def _score_block_to_visual_bbox_img(visual_bbox_img: List[float], block_bbox_img: List[float]) -> float:
    """
    Score how likely a text block is the caption/related text for a visual block.
    Lower is better.
    """
    try:
        vx1, vy1, vx2, vy2 = [float(v) for v in visual_bbox_img]
        bx1, by1, bx2, by2 = [float(v) for v in block_bbox_img]
    except Exception:
        return 1e9

    v = fitz.Rect(vx1, vy1, vx2, vy2)
    b = fitz.Rect(bx1, by1, bx2, by2)

    dy_below = b.y0 - v.y1
    dy_above = v.y0 - b.y1
    if dy_below >= 0:
        dy = dy_below
    else:
        dy = abs(dy_above) * 1.6

    xo = _rect_x_overlap(v, b)
    xo_pen = (1.0 - xo) * 140.0
    return float(dy + xo_pen)


def _pick_related_text_blocks(
    text_doc: Optional[dict],
    page_index: int,
    visual_bbox_img: List[float],
    kind: str,
    max_related: int = 3,
    max_candidates: int = 3,
) -> Tuple[List[dict], List[dict], Optional[str], List[str]]:
    """
    Uses the text_extraction artifact (text/{org}/{library}/{doc}.json) to:
    - pick related_text_blocks (top N by score)
    - pick caption_candidates (top N that look like Figure/Table captions)
    - pick best caption_text (text of best caption candidate if present)
    - collect nearby mentions ("as shown in Fig...") for Qwen context
    """
    if not text_doc:
        return [], [], None, []

    pages = text_doc.get("pages") or []
    by_page = {int(p.get("page")): p for p in pages if isinstance(p, dict) and "page" in p}
    p = by_page.get(int(page_index)) or {}
    blocks = p.get("blocks") or []

    related: List[Tuple[float, dict]] = []
    caption_like: List[Tuple[float, dict]] = []
    mentions: List[str] = []

    kind_word = "table" if kind == "table" else "fig"
    for b in blocks:
        if not isinstance(b, dict):
            continue
        if (b.get("kind") or "") != "text":
            continue
        bbox = b.get("bbox_img")
        if not bbox:
            continue
        score = _score_block_to_visual_bbox_img(visual_bbox_img, bbox)
        txt = _clean_text(str(b.get("text") or ""))
        if txt:
            low = txt.lower()
            if kind_word in low or "figure" in low or "fig." in low or "table" in low:
                # Keep small list of mention sentences for Qwen context.
                if len(txt) <= 300:
                    mentions.append(txt)
        rec = {"block_id": b.get("block_id"), "score": round(score, 2), "text_snippet": txt[:160]}
        related.append((score, rec))
        if _CAPTION_RE.match(txt):
            caption_like.append((score - 60.0, rec | {"text": txt}))

    related.sort(key=lambda x: x[0])
    caption_like.sort(key=lambda x: x[0])
    related_out = [r for _, r in related[:max_related] if r.get("block_id")]
    caption_out = [c for _, c in caption_like[:max_candidates] if c.get("block_id")]

    best_caption = None
    if caption_out:
        best_caption = caption_out[0].get("text")  # type: ignore

    return related_out, caption_out, best_caption, mentions[:8]


def _ocr_image(crop: Image.Image) -> str | None:
    if os.getenv("CAPTION_ENABLE_OCR", "1") not in {"1", "true", "yes", "on"}:
        return None
    try:
        engine = (os.getenv("VIS_OCR_ENGINE") or os.getenv("CAPTION_OCR_ENGINE") or "tesseract").strip().lower()
        if engine == "surya":
            txt = _ocr_surya(crop)
            if txt:
                return txt
            if os.getenv("VIS_OCR_FALLBACK_TESSERACT", "1") in {"1", "true", "yes", "on"}:
                return _ocr_tesseract_two_pass(crop)
            return None
        if engine in {"glm", "glm_ocr", "glm-ocr"}:
            return _ocr_glm(crop)
        return _ocr_tesseract_two_pass(crop)
    except Exception:
        return None


def _preprocess_for_ocr(img: Image.Image) -> Image.Image:
    try:
        from PIL import ImageOps, ImageEnhance

        out = img.convert("RGB")
        out = ImageOps.autocontrast(out)
        out = ImageEnhance.Sharpness(out).enhance(1.4)
        w, h = out.size
        if min(w, h) < 500:
            out = out.resize((w * 2, h * 2), resample=Image.BICUBIC)
        return out
    except Exception:
        return img


def _ocr_tesseract_two_pass(crop: Image.Image) -> str | None:
    try:
        import pytesseract  # type: ignore
    except Exception:
        return None

    img = _preprocess_for_ocr(crop)
    lang = os.getenv("CAPTION_OCR_LANG", "eng")
    cfg6 = os.getenv("CAPTION_OCR_CONFIG", "--psm 6")
    cfg11 = os.getenv("CAPTION_OCR_CONFIG_SPARSE", "--psm 11")
    t1 = ""
    t2 = ""
    try:
        t1 = _clean_text(pytesseract.image_to_string(img, lang=lang, config=cfg6))
    except Exception:
        pass
    try:
        t2 = _clean_text(pytesseract.image_to_string(img, lang=lang, config=cfg11))
    except Exception:
        pass
    best = t1 if len(t1) >= len(t2) else t2
    return best or None


def _ocr_surya(crop: Image.Image) -> str | None:
    # Best-effort optional backend; if not installed, return None.
    try:
        from surya.ocr import run_ocr  # type: ignore
    except Exception:
        return None
    try:
        img = _preprocess_for_ocr(crop)
        res = run_ocr([img])
        texts: List[str] = []
        if isinstance(res, list):
            for item in res:
                if isinstance(item, dict) and "text" in item:
                    texts.append(str(item.get("text") or ""))
                elif isinstance(item, str):
                    texts.append(item)
        txt = _clean_text("\n".join(texts))
        return txt or None
    except Exception:
        return None


def _ocr_surya_batch(images: List[Image.Image]) -> List[Optional[str]]:
    """
    Batched Surya OCR for speed (GPU-friendly).
    Returns list of texts aligned to input images.
    """
    if not images:
        return []
    try:
        from surya.ocr import run_ocr  # type: ignore
    except Exception:
        return [None for _ in images]

    try:
        imgs = [_preprocess_for_ocr(im) for im in images]
        res = run_ocr(imgs)
        out: List[Optional[str]] = []
        # Best-effort normalization: many surya versions return list[dict] or list[str]
        if isinstance(res, list) and len(res) == len(imgs):
            for item in res:
                if isinstance(item, dict):
                    txt = _clean_text(str(item.get("text") or ""))
                    out.append(txt or None)
                elif isinstance(item, str):
                    txt = _clean_text(item)
                    out.append(txt or None)
                else:
                    out.append(None)
            # If Surya fails on an image, try a cheap CPU fallback so charts/tables don't go empty.
            if os.getenv("VIS_OCR_FALLBACK_TESSERACT", "1") in {"1", "true", "yes", "on"}:
                fixed: List[Optional[str]] = []
                for im, txt in zip(images, out):
                    if txt:
                        fixed.append(txt)
                        continue
                    try:
                        fixed.append(_ocr_tesseract_two_pass(im))
                    except Exception:
                        fixed.append(None)
                return fixed
            return out
        # If shape unknown, fallback to per-image mode (still correct).
        for im in images:
            txt = _ocr_surya(im)
            if not txt and os.getenv("VIS_OCR_FALLBACK_TESSERACT", "1") in {"1", "true", "yes", "on"}:
                try:
                    txt = _ocr_tesseract_two_pass(im)
                except Exception:
                    txt = None
            out.append(txt)
        return out
    except Exception:
        return [None for _ in images]


_glm_ocr_model = None
_glm_ocr_processor = None


def _ocr_glm(crop: Image.Image) -> str | None:
    global _glm_ocr_model, _glm_ocr_processor
    model_id = os.getenv("VIS_GLM_OCR_MODEL", "zai-org/GLM-OCR")
    try:
        import torch
        from transformers import AutoProcessor, AutoModelForVision2Seq  # type: ignore
    except Exception:
        return None
    try:
        if _glm_ocr_model is None or _glm_ocr_processor is None:
            _glm_ocr_processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
            _glm_ocr_model = AutoModelForVision2Seq.from_pretrained(
                model_id,
                device_map="auto",
                torch_dtype=torch.float16 if torch.cuda.is_available() else None,
                trust_remote_code=True,
            )
            _glm_ocr_model.eval()

        img = _preprocess_for_ocr(crop)
        prompt = os.getenv("VIS_GLM_OCR_PROMPT", "Recognize all text.")
        inputs = _glm_ocr_processor(images=img, text=prompt, return_tensors="pt")
        if hasattr(_glm_ocr_model, "device"):
            inputs = {k: v.to(_glm_ocr_model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = _glm_ocr_model.generate(**inputs, max_new_tokens=int(os.getenv("VIS_GLM_OCR_MAX_TOKENS", "512")))
        txt = _glm_ocr_processor.batch_decode(out, skip_special_tokens=True)[0]
        txt = _clean_text(txt)
        return txt or None
    except Exception:
        return None


def _crop_rel(img: Image.Image, x0: float, y0: float, x1: float, y1: float) -> Image.Image:
    w, h = img.size
    a = max(0, min(w, int(x0 * w)))
    b = max(0, min(h, int(y0 * h)))
    c = max(0, min(w, int(x1 * w)))
    d = max(0, min(h, int(y1 * h)))
    if c <= a or d <= b:
        return img
    return img.crop((a, b, c, d))


def _ocr_chart_regions(crop: Image.Image) -> dict:
    """
    Extra OCR passes for charts:
    - legend-ish region (top-right)
    - y-axis labels (left strip)
    - x-axis labels (bottom strip)
    """
    if os.getenv("VIS_CHART_REGION_OCR", "1") not in {"1", "true", "yes", "on"}:
        return {"full": _ocr_image(crop)}

    full = _ocr_image(crop)
    legend = _ocr_image(_crop_rel(crop, 0.55, 0.00, 1.00, 0.35))
    yaxis = _ocr_image(_crop_rel(crop, 0.00, 0.00, 0.28, 1.00))
    xaxis = _ocr_image(_crop_rel(crop, 0.00, 0.72, 1.00, 1.00))
    return {"full": full, "legend": legend, "y_axis": yaxis, "x_axis": xaxis}


def _summarize_visual(kind: str, caption_text: str | None, ocr_text: str | None, table_text: str | None) -> dict:
    """
    No-LLM summarizer: keeps all raw signals and produces a short caption + bullets.
    This is intentionally conservative to avoid hallucinations.
    """
    short = _clean_text(caption_text or "")
    source = "pdf_caption" if short else "none"
    if not short and kind == "table" and table_text:
        short = "Table (text extracted from PDF)"
        source = "pdf_clip_text"
    if not short and ocr_text:
        short = f"{kind.title()} (OCR extracted)"
        source = "ocr"
    if not short:
        short = f"{kind.title()} (no caption found)"

    bullets: list[str] = []
    if caption_text:
        bullets.append(_clean_text(caption_text)[:240])
    if kind == "table" and table_text:
        t = [ln.strip() for ln in (table_text or "").splitlines() if ln.strip()]
        if t:
            bullets.append("Table text snippet: " + _clean_text(" ".join(t[:3]))[:240])
    if ocr_text:
        o = [ln.strip() for ln in (ocr_text or "").splitlines() if ln.strip()]
        if o:
            bullets.append("OCR snippet: " + _clean_text(" ".join(o[:3]))[:240])

    return {
        "short_caption": short,
        "bullets": bullets[:6],
        "confidence": 0.85 if source == "pdf_caption" else (0.65 if source in {"pdf_clip_text", "ocr"} else 0.35),
        "sources_used": [source] if source != "none" else [],
        "summary_source": source,
    }


def _extract_table_pdf_native(page: fitz.Page, bbox_pdf: fitz.Rect) -> Tuple[Optional[str], Optional[dict]]:
    """
    Best-effort table structure reconstruction from selectable PDF text.
    Returns (csv_text, json_payload) or (None, None).
    """
    try:
        d = page.get_text("dict", clip=bbox_pdf)
    except Exception:
        return None, None

    spans = []
    for b in (d.get("blocks") or []):
        for line in (b.get("lines") or []):
            for sp in (line.get("spans") or []):
                txt = _clean_text(str(sp.get("text") or ""))
                if not txt:
                    continue
                bb = sp.get("bbox") or None
                if not bb or len(bb) != 4:
                    continue
                x0, y0, x1, y1 = [float(v) for v in bb]
                spans.append({"text": txt, "x0": x0, "y0": y0, "x1": x1, "y1": y1})

    if not spans:
        return None, None

    # Group spans into rows by y-center.
    spans.sort(key=lambda s: (s["y0"] + s["y1"]) / 2.0)
    rows: List[List[dict]] = []
    y_tol = float(os.getenv("VIS_TABLE_Y_TOL", "3.5"))

    for sp in spans:
        yc = (sp["y0"] + sp["y1"]) / 2.0
        if not rows:
            rows.append([sp | {"yc": yc}])
            continue
        last_row = rows[-1]
        last_yc = statistics.mean([r["yc"] for r in last_row])  # type: ignore
        if abs(yc - last_yc) <= y_tol:
            last_row.append(sp | {"yc": yc})
        else:
            rows.append([sp | {"yc": yc}])

    # Sort each row by x0 and emit cells by simple gaps clustering.
    table_rows: List[List[str]] = []
    x_gap = float(os.getenv("VIS_TABLE_X_GAP", "10"))
    for row in rows:
        row.sort(key=lambda s: s["x0"])
        cells: List[str] = []
        cur = ""
        last_x1 = None
        for sp in row:
            if last_x1 is None:
                cur = sp["text"]
                last_x1 = sp["x1"]
                continue
            gap = float(sp["x0"] - (last_x1 or sp["x0"]))
            if gap >= x_gap:
                cells.append(cur.strip())
                cur = sp["text"]
            else:
                cur = (cur + " " + sp["text"]).strip()
            last_x1 = sp["x1"]
        if cur.strip():
            cells.append(cur.strip())
        if any(cells):
            table_rows.append(cells)

    if not table_rows:
        return None, None

    # Render CSV (ragged rows allowed).
    out = io.StringIO()
    w = csv.writer(out)
    for r in table_rows:
        w.writerow(r)
    csv_text = out.getvalue()

    payload = {"engine": "pdf_native", "rows": table_rows, "row_count": len(table_rows)}
    return csv_text, payload


def _ocr_boxes_tesseract(crop: Image.Image) -> Optional[List[dict]]:
    if os.getenv("VIS_OCR_WITH_BOXES", "0") not in {"1", "true", "yes", "on"}:
        return None
    try:
        import pytesseract  # type: ignore
    except Exception:
        return None
    try:
        img = _preprocess_for_ocr(crop)
        lang = os.getenv("CAPTION_OCR_LANG", "eng")
        cfg = os.getenv("CAPTION_OCR_CONFIG", "--psm 6")
        data = pytesseract.image_to_data(img, lang=lang, config=cfg, output_type=pytesseract.Output.DICT)
        n = len(data.get("text") or [])
        out = []
        for i in range(n):
            txt = _clean_text(str((data.get("text") or [""])[i]))
            if not txt:
                continue
            try:
                conf = float((data.get("conf") or ["-1"])[i])
            except Exception:
                conf = -1.0
            out.append(
                {
                    "text": txt,
                    "conf": conf,
                    "left": int((data.get("left") or [0])[i]),
                    "top": int((data.get("top") or [0])[i]),
                    "width": int((data.get("width") or [0])[i]),
                    "height": int((data.get("height") or [0])[i]),
                }
            )
        return out[: int(os.getenv("VIS_OCR_BOXES_MAX", "400"))]
    except Exception:
        return None


_qwen_model = None
_qwen_processor = None


def _extract_evidence_text(caption_text: Optional[str], ocr_text: Optional[str], table_csv: Optional[str], mentions: List[str]) -> str:
    parts = []
    if caption_text:
        parts.append(str(caption_text))
    if ocr_text:
        parts.append(str(ocr_text))
    if table_csv:
        parts.append(str(table_csv))
    if mentions:
        parts.append("\n".join(mentions[:6]))
    return "\n".join(parts)


def _qwen_postcheck(qwen_obj: dict, evidence_text: str) -> dict:
    """
    Penalize outputs that introduce unsupported hard facts (numbers/units).
    If VIS_QWEN_STRICT=1, drop the Qwen output entirely when violations occur.
    """
    if not isinstance(qwen_obj, dict):
        return qwen_obj
    strict = os.getenv("VIS_QWEN_STRICT", "0") in {"1", "true", "yes", "on"}
    ev = (evidence_text or "").lower()
    out_txt = json.dumps(qwen_obj, ensure_ascii=True).lower()

    ev_nums = set(_NUM_TOKEN_RE.findall(ev))
    out_nums = set(_NUM_TOKEN_RE.findall(out_txt))
    extra_nums = [n for n in out_nums if n not in ev_nums]

    ev_units = set(_UNIT_RE.findall(ev))
    out_units = set(_UNIT_RE.findall(out_txt))
    extra_units = [u for u in out_units if u not in ev_units]

    violations = len(extra_nums) + len(extra_units)
    if violations <= 0:
        return qwen_obj

    # Penalize confidence; add uncertainty note.
    try:
        conf = float(qwen_obj.get("confidence") or 0.5)
    except Exception:
        conf = 0.5
    conf = max(0.1, conf * 0.65)
    qwen_obj["confidence"] = conf
    unc = qwen_obj.get("uncertainties")
    if not isinstance(unc, list):
        unc = []
    unc.append("Some numeric/units claims could not be verified from caption/OCR/table text.")
    qwen_obj["uncertainties"] = unc[:8]

    if strict:
        return {"_discarded": True, "reason": "unsupported_hard_facts"}
    return qwen_obj


def _qwen_generate_batch(tasks: List[dict]) -> List[Optional[dict]]:
    """
    Batch Qwen inference for speed. Each task must include:
      crop, kind, caption_text, ocr_text, table_csv, nearby_mentions
    """
    if not tasks:
        return []
    if os.getenv("VIS_ENABLE_QWEN_FALLBACK", "1") not in {"1", "true", "yes", "on"}:
        return [None for _ in tasks]
    if os.getenv("VIS_QWEN_MODE", "auto").strip().lower() == "off":
        return [None for _ in tasks]

    model_id = os.getenv("VIS_QWEN_MODEL", "Qwen/Qwen2-VL-2B-Instruct")
    try:
        import torch
        from transformers import AutoProcessor  # type: ignore
        try:
            from transformers import Qwen2VLForConditionalGeneration as QwenModel  # type: ignore
        except Exception:
            from transformers import AutoModelForVision2Seq as QwenModel  # type: ignore
    except Exception:
        return [None for _ in tasks]

    global _qwen_model, _qwen_processor
    try:
        if _qwen_model is None or _qwen_processor is None:
            _qwen_processor = AutoProcessor.from_pretrained(model_id)
            _qwen_model = QwenModel.from_pretrained(
                model_id,
                device_map="auto",
                torch_dtype=torch.float16 if torch.cuda.is_available() else None,
            )
            _qwen_model.eval()

        messages = []
        images = []
        evidences = []
        for t in tasks:
            crop = t["crop"]
            kind = t.get("kind") or "figure"
            cap = (t.get("caption_text") or "").strip()
            ocr_text = (t.get("ocr_text") or None)
            table_csv = (t.get("table_csv") or None)
            nearby_mentions = t.get("nearby_mentions") or []

            prompt = {
                "task": "visual_understanding",
                "kind": kind,
                "instructions": (
                    "You are extracting grounded information from a document visual.\n"
                    "Use provided OCR/caption/table text as the source of truth for hard facts.\n"
                    "If something is not readable, say 'not specified'.\n"
                    "Return STRICT JSON with keys: short_caption, key_observations, extracted_entities, uncertainties, confidence.\n"
                    "If kind=='formula', also include key 'latex' with the best-effort LaTeX transcription.\n"
                ),
                "caption_text": cap or None,
                "ocr_text": (ocr_text or None),
                "table_csv": (table_csv or None),
                "nearby_mentions": nearby_mentions[:5],
            }
            messages.append(
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": crop},
                        {"type": "text", "text": json.dumps(prompt, ensure_ascii=True)},
                    ],
                }
            )
            images.append(crop)
            evidences.append(_extract_evidence_text(cap or None, ocr_text, table_csv, nearby_mentions))

        texts = [_qwen_processor.apply_chat_template([m], tokenize=False, add_generation_prompt=True) for m in messages]
        inputs = _qwen_processor(text=texts, images=images, return_tensors="pt", padding=True)
        try:
            device = _qwen_model.device  # type: ignore
            inputs = {k: v.to(device) for k, v in inputs.items()}
        except Exception:
            pass

        max_new = int(os.getenv("VIS_QWEN_MAX_TOKENS", "512"))
        with torch.no_grad():
            out = _qwen_model.generate(**inputs, max_new_tokens=max_new)
        decs = _qwen_processor.batch_decode(out, skip_special_tokens=True)

        outs: List[Optional[dict]] = []
        for dec, ev in zip(decs, evidences):
            s = (dec or "").strip()
            j0 = s.find("{")
            j1 = s.rfind("}")
            if j0 != -1 and j1 != -1 and j1 > j0:
                s = s[j0 : j1 + 1]
            try:
                obj = json.loads(s)
                if not isinstance(obj, dict):
                    outs.append(None)
                    continue
                obj = _qwen_postcheck(obj, ev)
                if obj.get("_discarded"):
                    outs.append(None)
                else:
                    outs.append(obj)
            except Exception:
                outs.append(None)
        return outs
    except Exception:
        return [None for _ in tasks]


def _maybe_qwen_fallback(
    crop: Image.Image,
    kind: str,
    caption_text: Optional[str],
    ocr_text: Optional[str],
    table_csv: Optional[str],
    nearby_mentions: List[str],
) -> Optional[dict]:
    if os.getenv("VIS_ENABLE_QWEN_FALLBACK", "1") not in {"1", "true", "yes", "on"}:
        return None

    # Heuristic triggers
    cap = (caption_text or "").strip()
    if len(cap) >= int(os.getenv("VIS_QWEN_MIN_CAPTION_CHARS", "40")) and kind != "figure":
        return None
    if not cap and not ocr_text and not table_csv and kind != "figure":
        return None

    model_id = os.getenv("VIS_QWEN_MODEL", "Qwen/Qwen2-VL-2B-Instruct")
    try:
        import torch
        from transformers import AutoProcessor  # type: ignore
        # Try both class names depending on transformers version.
        try:
            from transformers import Qwen2VLForConditionalGeneration as QwenModel  # type: ignore
        except Exception:
            from transformers import AutoModelForVision2Seq as QwenModel  # type: ignore
    except Exception:
        return None

    global _qwen_model, _qwen_processor
    try:
        if _qwen_model is None or _qwen_processor is None:
            _qwen_processor = AutoProcessor.from_pretrained(model_id)
            _qwen_model = QwenModel.from_pretrained(
                model_id,
                device_map="auto",
                torch_dtype=torch.float16 if torch.cuda.is_available() else None,
            )
            _qwen_model.eval()

        prompt = {
            "task": "visual_understanding",
            "kind": kind,
            "instructions": (
                "You are extracting grounded information from a document visual.\n"
                "Use provided OCR/caption/table text as the source of truth for hard facts.\n"
                "If something is not readable, say 'not specified'.\n"
                "Return STRICT JSON with keys: short_caption, key_observations, extracted_entities, uncertainties, confidence.\n"
                "If kind=='formula', also include key 'latex' with the best-effort LaTeX transcription.\n"
            ),
            "caption_text": cap or None,
            "ocr_text": (ocr_text or None),
            "table_csv": (table_csv or None),
            "nearby_mentions": nearby_mentions[:5],
        }
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": crop},
                    {"type": "text", "text": json.dumps(prompt, ensure_ascii=True)},
                ],
            }
        ]

        text = _qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = _qwen_processor(text=[text], images=[crop], return_tensors="pt")
        # Move tensors to the model's device if possible.
        try:
            device = _qwen_model.device  # type: ignore
            inputs = {k: v.to(device) for k, v in inputs.items()}
        except Exception:
            pass
        with torch.no_grad():
            out = _qwen_model.generate(**inputs, max_new_tokens=int(os.getenv("VIS_QWEN_MAX_TOKENS", "512")))
        dec = _qwen_processor.batch_decode(out, skip_special_tokens=True)[0]
        dec = dec.strip()

        # Extract JSON substring if model wraps it in text.
        j0 = dec.find("{")
        j1 = dec.rfind("}")
        if j0 != -1 and j1 != -1 and j1 > j0:
            dec = dec[j0 : j1 + 1]
        try:
            obj = json.loads(dec)
        except Exception:
            return None
        return obj if isinstance(obj, dict) else None
    except Exception:
        return None


def claim_caption_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "image_captioning")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(image_captioning.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]
    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(image_captioning.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _stage_order(stage: str) -> int:
    stages = _pipeline_stages()
    try:
        return stages.index(stage)
    except ValueError:
        return 10_000


def _compute_next_stage(library_id: str) -> str:
    remaining = _sb_execute(
        supabase.table("batch_stage_jobs").select("stage, status").eq("library_id", library_id).neq("status", "done"),
        context="batch_stage_jobs.select(remaining)",
    )
    stages = [str(r.get("stage") or "") for r in (remaining.data or []) if isinstance(r, dict)]
    stages = [s for s in stages if s]
    if not stages:
        return _pipeline_stages()[-1]
    stages.sort(key=_stage_order)
    return stages[0]


def _update_library_progress(library_id: str, stage: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)

    # `completed_batches` represents fully-processed batches (i.e. reached the last stage),
    # not "batches completed in the current stage".
    completed_batches = _count_done_stage_jobs(library_id, _pipeline_stages()[-1])

    next_stage = _compute_next_stage(library_id)
    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": next_stage,
                "completed_batches": completed_batches,
                "pipeline_progress_percent": progress,
            }
        ).eq("id", library_id),
        context="libraries.update(caption_progress)",
    )


def _maybe_finalize_pipeline(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.finalize)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    for st in stages:
        if _count_done_stage_jobs(library_id, st) < total_batches:
            return

    finished = now_iso()
    _sb_execute(
        supabase.table("libraries").update(
            {
                "status": "ready",
                "pipeline_status": "completed",
                "pipeline_stage": stages[-1],
                "pipeline_progress_percent": 100,
                "pipeline_error": None,
                "pipeline_finished_at": finished,
                "completed_batches": total_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline.completed)",
    )


def _render_page_image(page: fitz.Page, render_scale: float) -> Image.Image:
    if render_scale <= 0:
        render_scale = 1.0
    pix = page.get_pixmap(matrix=fitz.Matrix(render_scale, render_scale))
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")


def _crop_bbox(img: Image.Image, bbox_img):
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox_img]
    except Exception:
        return None
    w, h = img.size
    x1 = max(0, min(w, int(x1)))
    x2 = max(0, min(w, int(x2)))
    y1 = max(0, min(h, int(y1)))
    y2 = max(0, min(h, int(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img.crop((x1, y1, x2, y2))


def run_caption_stage_job(stage_job):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    # Stop early if canceled
    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    batch = _sb_execute(
        supabase.table("library_batches").select("doc_ids, doc_count").eq("id", batch_id).single(),
        context="library_batches.select(doc_ids)",
    )
    doc_ids = (batch.data or {}).get("doc_ids") or []
    total = int(stage_job.get("progress_total") or (batch.data or {}).get("doc_count") or len(doc_ids) or 0)
    current = int(stage_job.get("progress_current") or 0)

    _sb_execute(
        supabase.table("libraries").update({"pipeline_status": "running", "pipeline_stage": "image_captioning"}).eq(
            "id", library_id
        ),
        context="libraries.update(stage=image_captioning)",
    )

    # fetch docs metadata
    docs_by_id: dict[str, dict] = {}
    fetch_chunk = int(os.getenv("DOC_FETCH_CHUNK", "100"))
    for i in range(0, len(doc_ids), fetch_chunk):
        chunk = doc_ids[i : i + fetch_chunk]
        resp = _sb_execute(
            supabase.table("documents").select("id, storage_path_raw, mime_type").in_("id", chunk),
            context="documents.select(batch)",
        )
        for d in resp.data or []:
            docs_by_id[d["id"]] = d

    progress_every = int(os.getenv("STAGE_PROGRESS_EVERY", "1"))

    try:
        for doc_id in doc_ids:
            d = docs_by_id.get(doc_id) or {}
            pdf_key = d.get("storage_path_raw")
            mime = (d.get("mime_type") or "").lower()
            if not pdf_key or ("pdf" not in mime):
                current += 1
                if current % progress_every == 0:
                    _sb_execute(
                        supabase.table("batch_stage_jobs").update(
                            {"progress_current": current, "progress_total": total}
                        ).eq("id", job_id),
                        context="batch_stage_jobs.update(progress)",
                    )
                continue

            layout_key = f"layout/{org_id}/{library_id}/{doc_id}.json"
            layout = fetch_r2_json(layout_key) or {}
            render_scale = float(layout.get("render_scale") or os.getenv("LAYOUT_RENDER_SCALE", "1.5"))
            pages = layout.get("layout") or []

            # Load text extraction artifact (for linking visuals <-> nearby text blocks).
            text_key = f"text/{org_id}/{library_id}/{doc_id}.json"
            text_doc = fetch_r2_json(text_key) or None

            # If no visual blocks, still write an empty manifest so downstream can be deterministic.
            out_blocks: List[dict] = []

            pdf_bytes = fetch_r2_bytes(pdf_key)
            pdf_doc = fitz.open(stream=pdf_bytes, filetype="pdf")

            # render pages on demand and reuse per page
            page_img_cache: dict[int, Image.Image] = {}

            doc_profile = _infer_doc_profile(text_doc=text_doc, pages_layout=pages)

            min_area_px = int(os.getenv("VIS_MIN_BBOX_AREA_PX", "2500"))
            min_block_score = float(os.getenv("VIS_MIN_BLOCK_SCORE", "0.25"))
            max_financial_per_page = int(os.getenv("VIS_MAX_VISUALS_PER_PAGE_FINANCIAL", "5"))

            ocr_engine = (os.getenv("VIS_OCR_ENGINE") or "tesseract").strip().lower()
            ocr_mode = (os.getenv("VIS_OCR_MODE") or "auto").strip().lower()
            fig_ocr_mode = (os.getenv("VIS_FIGURE_OCR_MODE") or "auto").strip().lower()
            enable_qwen = os.getenv("VIS_ENABLE_QWEN_FALLBACK", "1") in {"1", "true", "yes", "on"}
            qwen_mode = (os.getenv("VIS_QWEN_MODE") or "auto").strip().lower()

            chart_region_ocr = os.getenv("VIS_CHART_REGION_OCR", "1") in {"1", "true", "yes", "on"}
            chart_region_min_len = int(os.getenv("VIS_CHART_REGION_OCR_MIN_LEN", "60"))

            ocr_batch = int(os.getenv("VIS_OCR_BATCH", "8"))
            if ocr_batch <= 0:
                ocr_batch = 1
            qwen_batch = int(os.getenv("VIS_QWEN_BATCH", "2"))
            if qwen_batch <= 0:
                qwen_batch = 1

            # Keep only the crops we still need for OCR/Qwen in memory. All crops are stored in R2 regardless.
            crops_by_block_id: dict[str, Image.Image] = {}
            blocks_by_id: dict[str, dict] = {}
            ocr_full_tasks: List[dict] = []

            def _bbox_area_px(bb: list[float]) -> float:
                try:
                    x1, y1, x2, y2 = [float(v) for v in bb]
                    return max(0.0, (x2 - x1)) * max(0.0, (y2 - y1))
                except Exception:
                    return 0.0

            def _visual_importance_score(bb: list[float], page_w: int, page_h: int, kind: str) -> float:
                # Score in [0, 1.5]-ish; later thresholding happens via envs.
                try:
                    x1, y1, x2, y2 = [float(v) for v in bb]
                except Exception:
                    return 0.0
                bw = max(1.0, x2 - x1)
                bh = max(1.0, y2 - y1)
                area_frac = (bw * bh) / max(1.0, float(page_w * page_h))
                y_mid = ((y1 + y2) / 2.0) / max(1.0, float(page_h))

                score = min(1.0, area_frac * 4.0)
                if kind == "table":
                    score += 0.20
                elif kind == "figure":
                    score += 0.10
                elif kind == "formula":
                    score += 0.05

                # downweight header/footer-ish regions
                if y_mid < 0.10 or y_mid > 0.90:
                    score -= 0.15
                else:
                    score += 0.05
                return max(0.0, score)

            def _should_full_ocr(kind: str, caption_text: Optional[str], table_text: Optional[str], table_csv: Optional[str]) -> bool:
                if ocr_mode == "off":
                    return False
                cap = (caption_text or "").strip()
                ttxt = (table_text or "").strip()
                tcsv = (table_csv or "").strip()

                if doc_profile == "scanned":
                    # Scanned docs need OCR almost always.
                    return True

                if kind == "table":
                    # If we already extracted a decent CSV from PDF-native text, OCR is usually redundant.
                    if len(tcsv) >= int(os.getenv("VIS_TABLE_CSV_STRONG_LEN", "120")):
                        return False
                    # If selectable text exists inside bbox, that might already be enough.
                    if len(ttxt) >= int(os.getenv("VIS_TABLE_TEXT_STRONG_LEN", "120")):
                        return False
                    return True

                if kind == "formula":
                    # Prefer OCR if formula text wasn't selectable.
                    return len(ttxt) < int(os.getenv("VIS_FORMULA_TEXT_STRONG_LEN", "25"))

                if kind == "figure":
                    if fig_ocr_mode == "off":
                        return False
                    # If caption is good and doc is digital/text-rich, OCR often adds little.
                    if doc_profile == "digital_text_rich" and len(cap) >= int(os.getenv("VIS_FIGURE_CAPTION_STRONG_LEN", "80")):
                        return False
                    # Financial reports often have key info embedded in the figure/table itself.
                    if doc_profile == "financial_report":
                        return True
                    # Default: OCR only when caption is weak.
                    return len(cap) < int(os.getenv("VIS_QWEN_MIN_CAPTION_CHARS", "40"))

                return False

            def _needs_qwen(kind: str, caption_text: Optional[str], ocr_text: Optional[str], table_csv: Optional[str], table_text: Optional[str]) -> bool:
                if not enable_qwen or qwen_mode == "off":
                    return False
                cap = (caption_text or "").strip()
                ocr = (ocr_text or "").strip()
                tcsv = (table_csv or "").strip()
                ttxt = (table_text or "").strip()

                if kind != "figure" and len(cap) >= int(os.getenv("VIS_QWEN_MIN_CAPTION_CHARS", "40")):
                    return False

                # Qwen is a fallback only: use it when we have insufficient evidence.
                if kind == "table":
                    return (len(tcsv) < 80) and (len(ttxt) < 120) and (len(ocr) < 80)
                if kind == "formula":
                    return (len(ttxt) < 25) and (len(ocr) < 25)
                # Figure: need semantic description when caption+OCR are weak.
                return (len(cap) < 40) and (len(ocr) < 80)

            for page_info in pages:
                page_index = int(page_info.get("page") or 0)
                blocks = page_info.get("blocks") or []
                if page_index not in page_img_cache:
                    page = pdf_doc.load_page(page_index)
                    page_img_cache[page_index] = _render_page_image(page, render_scale=render_scale)

                page_img = page_img_cache[page_index]
                page_w, page_h = page_img.size

                # Pre-score blocks so financial reports can process only top-K per page (Tier 1/2),
                # while still storing all crops losslessly (Tier 0).
                candidates: List[dict] = []
                for bi, b in enumerate(blocks):
                    if not isinstance(b, dict):
                        continue
                    t = b.get("type") or ""
                    kind = _normalize_type(t)
                    if kind not in {"figure", "table", "formula"}:
                        continue
                    bbox_img = b.get("bbox") or None
                    if not bbox_img:
                        continue
                    if _bbox_area_px(bbox_img) < float(min_area_px):
                        continue
                    score = _visual_importance_score(bbox_img, page_w, page_h, kind)
                    candidates.append({"bi": bi, "type": t, "kind": kind, "bbox_img": bbox_img, "score": score})

                full_process_ids: set[str] = set()
                if doc_profile == "financial_report":
                    candidates.sort(key=lambda c: float(c.get("score") or 0.0), reverse=True)
                    for c in candidates[: max(0, max_financial_per_page)]:
                        if float(c.get("score") or 0.0) >= float(min_block_score):
                            full_process_ids.add(f"p{page_index}_b{int(c['bi'])}")
                else:
                    for c in candidates:
                        full_process_ids.add(f"p{page_index}_b{int(c['bi'])}")

                page = pdf_doc.load_page(page_index)

                for c in candidates:
                    bi = int(c["bi"])
                    t = str(c.get("type") or "")
                    kind = str(c.get("kind") or "")
                    bbox_img = c.get("bbox_img")
                    if not bbox_img:
                        continue

                    crop = _crop_bbox(page_img, bbox_img)
                    if crop is None:
                        continue

                    block_id = f"p{page_index}_b{bi}"
                    visual_key = f"visuals/{org_id}/{library_id}/{doc_id}/{block_id}.png"
                    put_r2_png(visual_key, crop)

                    # Convert bbox from rendered-image pixels back to PDF points for text clipping.
                    try:
                        x1, y1, x2, y2 = [float(v) for v in bbox_img]
                        bbox_pdf = fitz.Rect(x1 / render_scale, y1 / render_scale, x2 / render_scale, y2 / render_scale)
                    except Exception:
                        bbox_pdf = None

                    # Prefer caption candidates from our text_extraction artifact (stable block_ids).
                    related_text_blocks, caption_candidates, caption_text_from_textdoc, nearby_mentions = _pick_related_text_blocks(
                        text_doc=text_doc,
                        page_index=page_index,
                        visual_bbox_img=bbox_img,
                        kind=kind,
                    )

                    caption_text = caption_text_from_textdoc
                    caption_debug = caption_candidates

                    # Caption grounding improvements: number linking + multi-block merge.
                    prefer_nums = _collect_ref_numbers(kind, nearby_mentions)
                    chosen_block_id = None
                    if caption_candidates:
                        # Prefer a candidate with matching Fig/Table number if present.
                        chosen_txt = None
                        for num in prefer_nums:
                            for cc in caption_candidates:
                                txt = _clean_text(str(cc.get("text") or cc.get("text_snippet") or ""))
                                if not txt:
                                    continue
                                if _extract_ref_number(kind, txt) == num:
                                    chosen_txt = txt
                                    chosen_block_id = cc.get("block_id")
                                    break
                            if chosen_txt:
                                break
                        if not chosen_txt:
                            cc0 = caption_candidates[0]
                            chosen_txt = _clean_text(str(cc0.get("text") or cc0.get("text_snippet") or ""))
                            chosen_block_id = cc0.get("block_id")
                        caption_text = chosen_txt or caption_text

                        # Multi-block caption merge (best effort).
                        try:
                            if text_doc and chosen_block_id:
                                page_blocks = []
                                for p in (text_doc.get("pages") or []):
                                    if int(p.get("page") or -1) == int(page_index):
                                        page_blocks = p.get("blocks") or []
                                        break
                                merged = _merge_caption_blocks(page_blocks, str(chosen_block_id))
                                if merged:
                                    caption_text = merged
                        except Exception:
                            pass

                    # Cross-page caption fallback (next page top).
                    if not caption_text and text_doc:
                        cap2, dbg2 = _cross_page_caption_fallback(text_doc=text_doc, page_index=page_index, kind=kind)
                        if cap2:
                            caption_text = cap2
                            if dbg2:
                                caption_debug = (caption_debug or []) + dbg2

                    # Fallback to raw PDF text blocks if text_extraction artifact is missing.
                    if not caption_text and bbox_pdf is not None:
                        caption_text, raw_dbg = _best_caption_near_bbox(page, bbox_pdf, kind=kind)
                        if raw_dbg and not caption_debug:
                            caption_debug = raw_dbg

                    # Extract selectable text inside the visual bbox (useful for tables/formulas, sometimes for plots).
                    table_text = None
                    if bbox_pdf is not None and kind in {"table", "formula"}:
                        try:
                            table_text = page.get_text("text", clip=bbox_pdf) or None
                        except Exception:
                            table_text = None
                        if table_text:
                            table_text = table_text.strip() or None

                    # Structured table extraction (pdf-native first).
                    table_csv_key = None
                    table_json_key = None
                    table_csv = None
                    table_struct = None
                    if bbox_pdf is not None and kind == "table":
                        eng = (os.getenv("VIS_TABLE_ENGINE") or "pdf_native").strip().lower()
                        if eng == "pdf_native":
                            table_csv, table_struct = _extract_table_pdf_native(page, bbox_pdf)
                            if table_csv:
                                table_csv_key = f"tables/{org_id}/{library_id}/{doc_id}/{block_id}.csv"
                                put_r2_text(table_csv_key, table_csv, content_type="text/csv; charset=utf-8")
                            if table_struct:
                                table_json_key = f"tables/{org_id}/{library_id}/{doc_id}/{block_id}.json"
                                put_r2_json(table_json_key, table_struct)

                    # Tiered OCR policy: decide later in a batched OCR pass (Surya GPU), not per-block.
                    full_process = block_id in full_process_ids
                    want_ocr = full_process and _should_full_ocr(kind, caption_text, table_text, table_csv)

                    # Keep crop in memory only when needed for OCR/Qwen or later processing.
                    if full_process:
                        crops_by_block_id[block_id] = crop
                    ocr_text = None
                    ocr_regions = None
                    ocr_boxes = None

                    if want_ocr:
                        ocr_full_tasks.append(
                            {
                                "block_id": block_id,
                                "kind": kind,
                                "crop": crop,
                                "doc_profile": doc_profile,
                            }
                        )

                    # Conservative baseline summary (before OCR/Qwen).
                    summary = _summarize_visual(kind=kind, caption_text=caption_text, ocr_text=None, table_text=table_text)

                    rec = {
                        "block_id": block_id,
                        "page": page_index,
                        "type": t,
                        "kind": kind,
                        "bbox_img": bbox_img,
                        "bbox_pdf": (list(bbox_pdf) if bbox_pdf is not None else None),
                        "visual_key": visual_key,
                        "caption_text": caption_text,
                        "caption_candidates": caption_debug,
                        "ocr_text": ocr_text,
                        "ocr_regions": ocr_regions,
                        "ocr_boxes": ocr_boxes,
                        "table_text": table_text,
                        "table_csv_key": table_csv_key,
                        "table_json_key": table_json_key,
                        "formula_tex_key": None,
                        "chart_json_key": None,
                        "related_text_blocks": related_text_blocks,
                        "nearby_mentions": nearby_mentions,
                        "table_csv_inline": table_csv,
                        "summary": summary,
                        "full_process": full_process,
                    }
                    out_blocks.append(rec)
                    blocks_by_id[block_id] = rec

            # ---- Batched OCR pass (GPU-friendly when using Surya) ----
            if ocr_full_tasks:
                # Chunk tasks to control memory spikes.
                for offset in range(0, len(ocr_full_tasks), ocr_batch):
                    chunk = ocr_full_tasks[offset : offset + ocr_batch]
                    imgs = [t["crop"] for t in chunk]
                    texts: List[Optional[str]] = []
                    if ocr_engine == "surya":
                        texts = _ocr_surya_batch(imgs)
                    else:
                        texts = [_ocr_image(im) for im in imgs]

                    for tsk, txt in zip(chunk, texts):
                        bid = tsk["block_id"]
                        rec = blocks_by_id.get(bid)
                        if not rec:
                            continue
                        rec["ocr_text"] = txt
                        if rec.get("kind") == "figure":
                            rec["ocr_regions"] = {"full": txt}

                        # OCR boxes are expensive; keep them only when useful.
                        want_boxes = (os.getenv("VIS_OCR_WITH_BOXES") or os.getenv("VIS_OCR_BOXES") or "0") in {
                            "1",
                            "true",
                            "yes",
                            "on",
                        }
                        if not want_boxes and (doc_profile == "scanned" or rec.get("kind") == "table"):
                            want_boxes = True
                        if want_boxes and ocr_engine in {"tesseract", ""}:
                            try:
                                rec["ocr_boxes"] = _ocr_boxes_tesseract(tsk["crop"])
                            except Exception:
                                rec["ocr_boxes"] = None

            # ---- Chart-region OCR pass (legend/axes) ----
            if chart_region_ocr:
                region_tasks: List[dict] = []
                policy = (os.getenv("VIS_CHART_REGION_OCR_POLICY") or "auto").strip().lower()
                for rec in out_blocks:
                    if not rec.get("full_process"):
                        continue
                    if rec.get("kind") != "figure":
                        continue
                    full = _clean_text(str(rec.get("ocr_text") or ""))
                    if policy == "auto" and len(full) >= chart_region_min_len:
                        continue
                    bid = rec.get("block_id")
                    crop = crops_by_block_id.get(str(bid)) if bid else None
                    if crop is None:
                        continue
                    # Sub-crops: legend/top-right, y-axis/left strip, x-axis/bottom strip
                    region_tasks.append({"block_id": bid, "region": "legend", "img": _crop_rel(crop, 0.55, 0.00, 1.00, 0.35)})
                    region_tasks.append({"block_id": bid, "region": "y_axis", "img": _crop_rel(crop, 0.00, 0.00, 0.28, 1.00)})
                    region_tasks.append({"block_id": bid, "region": "x_axis", "img": _crop_rel(crop, 0.00, 0.72, 1.00, 1.00)})

                for offset in range(0, len(region_tasks), ocr_batch):
                    chunk = region_tasks[offset : offset + ocr_batch]
                    imgs = [t["img"] for t in chunk]
                    if ocr_engine == "surya":
                        texts = _ocr_surya_batch(imgs)
                    else:
                        texts = [_ocr_image(im) for im in imgs]
                    for tsk, txt in zip(chunk, texts):
                        bid = str(tsk["block_id"])
                        rec = blocks_by_id.get(bid)
                        if not rec:
                            continue
                        regs = rec.get("ocr_regions") or {}
                        if not isinstance(regs, dict):
                            regs = {}
                        regs[str(tsk["region"])] = txt
                        rec["ocr_regions"] = regs

            # ---- Batched Qwen fallback (hard cases only) ----
            qwen_tasks: List[dict] = []
            qwen_task_ids: List[str] = []
            for rec in out_blocks:
                if not rec.get("full_process"):
                    continue
                kind = str(rec.get("kind") or "")
                cap = rec.get("caption_text")
                ocr_txt = rec.get("ocr_text")
                tcsv = rec.get("table_csv_inline")
                ttxt = rec.get("table_text")
                if not _needs_qwen(kind, cap, ocr_txt, tcsv, ttxt):
                    continue
                bid = str(rec.get("block_id"))
                crop = crops_by_block_id.get(bid)
                if not crop:
                    continue
                qwen_tasks.append(
                    {
                        "crop": crop,
                        "kind": kind,
                        "caption_text": cap,
                        "ocr_text": _clean_text(str(ocr_txt or "")) + ("\n" + _clean_text(str((rec.get("ocr_regions") or {}).get("legend") or "")) if kind == "figure" else ""),
                        "table_csv": tcsv,
                        "nearby_mentions": rec.get("nearby_mentions") or [],
                    }
                )
                qwen_task_ids.append(bid)

            if qwen_tasks:
                outs: List[Optional[dict]] = []
                for offset in range(0, len(qwen_tasks), qwen_batch):
                    outs.extend(_qwen_generate_batch(qwen_tasks[offset : offset + qwen_batch]))
                for bid, qobj in zip(qwen_task_ids, outs):
                    rec = blocks_by_id.get(bid)
                    if not rec:
                        continue
                    kind = str(rec.get("kind") or "")
                    caption_text = rec.get("caption_text")
                    ocr_text = rec.get("ocr_text")
                    table_text = rec.get("table_text")
                    base = _summarize_visual(kind=kind, caption_text=caption_text, ocr_text=ocr_text, table_text=table_text)
                    rec["summary"] = base

                    qwen_obj = qobj if isinstance(qobj, dict) else None
                    if qwen_obj:
                        rec["summary"] = {
                            "short_caption": _clean_text(str(qwen_obj.get("short_caption") or base.get("short_caption") or "")),
                            "bullets": qwen_obj.get("key_observations") or base.get("bullets") or [],
                            "confidence": float(qwen_obj.get("confidence") or base.get("confidence") or 0.5),
                            "sources_used": ["qwen"] + list(base.get("sources_used") or []),
                            "extracted_entities": qwen_obj.get("extracted_entities") or {},
                            "uncertainties": qwen_obj.get("uncertainties") or [],
                            "summary_source": "qwen",
                        }
                        if kind == "formula":
                            latex = qwen_obj.get("latex")
                            if latex:
                                rec["qwen_latex"] = latex

            # ---- Final per-block artifacts (formulas/charts) + final summary refresh ----
            for rec in out_blocks:
                kind = str(rec.get("kind") or "")
                block_id = str(rec.get("block_id") or "")
                page_index = int(rec.get("page") or 0)
                caption_text = rec.get("caption_text")
                ocr_text = rec.get("ocr_text")
                table_text = rec.get("table_text")
                table_csv = rec.get("table_csv_inline")

                # Refresh baseline summary if OCR arrived but Qwen didn't run.
                s0 = rec.get("summary") if isinstance(rec.get("summary"), dict) else {}
                if str(s0.get("summary_source") or "") != "qwen":
                    rec["summary"] = _summarize_visual(
                        kind=kind,
                        caption_text=caption_text,
                        ocr_text=ocr_text,
                        table_text=table_text,
                    )

                # Formula artifact (best-effort LaTeX/plaintext).
                if kind == "formula":
                    formula_tex_key = None
                    latex = rec.get("qwen_latex")
                    formula_text = _clean_text(str(latex or table_text or ocr_text or ""))
                    if formula_text:
                        formula_tex_key = f"formulas/{org_id}/{library_id}/{doc_id}/{block_id}.tex"
                        put_r2_text(formula_tex_key, formula_text, content_type="text/plain; charset=utf-8")
                    rec["formula_tex_key"] = formula_tex_key
                    rec.pop("qwen_latex", None)

                # Chart baseline artifact (labels + caption + entities). Numeric series extraction remains optional.
                if kind == "figure" and (os.getenv("VIS_CHART_ENGINE", "baseline").strip().lower() != "off"):
                    chart_json_key = f"charts/{org_id}/{library_id}/{doc_id}/{block_id}.json"
                    put_r2_json(
                        chart_json_key,
                        {
                            "engine": os.getenv("VIS_CHART_ENGINE", "baseline"),
                            "block_id": block_id,
                            "page": page_index,
                            "caption_text": caption_text,
                            "ocr_text": ocr_text,
                            "ocr_regions": rec.get("ocr_regions"),
                            "extracted_entities": (rec.get("summary") or {}).get("extracted_entities") if isinstance(rec.get("summary"), dict) else {},
                            "series": None,
                        },
                    )
                    rec["chart_json_key"] = chart_json_key

                # Cleanup internal-only fields (don’t store huge or redundant data in manifest).
                rec.pop("nearby_mentions", None)
                rec.pop("table_csv_inline", None)
                rec.pop("full_process", None)

            manifest_key = f"visuals_manifest/{org_id}/{library_id}/{doc_id}.json"
            payload = {
                "doc_id": doc_id,
                "library_id": library_id,
                "organization_id": org_id,
                "created_at": now_iso(),
                "layout_key": layout_key,
                "text_key": text_key,
                "source_pdf_key": pdf_key,
                "render_scale": render_scale,
                "stage": "image_captioning",
                "doc_profile": doc_profile,
                "blocks": out_blocks,
            }
            put_r2_json(manifest_key, payload)

            # Backwards compatible captions artifact (optional).
            if os.getenv("VIS_WRITE_LEGACY_CAPTIONS", "1") in {"1", "true", "yes", "on"}:
                out_key = f"captions/{org_id}/{library_id}/{doc_id}.json"
                put_r2_json(out_key, payload)

            current += 1
            if current % progress_every == 0:
                _sb_execute(
                    supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                        "id", job_id
                    ),
                    context="batch_stage_jobs.update(progress)",
                )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "done", "finished_at": now_iso(), "progress_current": total, "progress_total": total}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(done)",
        )

        # Fan-in: only enqueue the next stage after both parallel extraction stages are done.
        _maybe_enqueue_next_after_parallel(org_id=org_id, library_id=library_id, batch_id=batch_id, progress_total=total)

        _update_library_progress(library_id, stage="image_captioning")
        _maybe_finalize_pipeline(library_id)

    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(failed)",
        )
        _sb_execute(
            supabase.table("libraries").update(
                {
                    "pipeline_status": "failed",
                    "pipeline_stage": "image_captioning",
                    "pipeline_error": str(exc),
                    "status": "error",
                }
            ).eq("id", library_id),
            context="libraries.update(image_captioning.failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("CAPTION_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_caption_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No caption jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_caption_stage_job(job)


if __name__ == "__main__":
    worker_loop()


Overwriting caption_worker.py


# chunk worker

In [ ]:
%%writefile chunk_worker.py
import json
import os
import random
import time
from datetime import datetime, timezone

from dotenv import load_dotenv
from supabase import create_client
import boto3


load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "chunk-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["chunking"]


def _stage_order(stage: str) -> int:
    stages = _pipeline_stages()
    try:
        return stages.index(stage)
    except ValueError:
        return 10_000


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def fetch_r2_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
    return obj["Body"].read()


def fetch_r2_json(key: str) -> dict | None:
    try:
        raw = fetch_r2_bytes(key)
    except Exception:
        return None
    try:
        return json.loads(raw.decode("utf-8"))
    except Exception:
        return None


def put_r2_json(key: str, payload: dict):
    s3.put_object(
        Bucket=R2_BUCKET,
        Key=key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )


def _ensure_stage_job_exists(
    org_id: str,
    library_id: str,
    batch_id: str,
    stage: str,
    progress_total: int,
):
    existing = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).limit(1),
        context=f"batch_stage_jobs.select(exists:{stage})",
    )
    if existing.data:
        return
    _sb_execute(
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": int(progress_total or 0),
            }
        ),
        context=f"batch_stage_jobs.insert({stage})",
    )


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _compute_next_stage(library_id: str) -> str:
    # Earliest stage with any remaining work.
    remaining = _sb_execute(
        supabase.table("batch_stage_jobs").select("stage, status").eq("library_id", library_id).neq("status", "done"),
        context="batch_stage_jobs.select(remaining)",
    )
    stages = [str(r.get("stage") or "") for r in (remaining.data or []) if isinstance(r, dict)]
    stages = [s for s in stages if s]
    if not stages:
        return _pipeline_stages()[-1]
    stages.sort(key=_stage_order)
    return stages[0]


def _update_library_progress(library_id: str, stage: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)
    # `completed_batches` represents fully-processed batches (i.e. reached the last stage),
    # not "batches completed in the current stage".
    completed_batches = _count_done_stage_jobs(library_id, _pipeline_stages()[-1])
    next_stage = _compute_next_stage(library_id)

    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": next_stage,
                "completed_batches": completed_batches,
                "pipeline_progress_percent": progress,
            }
        ).eq("id", library_id),
        context="libraries.update(progress)",
    )


def _maybe_finalize_pipeline(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.finalize)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    for st in stages:
        if _count_done_stage_jobs(library_id, st) < total_batches:
            return

    finished = now_iso()
    _sb_execute(
        supabase.table("libraries").update(
            {
                "status": "ready",
                "pipeline_status": "completed",
                "pipeline_stage": stages[-1],
                "pipeline_progress_percent": 100,
                "pipeline_error": None,
                "pipeline_finished_at": finished,
                "completed_batches": total_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline.completed)",
    )


def claim_chunk_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "chunking")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(chunking.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]
    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(chunking.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def _visual_snippet(block: dict) -> str:
    s = block.get("summary") or {}
    short = (s.get("short_caption") or "").strip()
    bullets = s.get("bullets") or []
    ocr = (block.get("ocr_text") or "").strip()
    table_csv_key = block.get("table_csv_key")
    parts = []
    if short:
        parts.append(f"[VISUAL] {short}")
    for b in bullets[:4]:
        b = str(b).strip()
        if b:
            parts.append(f"- {b}")
    if ocr:
        parts.append("OCR: " + ocr[:400])
    if table_csv_key:
        parts.append(f"(table_csv_key: {table_csv_key})")
    return "\n".join(parts).strip()


def _clean_text(s: str) -> str:
    s = (s or "").replace("\x00", " ").strip()
    return " ".join(s.split())


def _approx_tokens(text: str) -> int:
    # Cheap heuristic to keep chunk sizes bounded without a tokenizer.
    return max(1, int(len(text) / 4))


def _is_heading_like(text: str) -> bool:
    t = _clean_text(text)
    if not t:
        return False
    if len(t) > int(os.getenv("CHUNK_HEADING_MAX_CHARS", "90")):
        return False
    # Section-style headings are often short and do not end with a period.
    if t.endswith(".") and len(t) > 25:
        return False
    # All-caps or Title: patterns.
    if t.upper() == t and any(ch.isalpha() for ch in t) and len(t) <= 60:
        return True
    if t.endswith(":") and len(t) <= 70:
        return True
    # Title-ish: few words, mostly alphabetic.
    words = t.split()
    if 1 <= len(words) <= 8 and sum(w[:1].isalpha() for w in words) >= max(1, len(words) - 1):
        # Avoid false positives like "Table 3" / "Figure 2" (handled as visuals elsewhere).
        low = t.lower()
        if low.startswith("table ") or low.startswith("figure ") or low.startswith("fig "):
            return False
        return True
    return False


def _bbox_sort_key(block: dict) -> tuple:
    # Prefer bbox_img y/x ordering if present to approximate reading order.
    bb = block.get("bbox_img") or None
    if not bb or not isinstance(bb, (list, tuple)) or len(bb) != 4:
        return (int(block.get("page") or 0), 1_000_000, 1_000_000)
    try:
        x1, y1, _x2, _y2 = [float(v) for v in bb]
    except Exception:
        return (int(block.get("page") or 0), 1_000_000, 1_000_000)
    return (int(block.get("page") or 0), y1, x1)


def _make_context_prefix(
    doc_title: str | None,
    section_heading: str | None,
    page_start: int | None,
    page_end: int | None,
) -> str:
    parts = []
    if doc_title:
        parts.append(f"Document: {doc_title}")
    if section_heading:
        parts.append(f"Section: {section_heading}")
    if page_start is not None and page_end is not None:
        if page_start == page_end:
            parts.append(f"Pages: {page_start + 1}")
        else:
            parts.append(f"Pages: {page_start + 1}-{page_end + 1}")
    return " | ".join(parts).strip()


def _semantic_boundaries(block_texts: list[str]) -> set[int]:
    """
    Optional semantic boundary detection using a small embedding model.
    Returns a set of indices i where a new chunk should start (before i).
    Uses GPU if available.
    """
    if os.getenv("CHUNK_SEMANTIC", "0") not in {"1", "true", "yes", "on"}:
        return set()
    try:
        import torch
        from transformers import AutoTokenizer, AutoModel  # type: ignore
    except Exception:
        return set()

    model_id = os.getenv("CHUNK_SEMANTIC_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
    batch = int(os.getenv("CHUNK_SEMANTIC_BATCH", "32"))
    thr = float(os.getenv("CHUNK_SEMANTIC_THRESHOLD", "0.25"))  # lower sim -> boundary
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
        mdl = AutoModel.from_pretrained(model_id)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        mdl.to(device)
        mdl.eval()

        def embed(texts: list[str]):
            outs = []
            for i in range(0, len(texts), batch):
                chunk = texts[i : i + batch]
                inp = tok(chunk, padding=True, truncation=True, max_length=256, return_tensors="pt")
                inp = {k: v.to(device) for k, v in inp.items()}
                with torch.no_grad():
                    o = mdl(**inp).last_hidden_state
                    mask = inp["attention_mask"].unsqueeze(-1)
                    pooled = (o * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                    pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
                    outs.append(pooled.detach().cpu())
            return torch.cat(outs, dim=0)

        vecs = embed([t[:2000] for t in block_texts])
        boundaries: set[int] = set()
        for i in range(1, vecs.shape[0]):
            sim = float((vecs[i - 1] * vecs[i]).sum().item())
            if sim < thr:
                boundaries.add(i)
        return boundaries
    except Exception:
        return set()


def run_chunk_stage_job(stage_job):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    # Stop early if canceled/deleted.
    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    batch = _sb_execute(
        supabase.table("library_batches").select("doc_ids, doc_count").eq("id", batch_id).single(),
        context="library_batches.select(doc_ids)",
    )
    doc_ids = (batch.data or {}).get("doc_ids") or []
    total = int(stage_job.get("progress_total") or (batch.data or {}).get("doc_count") or len(doc_ids) or 0)
    current = int(stage_job.get("progress_current") or 0)
    progress_every = int(os.getenv("STAGE_PROGRESS_EVERY", "3"))

    try:
        # Fetch doc titles once (helps contextual retrieval without extra LLM calls).
        docs_by_id: dict[str, dict] = {}
        fetch_chunk = int(os.getenv("DOC_FETCH_CHUNK", "100"))
        for i in range(0, len(doc_ids), fetch_chunk):
            chunk = doc_ids[i : i + fetch_chunk]
            resp = _sb_execute(
                supabase.table("documents").select("id, title").in_("id", chunk),
                context="documents.select(titles)",
            )
            for d in resp.data or []:
                docs_by_id[str(d.get("id"))] = d

        for doc_id in doc_ids:
            lib_check = _sb_execute(
                supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
                context="libraries.select(pipeline_status.loop)",
            )
            if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
                return

            text_key = f"text/{org_id}/{library_id}/{doc_id}.json"
            vis_key = f"visuals_manifest/{org_id}/{library_id}/{doc_id}.json"
            text_doc = fetch_r2_json(text_key) or {}
            vis_doc = fetch_r2_json(vis_key) or {}

            # Build a mapping from text block_id -> attached visual snippets (multi-attach).
            attach: dict[str, list[dict]] = {}
            for vb in (vis_doc.get("blocks") or []):
                if not isinstance(vb, dict):
                    continue
                snippet = _visual_snippet(vb)
                if not snippet:
                    continue
                vid = str(vb.get("block_id") or "")
                vkey = vb.get("visual_key")
                rel = vb.get("related_text_blocks") or []
                rel_ids: list[str] = []
                if isinstance(rel, list):
                    for item in rel:
                        if isinstance(item, dict) and item.get("block_id"):
                            rel_ids.append(str(item.get("block_id")))
                        elif isinstance(item, str):
                            rel_ids.append(item)
                rel_ids = [r for r in rel_ids if r]
                if not rel_ids:
                    continue
                for tid in rel_ids[: int(os.getenv("CHUNK_MAX_VISUAL_ATTACHMENTS", "3"))]:
                    attach.setdefault(str(tid), []).append(
                        {
                            "visual_block_id": vid,
                            "visual_key": vkey,
                            "snippet": snippet,
                        }
                    )

            # Flatten all text blocks in approximate reading order.
            flat_blocks: list[dict] = []
            for p in (text_doc.get("pages") or []):
                page_idx = int(p.get("page") or 0)
                blocks = [b for b in (p.get("blocks") or []) if isinstance(b, dict) and (b.get("kind") or "") == "text"]
                blocks.sort(key=_bbox_sort_key)
                for b in blocks:
                    txt = (b.get("text") or "").strip()
                    if not txt:
                        continue
                    flat_blocks.append(
                        {
                            "page": page_idx,
                            "block_id": str(b.get("block_id") or ""),
                            "text": txt,
                            "bbox_img": b.get("bbox_img"),
                        }
                    )

            # Optional semantic boundaries (GPU-accelerated if available).
            sem_boundaries = _semantic_boundaries([b["text"] for b in flat_blocks])

            # Chunking params (structure-based, with size budget + optional overlap).
            target_tokens = int(os.getenv("CHUNK_TARGET_TOKENS", "400"))
            max_tokens = int(os.getenv("CHUNK_MAX_TOKENS", "650"))
            min_tokens = int(os.getenv("CHUNK_MIN_TOKENS", "120"))
            overlap_blocks = int(os.getenv("CHUNK_OVERLAP_BLOCKS", "1"))
            if overlap_blocks < 0:
                overlap_blocks = 0

            doc_title = _clean_text(str((docs_by_id.get(str(doc_id)) or {}).get("title") or "")) or None

            chunks: list[dict] = []
            cur_blocks: list[dict] = []
            cur_heading: str | None = None
            cur_section: str | None = None
            cur_visuals: dict[str, dict] = {}  # visual_block_id -> visual rec

            def flush_chunk(force: bool = False):
                nonlocal cur_blocks, cur_heading, cur_visuals
                if not cur_blocks:
                    return
                text = "\n\n".join(_clean_text(b["text"]) for b in cur_blocks).strip()
                if not text:
                    cur_blocks = []
                    cur_visuals = {}
                    return
                tok = _approx_tokens(text)
                if not force and tok < min_tokens:
                    return

                page_start = int(cur_blocks[0]["page"])
                page_end = int(cur_blocks[-1]["page"])
                block_ids = [b["block_id"] for b in cur_blocks if b.get("block_id")]

                # Collect visuals attached to any block in this chunk (deduped).
                vis_snips: list[str] = []
                vis_ids: list[str] = []
                vis_keys: list[str] = []
                for bid in block_ids:
                    for vr in attach.get(bid) or []:
                        vbid = str(vr.get("visual_block_id") or "")
                        if not vbid or vbid in cur_visuals:
                            continue
                        cur_visuals[vbid] = vr
                for vbid, vr in cur_visuals.items():
                    vis_ids.append(vbid)
                    if vr.get("visual_key"):
                        vis_keys.append(str(vr.get("visual_key")))
                    sn = str(vr.get("snippet") or "").strip()
                    if sn:
                        vis_snips.append(sn)

                context_prefix = _make_context_prefix(doc_title, cur_heading or cur_section, page_start, page_end)
                embedding_text = context_prefix
                if embedding_text:
                    embedding_text += "\n\n"
                embedding_text += text
                if vis_snips:
                    embedding_text += "\n\n" + "\n\n".join(vis_snips[: int(os.getenv("CHUNK_MAX_VISUAL_SNIPPETS", "6"))])

                chunk_index = len(chunks)
                chunk_id = f"{doc_id}_c{chunk_index:04d}"
                chunks.append(
                    {
                        "chunk_id": chunk_id,
                        "chunk_index": chunk_index,
                        "doc_id": doc_id,
                        "library_id": library_id,
                        "organization_id": org_id,
                        "page_start": page_start,
                        "page_end": page_end,
                        "section_heading": cur_heading or cur_section,
                        "block_ids": block_ids,
                        "text": text,
                        "context_prefix": context_prefix,
                        "embedding_text": embedding_text,
                        "visual_ids": vis_ids,
                        "visual_keys": vis_keys,
                        "visual_snippets": vis_snips[: int(os.getenv("CHUNK_MAX_VISUAL_SNIPPETS", "6"))],
                    }
                )

                # Prepare overlap for next chunk.
                if overlap_blocks > 0:
                    cur_blocks = cur_blocks[-overlap_blocks:]
                else:
                    cur_blocks = []
                cur_visuals = {}

            for i, b in enumerate(flat_blocks):
                txt = _clean_text(b["text"])
                if not txt:
                    continue

                # Update section heading state when we see a heading-like block.
                if _is_heading_like(txt):
                    cur_section = txt
                    # If current chunk has enough content, flush before starting new section.
                    if cur_blocks and _approx_tokens("\n\n".join(bb["text"] for bb in cur_blocks)) >= min_tokens:
                        flush_chunk(force=True)
                    cur_heading = txt
                    continue

                # Semantic boundary hint: start new chunk before this block.
                if i in sem_boundaries and cur_blocks:
                    flush_chunk(force=True)

                cur_blocks.append(b)
                cur_tok = _approx_tokens("\n\n".join(bb["text"] for bb in cur_blocks))
                if cur_tok >= max_tokens:
                    flush_chunk(force=True)
                elif cur_tok >= target_tokens:
                    # Soft flush: wait for a natural boundary (heading/semantic/page break),
                    # but don't exceed max_tokens.
                    next_txt = ""
                    next_page = None
                    if i + 1 < len(flat_blocks):
                        next_txt = _clean_text(str(flat_blocks[i + 1].get("text") or ""))
                        next_page = int(flat_blocks[i + 1].get("page") or 0)
                    boundary = False
                    if next_txt and _is_heading_like(next_txt):
                        boundary = True
                    if (i + 1) in sem_boundaries:
                        boundary = True
                    if next_page is not None and next_page != int(b.get("page") or 0):
                        boundary = True
                    if boundary:
                        flush_chunk(force=True)

            flush_chunk(force=True)

            # Add neighbor pointers to support query-time expansion.
            for idx, ch in enumerate(chunks):
                ch["prev_chunk_id"] = chunks[idx - 1]["chunk_id"] if idx > 0 else None
                ch["next_chunk_id"] = chunks[idx + 1]["chunk_id"] if idx + 1 < len(chunks) else None

            out_key = f"chunks/{org_id}/{library_id}/{doc_id}.json"
            put_r2_json(
                out_key,
                {
                    "doc_id": doc_id,
                    "library_id": library_id,
                    "organization_id": org_id,
                    "created_at": now_iso(),
                    "text_key": text_key,
                    "visuals_manifest_key": vis_key,
                    "chunking": {
                        "strategy": os.getenv("CHUNK_STRATEGY", "layout_structured"),
                        "semantic_enabled": os.getenv("CHUNK_SEMANTIC", "0") in {"1", "true", "yes", "on"},
                        "target_tokens": target_tokens,
                        "max_tokens": max_tokens,
                        "overlap_blocks": overlap_blocks,
                    },
                    "chunks": chunks,
                },
            )

            current += 1
            if current % progress_every == 0:
                _sb_execute(
                    supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                        "id", job_id
                    ),
                    context="batch_stage_jobs.update(chunking.progress)",
                )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "done", "finished_at": now_iso(), "progress_current": total, "progress_total": total}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(chunking.done)",
        )

        # Enqueue next stage if configured (e.g. embedding) after chunking.
        stages = _pipeline_stages()
        try:
            idx = stages.index("chunking")
        except ValueError:
            idx = -1
        if 0 <= idx < len(stages) - 1:
            next_stage = stages[idx + 1]
            if next_stage:
                _ensure_stage_job_exists(org_id, library_id, batch_id, next_stage, total)

        _update_library_progress(library_id, stage="chunking")
        _maybe_finalize_pipeline(library_id)
    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(chunking.failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("CHUNK_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_chunk_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No chunk jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_chunk_stage_job(job)


if __name__ == "__main__":
    worker_loop()


Overwriting chunk_worker.py


# embed worker

In [ ]:
%%writefile embed_worker.py
import os
import json
import random
import time
from datetime import datetime, timezone

from dotenv import load_dotenv
from supabase import create_client
import boto3


load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "embed-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

R2_ENDPOINT = get_env("R2_ENDPOINT")
R2_BUCKET = get_env("R2_BUCKET")
R2_ACCESS_KEY = get_env("R2_ACCESS_KEY")
R2_SECRET_KEY = get_env("R2_SECRET_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=R2_ACCESS_KEY,
    aws_secret_access_key=R2_SECRET_KEY,
)


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking,embedding,clustering")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["embedding"]


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def fetch_r2_bytes(key: str) -> bytes:
    obj = s3.get_object(Bucket=R2_BUCKET, Key=key)
    return obj["Body"].read()


def fetch_r2_json(key: str) -> dict | None:
    try:
        raw = fetch_r2_bytes(key)
    except Exception:
        return None
    try:
        return json.loads(raw.decode("utf-8"))
    except Exception:
        return None


_embedder = None


def _get_embedder():
    global _embedder
    if _embedder is not None:
        return _embedder
    # SentenceTransformers provides stable pooling + normalization across many models.
    from sentence_transformers import SentenceTransformer  # type: ignore

    model_id = os.getenv("EMBED_MODEL", "BAAI/bge-large-en-v1.5")
    device = os.getenv("EMBED_DEVICE", "").strip()
    if not device:
        try:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            device = "cpu"
    _embedder = SentenceTransformer(model_id, device=device)
    return _embedder


def _embed_texts(texts: list[str]) -> list[list[float]]:
    embedder = _get_embedder()
    batch_size = int(os.getenv("EMBED_BATCH", "32"))
    if batch_size <= 0:
        batch_size = 16
    # Normalize for cosine/dot-product equivalence.
    vecs = embedder.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    try:
        import numpy as np

        if isinstance(vecs, np.ndarray):
            return vecs.astype("float32").tolist()
    except Exception:
        pass
    return [list(map(float, v)) for v in vecs]


def claim_embedding_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "embedding")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(embedding.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]
    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(embedding.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _update_library_progress(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)
    # Batch-completed means reached the last stage.
    completed_batches = _count_done_stage_jobs(library_id, stages[-1])

    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_progress_percent": progress,
                "completed_batches": completed_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(progress)",
    )


def _ensure_stage_job_exists(org_id: str, library_id: str, batch_id: str, stage: str, progress_total: int):
    existing = _sb_execute(
        supabase.table("batch_stage_jobs").select("id").eq("batch_id", batch_id).eq("stage", stage).limit(1),
        context=f"batch_stage_jobs.select(exists:{stage})",
    )
    if existing.data:
        return
    _sb_execute(
        supabase.table("batch_stage_jobs").insert(
            {
                "organization_id": org_id,
                "library_id": library_id,
                "batch_id": batch_id,
                "stage": stage,
                "status": "queued",
                "attempts": 0,
                "payload": {},
                "progress_current": 0,
                "progress_total": int(progress_total or 0),
            }
        ),
        context=f"batch_stage_jobs.insert({stage})",
    )


def _embedding_done_for_library(library_id: str) -> bool:
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.embedding_check)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return False
    return _count_done_stage_jobs(library_id, "embedding") >= total_batches


def run_embedding_stage_job(stage_job: dict):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    batch_id = stage_job["batch_id"]
    job_id = stage_job["id"]

    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    batch = _sb_execute(
        supabase.table("library_batches").select("doc_ids, doc_count").eq("id", batch_id).single(),
        context="library_batches.select(doc_ids)",
    )
    doc_ids = (batch.data or {}).get("doc_ids") or []
    total = int(stage_job.get("progress_total") or (batch.data or {}).get("doc_count") or len(doc_ids) or 0)
    current = int(stage_job.get("progress_current") or 0)
    progress_every = int(os.getenv("STAGE_PROGRESS_EVERY", "2"))

    # Keep UI stable.
    _sb_execute(
        supabase.table("libraries").update({"pipeline_status": "running", "pipeline_stage": "embedding"}).eq(
            "id", library_id
        ),
        context="libraries.update(stage.embedding)",
    )

    table_name = os.getenv("EMBED_TABLE", "chunk_embeddings").strip() or "chunk_embeddings"
    model_id = os.getenv("EMBED_MODEL", "BAAI/bge-large-en-v1.5")

    try:
        for doc_id in doc_ids:
            lib_check = _sb_execute(
                supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
                context="libraries.select(pipeline_status.loop)",
            )
            if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
                return

            chunks_key = f"chunks/{org_id}/{library_id}/{doc_id}.json"
            chunks_doc = fetch_r2_json(chunks_key) or {}
            chunks = chunks_doc.get("chunks") or []
            if not isinstance(chunks, list) or not chunks:
                current += 1
                if current % progress_every == 0:
                    _sb_execute(
                        supabase.table("batch_stage_jobs").update(
                            {"progress_current": current, "progress_total": total}
                        ).eq("id", job_id),
                        context="batch_stage_jobs.update(embedding.progress)",
                    )
                continue

            texts = []
            metas = []
            for ch in chunks:
                if not isinstance(ch, dict):
                    continue
                emb_text = str(ch.get("embedding_text") or ch.get("text") or "").strip()
                if not emb_text:
                    continue
                texts.append(emb_text)
                metas.append(ch)

            vecs = _embed_texts(texts) if texts else []
            rows = []
            for ch, vec in zip(metas, vecs):
                chunk_id = str(ch.get("chunk_id") or "")
                if not chunk_id:
                    continue
                rows.append(
                    {
                        "organization_id": org_id,
                        "library_id": library_id,
                        "doc_id": doc_id,
                        "chunk_id": chunk_id,
                        "chunk_index": int(ch.get("chunk_index") or 0),
                        "page_start": int(ch.get("page_start") or 0),
                        "page_end": int(ch.get("page_end") or 0),
                        "section_heading": ch.get("section_heading"),
                        "text": ch.get("text"),
                        "context_prefix": ch.get("context_prefix"),
                        "embedding_text": ch.get("embedding_text"),
                        "embedding_model": model_id,
                        "embedding_dim": len(vec) if isinstance(vec, list) else None,
                        "embedding": vec,
                        "visual_ids": ch.get("visual_ids") or [],
                        "visual_keys": ch.get("visual_keys") or [],
                        "updated_at": now_iso(),
                    }
                )

            if rows:
                chunk_size = int(os.getenv("EMBED_UPSERT_CHUNK", "200"))
                for i in range(0, len(rows), chunk_size):
                    part = rows[i : i + chunk_size]
                    _sb_execute(
                        supabase.table(table_name).upsert(part, on_conflict="chunk_id"),
                        context=f"{table_name}.upsert",
                    )

            current += 1
            if current % progress_every == 0:
                _sb_execute(
                    supabase.table("batch_stage_jobs").update({"progress_current": current, "progress_total": total}).eq(
                        "id", job_id
                    ),
                    context="batch_stage_jobs.update(embedding.progress)",
                )

        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "done", "finished_at": now_iso(), "progress_current": total, "progress_total": total}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(embedding.done)",
        )

        # Gate clustering: only enqueue clustering jobs after *all* embedding batch-jobs are done.
        if _embedding_done_for_library(library_id):
            if "clustering" in _pipeline_stages():
                batches = _sb_execute(
                    supabase.table("library_batches").select("id, doc_count").eq("library_id", library_id),
                    context="library_batches.select(all)",
                )
                for b in (batches.data or []):
                    bid = str(b.get("id") or "")
                    if not bid:
                        continue
                    _ensure_stage_job_exists(org_id, library_id, bid, "clustering", int(b.get("doc_count") or 0))

        _update_library_progress(library_id)
    except Exception as exc:
        _sb_execute(
            supabase.table("batch_stage_jobs").update(
                {"status": "failed", "last_error": str(exc), "finished_at": now_iso()}
            ).eq("id", job_id),
            context="batch_stage_jobs.update(embedding.failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("EMBED_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_embedding_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No embedding jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_embedding_stage_job(job)


if __name__ == "__main__":
    worker_loop()



Overwriting embed_worker.py


# cluster worker

In [ ]:
%%writefile cluster_worker.py
import os
import json
import random
import time
from datetime import datetime, timezone

from dotenv import load_dotenv
from supabase import create_client


load_dotenv("/workspace/.env")

WORKER_ID = os.getenv("WORKER_ID", "cluster-1")


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def get_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise RuntimeError(f"Missing env var: {name}")
    return v


SUPABASE_URL = get_env("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = get_env("SUPABASE_SERVICE_ROLE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)


def _pipeline_stages():
    raw = os.getenv("PIPELINE_STAGES", "sync,layout_parser,text_extraction,image_captioning,chunking,embedding,clustering")
    stages = [s.strip() for s in raw.split(",") if s.strip()]
    return stages or ["clustering"]


def _stage_order(stage: str) -> int:
    stages = _pipeline_stages()
    try:
        return stages.index(stage)
    except ValueError:
        return 10_000


def _is_retryable_supabase_error(exc: Exception) -> bool:
    msg = str(exc) or ""
    m = msg.lower()
    if "json could not be generated" in m:
        return True
    if "bad gateway" in m or "error code 502" in m or " 502" in m:
        return True
    if "web server is down" in m or "error code 521" in m or " 521" in m:
        return True
    if "timeout" in m or "timed out" in m:
        return True
    if "too many requests" in m or " 429" in m:
        return True
    return False


def _sb_execute(query, context: str = "", max_attempts: int | None = None):
    attempts = int(os.getenv("SUPABASE_MAX_RETRIES", "6")) if max_attempts is None else int(max_attempts)
    base = float(os.getenv("SUPABASE_RETRY_BASE_SECONDS", "0.6"))
    last_exc: Exception | None = None
    for i in range(attempts):
        try:
            return query.execute()
        except Exception as exc:
            last_exc = exc
            if not _is_retryable_supabase_error(exc) or i == attempts - 1:
                raise
            sleep_s = min(20.0, base * (2**i)) * (0.85 + random.random() * 0.3)
            print(f"[supabase-retry] {context or 'query'} attempt={i+1}/{attempts} sleep={sleep_s:.2f}s err={exc}")
            time.sleep(sleep_s)
    if last_exc:
        raise last_exc


def claim_clustering_stage_job(worker_id: str):
    jobs = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("*")
        .eq("stage", "clustering")
        .eq("status", "queued")
        .order("created_at")
        .limit(1),
        context="batch_stage_jobs.select(clustering.queued)",
    )
    if not jobs.data:
        return None
    job = jobs.data[0]
    claimed = _sb_execute(
        supabase.table("batch_stage_jobs")
        .update(
            {
                "status": "running",
                "assigned_worker": worker_id,
                "started_at": now_iso(),
                "attempts": int(job.get("attempts") or 0) + 1,
            }
        )
        .eq("id", job["id"])
        .eq("status", "queued"),
        context="batch_stage_jobs.update(clustering.claim)",
    )
    if not claimed.data:
        return None
    return claimed.data[0]


def _count_done_stage_jobs(library_id: str, stage: str) -> int:
    resp = _sb_execute(
        supabase.table("batch_stage_jobs")
        .select("id", count="exact")
        .eq("library_id", library_id)
        .eq("stage", stage)
        .eq("status", "done"),
        context=f"batch_stage_jobs.count(done:{stage})",
    )
    return int(resp.count or 0)


def _compute_next_stage(library_id: str) -> str:
    remaining = _sb_execute(
        supabase.table("batch_stage_jobs").select("stage, status").eq("library_id", library_id).neq("status", "done"),
        context="batch_stage_jobs.select(remaining)",
    )
    stages = [str(r.get("stage") or "") for r in (remaining.data or []) if isinstance(r, dict)]
    stages = [s for s in stages if s]
    if not stages:
        return _pipeline_stages()[-1]
    stages.sort(key=_stage_order)
    return stages[0]


def _update_library_progress(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    done_total = 0
    for st in stages:
        done_total += _count_done_stage_jobs(library_id, st)

    denom = max(1, total_batches * len(stages))
    progress = round((done_total / denom) * 100, 2)
    completed_batches = _count_done_stage_jobs(library_id, stages[-1])
    next_stage = _compute_next_stage(library_id)

    _sb_execute(
        supabase.table("libraries").update(
            {
                "pipeline_status": "running",
                "pipeline_stage": next_stage,
                "pipeline_progress_percent": progress,
                "completed_batches": completed_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(progress)",
    )


def _maybe_finalize_pipeline(library_id: str):
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.finalize)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return

    stages = _pipeline_stages()
    for st in stages:
        if _count_done_stage_jobs(library_id, st) < total_batches:
            return

    finished = now_iso()
    _sb_execute(
        supabase.table("libraries").update(
            {
                "status": "ready",
                "pipeline_status": "completed",
                "pipeline_stage": stages[-1],
                "pipeline_progress_percent": 100,
                "pipeline_error": None,
                "pipeline_finished_at": finished,
                "completed_batches": total_batches,
            }
        ).eq("id", library_id),
        context="libraries.update(pipeline.completed)",
    )


def _choose_k(num_chunks: int) -> int:
    # sqrt heuristic with caps
    k = int(round((num_chunks ** 0.5)))
    k = max(int(os.getenv("CLUSTER_K_MIN", "8")), k)
    k = min(int(os.getenv("CLUSTER_K_MAX", "50")), k)
    k = min(k, max(1, num_chunks))
    return k


def _fetch_all_embeddings(library_id: str, table_name: str) -> tuple[list[str], list[list[float]]]:
    ids: list[str] = []
    vecs: list[list[float]] = []
    page_size = int(os.getenv("CLUSTER_FETCH_PAGE", "1000"))
    offset = 0
    while True:
        resp = _sb_execute(
            supabase.table(table_name)
            .select("chunk_id, embedding")
            .eq("library_id", library_id)
            .range(offset, offset + page_size - 1),
            context=f"{table_name}.select(embeddings)",
        )
        rows = resp.data or []
        if not rows:
            break
        for r in rows:
            cid = str(r.get("chunk_id") or "")
            emb = r.get("embedding")
            if not cid or not isinstance(emb, list) or not emb:
                continue
            ids.append(cid)
            vecs.append([float(x) for x in emb])
        if len(rows) < page_size:
            break
        offset += page_size
    return ids, vecs


def _run_kmeans(ids: list[str], vecs: list[list[float]]) -> tuple[dict[str, int], list[list[float]]]:
    # Default CPU engine: MiniBatchKMeans.
    import numpy as np
    from sklearn.cluster import MiniBatchKMeans  # type: ignore

    x = np.asarray(vecs, dtype="float32")
    k = int(os.getenv("CLUSTER_K", "0")) or _choose_k(int(x.shape[0]))
    batch = int(os.getenv("CLUSTER_BATCH", "2048"))
    km = MiniBatchKMeans(
        n_clusters=k,
        batch_size=batch,
        n_init="auto",
        random_state=42,
        max_iter=int(os.getenv("CLUSTER_MAX_ITER", "200")),
    )
    labels = km.fit_predict(x)
    centers = km.cluster_centers_.astype("float32").tolist()
    mapping = {cid: int(lbl) for cid, lbl in zip(ids, labels)}
    return mapping, centers


def run_clustering_stage_job(stage_job: dict):
    library_id = stage_job["library_id"]
    org_id = stage_job["organization_id"]
    job_id = stage_job["id"]

    lib_check = _sb_execute(
        supabase.table("libraries").select("id, pipeline_status").eq("id", library_id).limit(1),
        context="libraries.select(pipeline_status)",
    )
    if not lib_check.data or (lib_check.data[0].get("pipeline_status") == "canceled"):
        return

    # Ensure embeddings are complete before clustering.
    lib = _sb_execute(
        supabase.table("libraries").select("total_batches").eq("id", library_id).single(),
        context="libraries.select(total_batches.cluster_gate)",
    )
    total_batches = int((lib.data or {}).get("total_batches") or 0)
    if total_batches <= 0:
        return
    if _count_done_stage_jobs(library_id, "embedding") < total_batches:
        # Re-queue and try later.
        _sb_execute(
            supabase.table("batch_stage_jobs").update({"status": "queued", "assigned_worker": None}).eq("id", job_id),
            context="batch_stage_jobs.update(clustering.requeue.wait_embed)",
        )
        time.sleep(2)
        return

    # Library-level lock via table.
    run_table = os.getenv("CLUSTER_RUN_TABLE", "library_cluster_runs").strip() or "library_cluster_runs"
    emb_table = os.getenv("EMBED_TABLE", "chunk_embeddings").strip() or "chunk_embeddings"
    cluster_table = os.getenv("CLUSTER_TABLE", "library_clusters").strip() or "library_clusters"

    # If already completed, mark this job done.
    existing = _sb_execute(
        supabase.table(run_table).select("status").eq("library_id", library_id).limit(1),
        context=f"{run_table}.select(status)",
    )
    if existing.data and str(existing.data[0].get("status") or "") == "done":
        _sb_execute(
            supabase.table("batch_stage_jobs").update({"status": "done", "finished_at": now_iso()}).eq("id", job_id),
            context="batch_stage_jobs.update(clustering.done.fast)",
        )
        _update_library_progress(library_id)
        _maybe_finalize_pipeline(library_id)
        return

    # Acquire lock (best effort).
    lock_ok = False
    try:
        _sb_execute(
            supabase.table(run_table).insert(
                {
                    "library_id": library_id,
                    "organization_id": org_id,
                    "status": "running",
                    "started_at": now_iso(),
                    "updated_at": now_iso(),
                }
            ),
            context=f"{run_table}.insert(lock)",
        )
        lock_ok = True
    except Exception:
        # Someone else is clustering. Re-queue.
        lock_ok = False

    if not lock_ok:
        _sb_execute(
            supabase.table("batch_stage_jobs").update({"status": "queued", "assigned_worker": None}).eq("id", job_id),
            context="batch_stage_jobs.update(clustering.requeue.lock_busy)",
        )
        time.sleep(2)
        return

    try:
        _sb_execute(
            supabase.table("libraries").update({"pipeline_status": "running", "pipeline_stage": "clustering"}).eq(
                "id", library_id
            ),
            context="libraries.update(stage.clustering)",
        )

        ids, vecs = _fetch_all_embeddings(library_id, emb_table)
        if not ids:
            raise RuntimeError("No embeddings found for library; cannot cluster.")

        mapping, centers = _run_kmeans(ids, vecs)

        # Update chunk embeddings with cluster_id.
        upserts = [{"chunk_id": cid, "cluster_id": int(lbl)} for cid, lbl in mapping.items()]
        chunk = int(os.getenv("CLUSTER_UPSERT_CHUNK", "2000"))
        for i in range(0, len(upserts), chunk):
            part = upserts[i : i + chunk]
            _sb_execute(
                supabase.table(emb_table).upsert(part, on_conflict="chunk_id"),
                context=f"{emb_table}.upsert(cluster_id)",
            )

        # Write cluster centroids + size for routing/UI.
        counts: dict[int, int] = {}
        for lbl in mapping.values():
            counts[int(lbl)] = counts.get(int(lbl), 0) + 1
        cluster_rows = []
        for cid, center in enumerate(centers):
            cluster_rows.append(
                {
                    "organization_id": org_id,
                    "library_id": library_id,
                    "cluster_id": int(cid),
                    "size": int(counts.get(int(cid), 0)),
                    "centroid": center,
                    "updated_at": now_iso(),
                }
            )
        if cluster_rows:
            _sb_execute(
                supabase.table(cluster_table).upsert(cluster_rows, on_conflict="library_id,cluster_id"),
                context=f"{cluster_table}.upsert",
            )

        _sb_execute(
            supabase.table(run_table).upsert(
                {
                    "library_id": library_id,
                    "organization_id": org_id,
                    "status": "done",
                    "finished_at": now_iso(),
                    "updated_at": now_iso(),
                    "last_error": None,
                },
                on_conflict="library_id",
            ),
            context=f"{run_table}.upsert(done)",
        )

        # Mark all clustering jobs for this library done (single library-level run).
        _sb_execute(
            supabase.table("batch_stage_jobs")
            .update({"status": "done", "finished_at": now_iso()})
            .eq("library_id", library_id)
            .eq("stage", "clustering")
            .neq("status", "done"),
            context="batch_stage_jobs.update(clustering.done.all)",
        )

        _update_library_progress(library_id)
        _maybe_finalize_pipeline(library_id)
    except Exception as exc:
        _sb_execute(
            supabase.table(run_table).upsert(
                {
                    "library_id": library_id,
                    "organization_id": org_id,
                    "status": "failed",
                    "finished_at": now_iso(),
                    "updated_at": now_iso(),
                    "last_error": str(exc),
                },
                on_conflict="library_id",
            ),
            context=f"{run_table}.upsert(failed)",
        )
        _sb_execute(
            supabase.table("batch_stage_jobs").update({"status": "failed", "last_error": str(exc)}).eq("id", job_id),
            context="batch_stage_jobs.update(clustering.failed)",
        )
        raise


def worker_loop():
    idle = 0
    idle_limit = int(os.getenv("CLUSTER_IDLE_LIMIT", "60"))
    print(f"[{WORKER_ID}] ready (idle_limit={idle_limit})")
    while True:
        job = claim_clustering_stage_job(WORKER_ID)
        if not job:
            idle += 1
            if idle >= idle_limit:
                print("No clustering jobs remaining. Exiting.")
                return
            time.sleep(2)
            continue
        idle = 0
        run_clustering_stage_job(job)


if __name__ == "__main__":
    worker_loop()



Overwriting cluster_worker.py


### app.py


In [ ]:
%%writefile app.py
# app.py
from fastapi import FastAPI
from hardware import auto_worker_plan
from fastapi.middleware.cors import CORSMiddleware
from worker_bootstrap import start_worker_pool, stop_worker_pool
from dotenv import load_dotenv
from pydantic import BaseModel
from chat_api import router as chat_router  # ✅ ADD THIS
import boto3
import os, subprocess

load_dotenv("/workspace/.env")

app = FastAPI()

# ✅ ADD THIS (register /chat and /chat/compact)
app.include_router(chat_router)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

_worker_stop = None
_worker_procs = None

class DeletePrefixRequest(BaseModel):
    prefix: str

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("R2_ENDPOINT"),
    aws_access_key_id=os.getenv("R2_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("R2_SECRET_KEY"),
)

R2_BUCKET = os.getenv("R2_BUCKET")

@app.post("/r2/delete-prefix")
def delete_prefix(req: DeletePrefixRequest):
    paginator = s3.get_paginator("list_objects_v2")
    to_delete = []
    for page in paginator.paginate(Bucket=R2_BUCKET, Prefix=req.prefix):
        for obj in page.get("Contents", []):
            to_delete.append({"Key": obj["Key"]})
            if len(to_delete) == 1000:
                s3.delete_objects(Bucket=R2_BUCKET, Delete={"Objects": to_delete})
                to_delete = []
    if to_delete:
        s3.delete_objects(Bucket=R2_BUCKET, Delete={"Objects": to_delete})
    return {"ok": True}

@app.on_event("startup")
def on_startup():
    global _worker_stop, _worker_procs
    _worker_stop, _worker_procs = start_worker_pool()

@app.on_event("shutdown")
def on_shutdown():
    if _worker_stop and _worker_procs:
        stop_worker_pool(_worker_stop, _worker_procs)

@app.get("/hardware")
def hardware():
    return auto_worker_plan()


Overwriting app.py


## 4) Quick commands (optional)
Use these to run the API / workers inside Colab.


In [ ]:
# Start the FastAPI backend (uvicorn) in the background
# You should see: uvicorn PID + "Backend is ready"

import py_compile
import subprocess
import sys
import time
from pathlib import Path

for _file in ["hardware.py", "sync_worker.py", "worker_bootstrap.py", "extraction_worker.py", "app.py"]:
    if Path(_file).exists():
        py_compile.compile(_file, doraise=True)
print("Python modules compile OK")

UVICORN_LOG = Path('/tmp/uvicorn.log')
subprocess.run(['pkill', '-f', 'uvicorn app:app'], check=False)
time.sleep(1)
UVICORN_LOG.write_text('', encoding='utf-8')
_uvicorn_log_f = open(UVICORN_LOG, 'w', encoding='utf-8')
uvicorn_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=_uvicorn_log_f,
    stderr=subprocess.STDOUT,
)
print('uvicorn PID:', uvicorn_proc.pid)
print('Uvicorn logs:', str(UVICORN_LOG))

origin = 'http://127.0.0.1:8000'
for _ in range(60):
    try:
        res = subprocess.run(['curl', '-sS', '-i', f'{origin}/hardware'], capture_output=True, text=True, timeout=3)
        first = (res.stdout.splitlines() or [''])[0]
        if ' 200 ' in first:
            print('Backend is ready:', f'{origin}/hardware')
            break
    except Exception:
        pass
    time.sleep(0.5)
else:
    print('Backend did not become ready. Tail uvicorn log:')
    try:
        print(UVICORN_LOG.read_text(encoding='utf-8', errors='ignore')[-2000:])
    except Exception as exc:
        print('Could not read uvicorn log:', exc)


Python modules compile OK
uvicorn PID: 20784
Uvicorn logs: /tmp/uvicorn.log
Backend is ready: http://127.0.0.1:8000/hardware


In [ ]:
# Start extraction workers bootstrap (spawns N workers)
# !python extraction_bootstrap.py


## 5) Public backend link (Cloudflare quick tunnel)
Run this after the backend is ready (the uvicorn cell must print "Backend is ready").


In [ ]:
import re
import subprocess
import time
from pathlib import Path

ORIGIN = 'http://127.0.0.1:8000'
CLOUDFLARED = Path('/content/cloudflared')
LOG_PATH = Path('/tmp/cloudflared.log')

res = subprocess.run(['curl','-sS','-i', f'{ORIGIN}/hardware'], capture_output=True, text=True, timeout=5)
first_line = (res.stdout.splitlines() or [''])[0]
if ' 200 ' not in first_line:
    print('ERROR: Origin is not returning 200 yet:', first_line)
    print('Run the uvicorn cell first and wait for readiness.')
    raise SystemExit(1)

subprocess.run(['pkill','-f','cloudflared'], check=False)
time.sleep(1)

# Download a fresh cloudflared binary
try:
    CLOUDFLARED.unlink()
except FileNotFoundError:
    pass
subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O', str(CLOUDFLARED)], check=True)
subprocess.run(['chmod','+x', str(CLOUDFLARED)], check=True)
subprocess.run([str(CLOUDFLARED),'--version'], check=False)

LOG_PATH.write_text('', encoding='utf-8')
_log_f = open(LOG_PATH, 'w', encoding='utf-8')
cloudflared_proc = subprocess.Popen([str(CLOUDFLARED), 'tunnel', '--url', ORIGIN, '--no-autoupdate'], stdout=_log_f, stderr=subprocess.STDOUT)
print('cloudflared PID:', cloudflared_proc.pid)

public_url = None
for _ in range(60):
    content = LOG_PATH.read_text(encoding='utf-8', errors='ignore') if LOG_PATH.exists() else ''
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

if public_url:
    print('Backend public URL:', public_url)
    print('Set NEXT_PUBLIC_RUNPOD_API_URL=' + public_url)
    print('Set BACKEND_API_URL=' + public_url)
else:
    print('Could not detect a public URL yet. Check /tmp/cloudflared.log')


cloudflared PID: 21091
Backend public URL: https://enforcement-grad-kevin-enhancements.trycloudflare.com
Set NEXT_PUBLIC_RUNPOD_API_URL=https://enforcement-grad-kevin-enhancements.trycloudflare.com
Set BACKEND_API_URL=https://enforcement-grad-kevin-enhancements.trycloudflare.com


In [ ]:
!curl -sS -i http://127.0.0.1:8000/hardware | head -n 5


HTTP/1.1 200 OK
date: Sat, 02 May 2026 10:07:28 GMT
server: uvicorn
content-length: 140
content-type: application/json


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8000/hardware

2026-05-02T10:07:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-02T10:07:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-02T10:07:33Z INF +--------------------------------------------------------------------------------------------+
2026-05-02T10:07:33Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-02T10:07:33Z INF |  https://transit-machine-marion-random.trycloudflare.c